# 6. 卷积、空间滤波与形态学

**目标**：掌握空间域滤波与形态学运算，能完成去噪、锐化、边缘增强、形状处理。

## 知识点清单

- 滤波的本质：核滑动 + 加权求和
- 相关运算与卷积运算：区别、filter2D
- 可分离卷积核
- 边界扩充：BORDER_CONSTANT、BORDER_REFLECT、BORDER_REPLICATE 等
- 空间滤波：盒式、高斯、统计排序、自适应、双边、钝化掩蔽、Laplacian、Sobel/Scharr
- 图像金字塔：高斯金字塔、拉普拉斯金字塔
- 形态学基础：腐蚀、膨胀、结构元 getStructuringElement
- 形态学高级运算：开运算、闭运算、梯度、顶帽、黑帽、击中击不中
- 形态学处理函数：morphologyEx
- 灰度形态学：灰度腐蚀、灰度膨胀、灰度开闭、灰度顶帽/底帽
- 形态学算法：边界提取、直线提取、线条细化
- 形态学重建：边界清除、孔洞填充、泛洪填充、骨架提取、粒径分离
- 基于形态学的粒度测定
- 形态学算法之边缘检测和角点检测

## 6.0 滤波的本质：核滑动 + 加权求和

### 内容介绍
滤波（Filtering）说白了就是：**用一个小窗口（核）在图像上滑动，每个位置算出一个新像素值**。

**核心思想**：
1. 准备一个小矩阵（如 3×3），叫**核（Kernel）**或**滤波器**。
2. 把核的中心对准图像某个像素。
3. 核覆盖的邻域像素 × 核的权重，全部相乘后求和。
4. 结果作为该像素的新值。
5. 核向右滑一格，重复直到遍历整幅图像。

```text
图像局部(3×3)        核(3×3)           输出
┌─────────┐      ┌─────────┐
│ 10 20 30│      │ 1  1  1 │
│ 40 50 60│  ×   │ 1  1  1 │  →  求和/9 = 新值
│ 70 80 90│      │ 1  1  1 │
└─────────┘      └─────────┘
```

**核的权重分布决定了滤波效果**：

| 核的特点 | 效果 | 代表 |
|---------|------|------|
| 全正、和为 1 | 平滑（低通） | 均值、高斯 |
| 中心正、周围负、和为 1 | 锐化（高通） | Laplacian、USM |
| 左右有正负 | 检测竖直边缘 | Sobel X |
| 上下有正负 | 检测水平边缘 | Sobel Y |
| 中心大、周围小 | 高斯平滑 | GaussianBlur |

**线性 vs 非线性**：
- 线性滤波：加权求和（均值、高斯、Sobel）。
- 非线性滤波：排序取值（中值、最大值、最小值）、双边滤波。

**保边 vs 不保边**：
- 普通平滑：边缘和噪声一起糊掉。
- 双边滤波：值域核让"差异大的邻居"权重接近 0 → 平滑但保留边缘。
- 中值滤波：取中位数，椒盐噪声被直接剔除。

**一句话总结**：滤波 = 设计不同的核 + 不同的组合方式，改变图像的频率特性。

---

### 先讲清楚：什么是"边缘"

后面反复出现"保边去噪"、"边缘区"、"平坦区"这些说法，先必须把"**边缘**"这个词讲清楚，否则很容易误解。

#### 一、"边缘"这个词有两个含义，别混淆

| 说法 | 含义 | 范围 |
|------|------|------|
| **图像最外圈的边缘** | 图片的上下左右四条边框 | 整张图 |
| **图像内部的边缘**（本节及后续讨论的边缘） | 画面中亮度/颜色**突变的位置**——物体轮廓、明暗交界 | **局部** |

**本节及后续说的"边缘"，一律指第二种**——画面内部的轮廓/交界，不是图像最外圈那四条边。

#### 二、用一行像素看清"边缘"是什么

假设图像某一行像素值：

```
位置：  1    2    3    4    5    6
值：    100  100  100  200  200  200
```

- 位置 1、2、3 都是 100；
- 位置 4、5、6 都是 200；
- **位置 3 和位置 4 之间，值发生了跳变**（100 → 200）；
- 这个"跳变的位置"就是**边缘**。

#### 三、严格来说：边缘是"相邻像素之间的差"，不是"某个像素"

| 概念 | 是什么 | 举例 |
|------|-------|------|
| **边缘** | 相邻像素之间的**跳变（差值）** | 位置 3（100）与位置 4（200）之间，差 = 100 |
| **边缘上的像素** | 构成这条跳变的那些**像素** | 位置 3、位置 4（边缘两侧的像素） |
| **中心点** | 当前窗口正对着的那个像素（要被改写的那一个） | 窗口滑到位置 3 时，中心点 = 位置 3 |

**关键区别**：

- **边缘**是"两个像素之间的关系"（差），不是"一个像素"；
- **边缘上的像素**才是具体的"位置"（位置 3、位置 4）；
- **中心点**是"当前处理的是哪一个像素"（角色，逐像素轮换）。

#### 四、"中心点落在边缘上"是什么意思

中心点是窗口当前正对着的像素，它在整张图上滑过每一个位置。中心点和"边缘"的关系分两种：

**情况 1：中心点落在边缘上**

比如窗口滑到位置 3（边缘上的像素），窗口覆盖 2、3、4：

```
窗口覆盖：100, 100, 200
        一半暗侧，一半亮侧
```

窗口**跨在边缘两侧** → 中心点落在边缘上。

**情况 2：中心点落在平坦区**

比如窗口滑到位置 1（远离边缘），窗口覆盖 1、2、3：

```
窗口覆盖：100, 100, 100
        全是差不多的值
```

窗口里没有跳变 → 中心点落在平坦区。

#### 五、"边缘区"和"平坦区"指的是窗口的状态

| 区域 | 窗口的观察 | 说明 |
|------|-----------|------|
| **平坦区** | 窗口内像素值都差不多 | 没有结构，只有噪声 |
| **边缘区** | 窗口内像素值差异很大（一半暗一半亮） | 窗口"跨越"了一条边缘 |

**"边缘区"不是"图像边框的区域"，而是"窗口里包含边缘两侧像素的区域"。**

#### 六、为什么均值滤波会"糊边缘"——关键在于逐像素滑动

有人会问：窗口处理只写回中心一个像素，其他像素没动，**边缘怎么会糊**？

答案：**滤波不是只对边缘上的一个像素做一次，而是整张图每个像素都当一次中心点，各做一次。**

以一行像素为例，用 3 点平均：

| 位置 | 原值 | 作为中心点处理后的值 |
|------|------|---------------------|
| 3 | 100 | 133（窗口 100,100,200 平均） |
| 4 | 200 | 167（窗口 100,200,200 平均） |

处理后：

| 位置 | 1 | 2 | 3 | 4 | 5 | 6 |
|------|---|---|---|---|---|---|
| 原值 | 100 | 100 | 100 | 200 | 200 | 200 |
| 滤波后 | 100 | 100 | **133** | **167** | 200 | 200 |

**边缘的跳变从 100 → 200（差 100），变成了 133 → 167（差 34）。**

**"边缘被糊"不是"某个像素被改成灰色"，而是"边缘两侧的像素各自被处理一次后，它们之间的差变小了"。** 差变小 = 对比度降低 = 边缘变模糊。

#### 七、一句话总结

> **本节说的"边缘"，是"画面内部的跳变位置"（明暗交界、物体轮廓），不是图像最外圈的边框。**
>
> - **边缘** = 相邻像素之间的差（关系），不是某个像素；
> - **边缘上的像素** = 构成这条差的具体像素（位置）；
> - **中心点** = 当前窗口正对着的像素（角色，逐像素轮换）；
> - **"中心点落在边缘上"** = 当前窗口跨在跳变两侧；
> - **"中心点落在平坦区"** = 当前窗口里全是差不多的值；
> - **"边缘被糊"** = 边缘两侧的像素都被各自处理一遍后，它们之间的差变小了。

> **为什么叫"滤波"?**
> 
> **滤波** = 借用信号处理的概念，指"**按频率对信号做选择性通过/抑制**"的操作。
>
> - "波" = 信号（图像亮度随空间的变化）
> - "滤" = 筛选
> - 在图像里，就是用核在空间域做加权组合，等价于在频率域做低通/高通/带通筛选。

你笔记里的均值、高斯 = 低通；Laplacian、Sobel = 高通；双边 = 保边低通——全都是"滤波"这个大家族的不同成员。

## 6.1 相关运算与卷积运算：filter2D

### 内容介绍
**相关运算（Correlation）** 与 **卷积运算（Convolution）** 是图像空间滤波的数学基础。

**相关运算**：将核（Kernel）直接滑过图像，在每个位置将核与图像对应像素相乘后求和。数学上不翻转核。

**卷积运算**：先将核旋转 180 度（水平和垂直方向都翻转），再滑过图像做同样的乘加运算。

对于**对称核**（如均值核、高斯核），相关与卷积的结果完全相同。但对于**非对称核**（如 Sobel 算子、方向性核），两者结果不同。

**OpenCV 的 `cv2.filter2D()` 默认执行的是相关运算**，而不是严格数学意义上的卷积。如果要做真正的卷积，需要手动翻转核：`cv2.filter2D(src, -1, cv2.flip(kernel, -1))`。

**算法过程**：
1. 将核中心对准图像当前像素。
2. 核覆盖区域与图像对应位置逐元素相乘。
3. 所有乘积求和，作为输出图像当前像素值。
4. 滑动核，重复直到遍历整幅图像。

```python
dst = cv2.filter2D(src, ddepth, kernel)
```
- `ddepth`：输出图像深度(每个像素用多少bit来存储)，常用 `-1` 表示与输入相同。

### 本节示例中两个核的作用

本节代码用了两个不同的核来对比相关与卷积，它们的作用如下：

**① 对称核：3×3 均值核**
```
1/9  1/9  1/9
1/9  1/9  1/9
1/9  1/9  1/9
```
- 这是一个**真实可用的滤波器**——最经典的盒式/均值滤波器。
- 核内所有权重相等且和为 1，输出像素 = 邻域 9 个像素的平均值。
- 效果：**去噪、模糊**，图像变得柔和，细节被抹平。
- 在本节中，它的**主要目的是充当"对称核"的示例**——用来证明"对称核下相关 = 卷积"，顺便展示平滑效果。

**② 非对称核：`[[1,2,3],[4,5,6],[7,8,9]]/45`**
```
1/45  2/45  3/45
4/45  5/45  6/45
7/45  8/45  9/45
```
- 这个核**不是任何标准图像滤波器**，纯粹是**教学道具**。
- 设计目的有三个：
  1. **构造一个非对称核**，让"相关 ≠ 卷积"的差异能显现出来。
  2. **权重和为 1**（1+2+…+9 = 45，除以 45），保证输出亮度大致不变，否则图像会整体变亮或变暗，干扰观察。
  3. **权重单调递增**（左上小、右下大），使核具有**方向性**——对从左上到右下方向的亮度变化更敏感，等效于一个"带方向性偏置的加权平均"，视觉上像"斜向拖影"的模糊。
- 它本身没有实际工程价值，选择它是**出于教学演示的简洁性**：让读者一眼就能看出"这两个结果不同"，而不被负值截断、归一化等细节干扰。

**③ 为什么不用 Sobel 核来演示？**
既然要演示非对称核，为什么不用真实的 Sobel 算子？因为：
- Sobel 核含负值，输出有正有负，需要 `CV_64F` + `convertScaleAbs`，代码更复杂。
- Sobel X 和 Sobel Y 需要两个核，对比不直观。
- 用 `[1..9]/45` 核，权重全正、和为 1，输出仍是"正常亮度图像"，相关和卷积结果都是可视化的灰度图，差异图也容易理解。

**④ 关于差异图"放大"的含义**
代码里 `cv2.normalize(diff, None, 0, 255, cv2.NORM_MINMAX)` 中的"放大"**不是指放大图像尺寸**，而是指**放大数值差异（增强对比度）**：
- 相关与卷积的差异非常小（大部分像素差 0~10，边缘 20~40）。
- 直接显示的话，整张图几乎全黑，看不清。
- `cv2.normalize` 做 min-max 线性拉伸，把原本集中在 0~40 的差异铺满 0~255 的显示范围，对比度被拉大，微小差异也能看出来。
- 类比：一张很暗的云照片，直接看是一团黑；把最暗当纯黑、最亮当纯白重新拉伸，云内部的纹理就显现了。

### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

img = cv2.imread('P&V/cat.jpg')                              # 读取图片
if img is not None:                                          # 如果图片存在
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)         # 转灰度便于观察

    # --- 对称核：相关与卷积结果相同 ---
    kernel_sym = np.ones((3, 3), np.float32) / 9.0           # 3x3 均值核（真实滤波器，对称）
    dst_corr = cv2.filter2D(img_gray, -1, kernel_sym)        # 相关
    dst_conv = cv2.filter2D(img_gray, -1, cv2.flip(kernel_sym, -1))  # 卷积（翻转核）

    # --- 非对称核：相关与卷积结果不同 ---
    kernel_asym = np.array([[1, 2, 3],                       # 非对称核（教学道具，权重递增）
                            [4, 5, 6],                       # 权重和为 1，保证亮度不变
                            [7, 8, 9]], dtype=np.float32) / 45.0
    dst_corr_a = cv2.filter2D(img_gray, -1, kernel_asym)     # 相关
    dst_conv_a = cv2.filter2D(img_gray, -1, cv2.flip(kernel_asym, -1))  # 卷积

    plt.figure(figsize=(15, 8))                              # 创建画布
    titles = ['原灰度图', '对称核-相关', '对称核-卷积',          # 标题
              '非对称核-相关', '非对称核-卷积', '非对称核差异(放大)']
    diff = cv2.absdiff(dst_corr_a, dst_conv_a)               # 相关与卷积的差异
    diff_vis = cv2.normalize(diff, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)  # 归一化（对比度拉伸）便于观察
    images = [img_gray, dst_corr, dst_conv, dst_corr_a, dst_conv_a, diff_vis]  # 图像列表
    for i in range(6):                                       # 遍历
        plt.subplot(2, 3, i+1)                               # 子图
        plt.title(titles[i])                                 # 标题
        plt.imshow(images[i], cmap='gray')                   # 显示图像
        plt.axis('off')                                      # 关闭坐标轴
    plt.show()                                               # 显示图像
else:
    print("请确保 'P&V/cat.jpg' 存在")                       # 提示信息

## 6.2 可分离卷积核

### 内容介绍
**"可分离"是卷积运算里的一个数学性质**，意思是：一个二维卷积核可以拆成两个一维卷积核的外积，从而把一次二维卷积拆成两次一维卷积来做。

### 一、什么叫"可分离"？

**数学定义**：如果二维核 $K$ 能写成两个向量的外积：

$$K = v \cdot h^T$$

其中 $v$ 是列向量（竖直方向），$h$ 是行向量（水平方向），那么这个核就是**可分离的**。

### 二、举例：二维高斯核

设一维高斯向量（5 个元素）为：

$$g = [0.05, 0.24, 0.40, 0.24, 0.05]$$

它的外积 $g \cdot g^T$ 得到一个 5×5 矩阵，这就是**可分离的二维高斯核**：

```text
0.0025  0.012  0.020  0.012  0.0025
0.012   0.058  0.096  0.058  0.012
0.020   0.096  0.160  0.096  0.020
0.012   0.058  0.096  0.058  0.012
0.0025  0.012  0.020  0.012  0.0025
```

**关键点**：这个 25 个元素的二维核，完全由那 **5 个元素的一维向量** 决定。知道一维向量，就能还原整个二维核。

### 三、可分离带来什么好处？——省计算

**3.1 不可分离：直接做二维卷积**

对每个像素，核覆盖 $k \times k$ 个邻域，要做 $k^2$ 次乘加。

- 5×5 核 → 每个像素 **25 次**乘加

**3.2 可分离：拆成两次一维卷积**

1. 先用行向量 $h^T$（1×k）对图像做**水平一维卷积** → 每个像素 k 次乘加
2. 再用列向量 $v$（k×1）对结果做**垂直一维卷积** → 每个像素 k 次乘加

总共每个像素 $k + k = 2k$ 次乘加。

- 5×5 核 → 每个像素 **10 次**乘加

**3.3 对比表**

| 核大小 | 二维卷积（$k^2$） | 可分离（$2k$） | 加速比 |
|-------|-----------------|--------------|-------|
| 3×3   | 9               | 6            | 1.5×  |
| 5×5   | 25              | 10           | 2.5×  |
| 11×11 | 121             | 22           | 5.5×  |
| 21×21 | 441             | 42           | 10.5× |

**核越大，加速越明显**。这就是为什么 OpenCV 的 `GaussianBlur` 内部自动用可分离实现——大核高斯模糊几乎不卡。

### 四、哪些核可分离？

| 核 | 是否可分离 | 说明 |
|----|----------|------|
| 均值核（全 1） | ✅ | $K = \mathbf{1} \cdot \mathbf{1}^T$ |
| 高斯核         | ✅ | $K = g \cdot g^T$ |
| 盒式核         | ✅ | 同上 |
| Sobel X        | ✅ | 可拆为竖直平滑 × 水平差分 |
| Sobel Y        | ✅ | 可拆为水平平滑 × 竖直差分 |
| Scharr X       | ✅ | 可拆为竖直平滑 × 水平差分 |
| Scharr Y       | ✅ | 可拆为水平平滑 × 竖直差分 |
| 任意随机核     | ❌ | 大多数随机核不可分离 |
| 非对称核 `[1..9]/45` | ❌ | 无法写成外积 |

**判断方法**：对核做 SVD（奇异值分解），如果只有一个非零奇异值，就可分离。

### 五、几何直觉：为什么叫"分离"？

- **二维卷积** = 在水平和垂直两个方向上**同时**做加权组合。
- **可分离卷积** = 先在水平方向做一次加权组合，再在垂直方向做一次加权组合，两个方向**分开**做。

**类比做菜**：
- **不可分离**：一次性把所有调料（盐、糖、酱油）一起放进去炒
- **可分离**：先放盐炒一下，再放糖炒一下

如果顺序不影响最终味道（线性操作满足交换律和结合律），那就可以分开做，效果一样但更省力。

对线性滤波来说，**先横后竖** 和 **一次性二维** 数学上完全等价，但分开做更省算力。

### 六、代码里的体现

```python
kernel_x = cv2.getGaussianKernel(5, 1.5)                       # 一维水平高斯核
kernel_y = cv2.getGaussianKernel(5, 1.5)                       # 一维垂直高斯核
dst_sep   = cv2.sepFilter2D(img_gray, -1, kernel_x, kernel_y)  # 先横后竖
dst_gauss = cv2.GaussianBlur(img_gray, (5, 5), 1.5)            # 内部也是可分离
```

- `getGaussianKernel(5, 1.5)` 返回的是**一维**向量（5 个元素），不是 5×5 矩阵。
- `sepFilter2D` 接收两个一维核，分别做水平、垂直卷积。
- 结果与直接做 5×5 二维高斯卷积**完全相同**，但更快。
- 代码里两者结果看起来一致，正是为了证明这个等价性。

### 七、一句话总结

> **"可分离"= 二维核能拆成"水平一维 × 垂直一维"，于是二维卷积可以拆成两次一维卷积来做。数学上等价，计算量从 $O(k^2)$ 降到 $O(2k)$，大核时能大幅加速。**

高斯核、均值核、Sobel 核、Scharr 核都可分离，所以 OpenCV 内部都用这种方式实现。

### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

img = cv2.imread('P&V/cat.jpg')                              # 读取图片
if img is not None:                                          # 如果图片存在
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)         # 转换为灰度图
    kernel_x = cv2.getGaussianKernel(5, 1.5)                 # 水平方向一维高斯核（5×1 列向量）
    kernel_y = cv2.getGaussianKernel(5, 1.5)                 # 垂直方向一维高斯核（5×1 列向量）
    dst_sep = cv2.sepFilter2D(img_gray, -1, kernel_x, kernel_y)  # 可分离滤波（先横后竖）
    dst_gauss = cv2.GaussianBlur(img_gray, (5, 5), 1.5)      # 直接高斯模糊（内部也是可分离实现）
    
    plt.figure(figsize=(12, 4))                              # 创建画布
    plt.subplot(1, 3, 1)                                     # 子图 1
    plt.title('原灰度图')                                     # 标题
    plt.imshow(img_gray, cmap='gray')                        # 显示原灰度图
    plt.axis('off')                                          # 关闭坐标轴
    
    plt.subplot(1, 3, 2)                                     # 子图 2
    plt.title('sepFilter2D (可分离)')                         # 标题
    plt.imshow(dst_sep, cmap='gray')                         # 显示可分离滤波结果
    plt.axis('off')                                          # 关闭坐标轴
    
    plt.subplot(1, 3, 3)                                     # 子图 3
    plt.title('GaussianBlur (对比)')                          # 标题
    plt.imshow(dst_gauss, cmap='gray')                       # 显示高斯模糊结果
    plt.axis('off')                                          # 关闭坐标轴
    plt.show()                                               # 显示图像
else:
    print("请确保 'P&V/cat.jpg' 存在")                       # 提示信息

**图像解读**

一、三张子图分别是什么

| 子图 | 内容 | 对应代码 |
|------|------|---------|
| 左：原灰度图 | 输入的猫图（灰度） | `img_gray` |
| 中：sepFilter2D (可分离) | 用两次**一维**高斯卷积 | `cv2.sepFilter2D(img_gray, -1, kernel_x, kernel_y)` |
| 右：GaussianBlur (对比) | 用一次**二维**高斯卷积 | `cv2.GaussianBlur(img_gray, (5, 5), 1.5)` |

---

二、观察到的现象

1. **中图和右图看起来完全一样**：都是原图被柔和地模糊了一遍，猫的轮廓、胡须、斑纹都变模糊但整体亮度、结构保持一致。
2. **与左图对比**：中、右两图明显比原图更“糊”——细节被抹平，说明高斯平滑确实生效了。
3. **中图和右图之间**：肉眼**看不出任何差异**（像素级别上也只是浮点误差范围内的一致）。

---

三、这说明了什么

1. 可分离滤波 = 二维高斯滤波

`sepFilter2D` 先做水平 1D 高斯卷积，再做垂直 1D 高斯卷积，两步合起来的效果，**和直接用 5×5 二维高斯核做一次卷积，结果完全相同**。这正好印证了 6.2 节的核心结论：

$$K_{2D} = g \cdot g^T \quad\Longrightarrow\quad I * K_{2D} = (I * g^T) * g$$

即：二维高斯核可以拆成两个一维高斯核的外积，卷积可以拆成“先横后竖”两次一维卷积。

2. 等价但更快

虽然结果一样，但计算量差别很大：

| 方法 | 每像素乘加次数（5×5 核） | 相对速度 |
|------|----------------------|---------|
| GaussianBlur（二维） | 25 | 基准 |
| sepFilter2D（可分离） | 5 + 5 = 10 | 约 2.5× 更快 |

代码里两图结果一致，正是为了**用可视化的方式证明这种等价性**，让读者放心：以后遇到大核高斯模糊，就可以放心使用可分离实现，不用牺牲效果换速度。

3. 为什么视觉上“看不出区别”

- 两次一维高斯卷积和一次二维高斯卷积在**数学上严格等价**（对线性滤波成立）；
- OpenCV 的 `sepFilter2D` 和 `GaussianBlur` 都使用浮点或定点高精度中间结果，浮点误差极小；
- 因此两图在肉眼和像素层面都几乎一致。

---

四、小结

> 这张图用“中图和右图完全一致”这一事实，直观地验证了 **“可分离 = 二维核拆成两方向一维核，数学等价，但算得更快”** 这个知识点。

- 左图是参照；
- 中图：**先横后竖两次一维卷积**（可分离，快）；
- 右图：**一次二维卷积**（直接用 5×5 高斯核，慢）；
- 中 = 右 → 等价性得证。

## 6.3 边界扩充：BORDER_CONSTANT、BORDER_REFLECT、BORDER_REPLICATE 等

### 内容介绍
滤波（尤其是较大核）时，核会超出图像边界，需要人为补充像素，这叫**边界扩充**。

常用方式：
- `BORDER_CONSTANT`：用固定常数填充（默认为 0）。
- `BORDER_REPLICATE`：复制最边缘的像素，向后延展。
- `BORDER_REFLECT`：镜像反射（如 `fedcba|abcdefgh|hgfedcb`）。
- `BORDER_REFLECT_101`（默认）：不重复边界像素的镜像（如 `gfedcb|abcdefgh|gfedcba`）。
- `BORDER_WRAP`：环绕填充，从对侧拿像素。

**算法过程**：OpenCV 在内部用一个"虚拟图像"做卷积，将超出边界的部分按照指定规则生成。不同扩充方式对边缘结果有明显影响，特别是在去噪和检测边缘时。

> **注意**：这里的"边界"指**图像最外圈的边框**，和 6.0 节讲的"画面内部的边缘"是两个完全不同的概念，别混淆。

### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

# --- 构造原始 ROI：10×10 全黑，中间 4×4 置白 ---
roi = np.zeros((10, 10), dtype=np.uint8)                     # 10×10 全黑
roi[3:7, 3:7] = 255                                          # 中间 4×4 置白

borders = {                                                  # 定义边界类型
    'CONSTANT': cv2.BORDER_CONSTANT,                         # 常数填充
    'REPLICATE': cv2.BORDER_REPLICATE,                       # 复制边缘
    'REFLECT': cv2.BORDER_REFLECT,                           # 镜像反射
    'REFLECT_101': cv2.BORDER_REFLECT_101,                   # 镜像反射（默认）
    'WRAP': cv2.BORDER_WRAP                                  # 环绕填充
}

SIZE = 20                                                    # 统一画布尺寸
CELL = 2                                                     # 棋盘格单元格大小

def make_checkerboard(size, cell=CELL, c1=200, c2=230):      # 生成棋盘格背景
    board = np.zeros((size, size), dtype=np.uint8)           # 初始化
    for i in range(size):                                    # 遍历行
        for j in range(size):                                # 遍历列
            board[i, j] = c1 if ((i // cell) + (j // cell)) % 2 == 0 else c2
    return board                                             # 返回棋盘格

def centered_on_checker(img, size):                          # 把图像居中贴在棋盘格上
    h, w = img.shape                                         # 图像实际尺寸
    canvas = make_checkerboard(size)                         # 棋盘格背景
    top = (size - h) // 2                                    # 垂直居中偏移
    left = (size - w) // 2                                   # 水平居中偏移
    canvas[top:top+h, left:left+w] = img                     # 贴入图像内容
    return canvas                                            # 返回结果

plt.figure(figsize=(18, 3))                                  # 创建画布

ax = plt.subplot(1, 6, 1)                                    # 第 1 张：原始 ROI
ax.imshow(centered_on_checker(roi, SIZE), cmap='gray', vmin=0, vmax=255)
ax.set_title('原始 ROI (10×10)')                             # 标题带尺寸
ax.axis('off')                                               # 关闭坐标轴

for i, (name, border_type) in enumerate(borders.items()):    # 遍历边界类型
    padded = cv2.copyMakeBorder(roi, 5, 5, 5, 5, border_type)  # 边界扩充
    ax = plt.subplot(1, 6, i+2)                              # 第 2~6 张
    ax.imshow(padded, cmap='gray', vmin=0, vmax=255)         # 正常显示
    ax.set_title(f'{name} ({padded.shape[1]}×{padded.shape[0]})')  # 标题带尺寸
    ax.axis('off')                                           # 关闭坐标轴

plt.show()                                                   # 显示图像

**边界扩充结果解读**

这张图是 6.3 边界扩充 这一节代码的运行结果，用来直观对比 OpenCV 中 5 种边界扩充方式在图像边缘处补像素的差异。

一、实验设置

- ROI 用合成图：10×10 全黑，中间 4×4 置白（白块占第 3~6 行、第 3~6 列）。
- 用 cv2.copyMakeBorder(roi, 5, 5, 5, 5, border_type) 在上下左右各补 5 像素。
- 补边后从 10×10 变成 20×20，中间白块尺寸不变，四周一圈是扩充出来的像素。
- 画布共 5 张子图，从左到右依次对应：BORDER_CONSTANT、BORDER_REPLICATE、BORDER_REFLECT、BORDER_REFLECT_101、BORDER_WRAP。
- 每张子图标题都标注了尺寸：补边后各为 (20×20)，方便对照。
- 显示时用 imshow(..., vmin=0, vmax=255) 固定范围，避免 imshow 自动对比度拉伸造成的误导。

二、ROI 结构（补边之前）

原始 ROI 是 10×10 的小图，用行号 0~9 标记，结构如下（列同理）：

行号: 0  1  2  3  4  5  6  7  8  9
颜色: 黑 黑 黑 白 白 白 白 黑 黑 黑

即：黑(0~2) + 白(3~6) + 黑(7~9) = 10。

三、尺寸与结构对照

原始 ROI（10×10）：

| 段 | 行号 | 长度 |
|----|------|------|
| 上黑边 | 0~2 | 3 |
| 白块 | 3~6 | 4 |
| 下黑边 | 7~9 | 3 |
| 合计 | | 10 |

补边 5 像素后（20×20）：

| 段 | 长度 |
|----|------|
| 黑边（每侧） | 5(补) + 3(原) = 8 |
| 白块（中间） | 4（不变） |
| 合计 | 8 + 4 + 8 = 20 |

关键对比：白块尺寸始终是 4×4，补边只是把画布从 10×10 撑到 20×20，白块在整幅图中的占比从 40% 降到 20%。

四、copyMakeBorder 的内部机制：四个角怎么被填上

很多人以为补边是“四个方向各自独立补一条边”，那四角就会空出来。其实不然。

cv2.copyMakeBorder 的内部逻辑是分两步：

1. 先上下补：把 10×10 扩成 (10+5+5) × 10 = 20×10；
2. 再左右补：把 20×10 扩成 20 × (10+5+5) = 20×20。

关键是第二步：左右补不是只作用在“原图的 10 行”上，而是作用在“已经上下补过的全部 20 行”上。因此：

- 顶部补出来的 5 行，其左右两端也会被补；
- 底部补出来的 5 行，其左右两端也会被补；
- 这四个“顶部/底部的行 × 左右补的列”区域，正好就是四个角。

所以四个角不会空：它们由“上下补”决定了行、由“左右补”决定了列，两次补边共同填满。

用小例子验证（4×4 原图，四边各补 2，WRAP 规则）：

原图：
a b c d
e f g h
i j k l
m n o p

第一步：上下补，得到 8×4：
i j k l   ← 上补第1行（取原图倒数第2行）
m n o p   ← 上补第2行（取原图最后1行）
a b c d   ← 原图第1行
e f g h
i j k l
m n o p
a b c d   ← 下补第1行（取原图第1行）
e f g h   ← 下补第2行（取原图第2行）

第二步：对 8 行每行左右各补 2 列，得到 8×8：
l i j k l i j k
p m n o p m n o
d a b c d a b c
h e f g h e f g
l i j k l i j k
p m n o p m n o
d a b c d a b c
h e f g h e f g

四个角全被填上：左上 l,i；右上 j,k；左下 d,a；右下 b,c。

结论：copyMakeBorder 先扩行、再扩列，第二步作用在“全部已扩过的行”上，所以四角自动填满，不会有空缺。

五、CONSTANT 是怎么补的

CONSTANT 的规则：补边区域全部填固定常数（默认 0，即黑色），完全不参考 ROI 的内容。

补边时，上下左右各补 5 像素，全部填常数 0：

- 上边补 5 行 → 全 0（黑）；
- 下边补 5 行 → 全 0（黑）；
- 左边补 5 列 → 全 0（黑）；
- 右边补 5 列 → 全 0（黑）；
- 四个角 → 也是 0（黑）。

结果：20×20 的大黑方块，中间一个 4×4 白块。

为什么看不出“哪部分是原图、哪部分是补出来的”？

因为补边填的是 0（黑），而 ROI 最外圈本身也是黑（行 0~2、行 7~9、列 0~2、列 7~9，灰度值也是 0）。两者颜色完全一样，视觉上无缝融合。

六、REPLICATE 是怎么补的

REPLICATE 的规则：补边时复制最外侧那一行/列，向外延展。

具体到这块 ROI：

- 上边补 5 行 → 全部复制行 0（黑）；
- 下边补 5 行 → 全部复制行 9（黑）；
- 左边补 5 列 → 全部复制列 0（黑）；
- 右边补 5 列 → 全部复制列 9（黑）；
- 四个角 → 复制对应角的边界像素（也是黑）。

因为这块 ROI 的最外一圈本来就是黑色（灰度 0），所以：

| 补边方式 | 补的内容 | 灰度值 |
|---------|---------|--------|
| CONSTANT | 常数 0 | 0（黑） |
| REPLICATE | 复制行 0 / 行 9 | 0（黑） |

两者补出来都是 0，视觉上和 CONSTANT 完全一样：一大块 20×20 的黑底，中间一个 4×4 白块。

这是这张 ROI 的一个“巧合”：黑边恰好是纯黑，所以“复制黑边”和“填常数黑”结果相同。

什么时候 REPLICATE 和 CONSTANT 才会不一样？

当 ROI 最外圈不是常数色时，两者立刻不同。例如 ROI 顶行是渐变 10 20 ... 100：

- CONSTANT 补上边 → 全 0（黑），和顶行完全不搭，出现突兀黑边；
- REPLICATE 补上边 → 全是顶行的值（10 20 ... 100），和顶行一模一样，视觉上“边缘被拉伸”，过渡自然。

所以：REPLICATE 和 CONSTANT 的区别，只在“最外圈不是常数色”时才能体现出来。

七、REFLECT 是怎么补的（用行号 0~9 推导）

REFLECT 的规则：以边界像素本身为镜面，向外镜像，边界像素重复一次。

用序列写法（以行为例）：

9,8,7,6,5,4,3,2,1,0 | 0,1,2,3,4,5,6,7,8,9 | 9,8,7,6,5,4,3,2,1,0

- 中间 0,1,...,9 是原 ROI 的行；
- 左边 9,8,...,0 是上边补出来的行（从边界行 0 开始镜像，行 0 重复一次）；
- 右边 9,8,...,0 是下边补出来的行（从边界行 9 开始镜像，行 9 重复一次）。

上边补 5 行，镜像序列从边界行 0 开始：0, 1, 2, 3, 4 ...

- 补的第 1 行 = 行 0 = 黑
- 补的第 2 行 = 行 1 = 黑
- 补的第 3 行 = 行 2 = 黑
- 补的第 4 行 = 行 3 = 白
- 补的第 5 行 = 行 4 = 白

上边补出来的 5 行（从外到内）：黑 黑 黑 白 白。

下边补 5 行，镜像序列从边界行 9 开始：9, 8, 7, 6, 5 ...

- 补的第 1 行 = 行 9 = 黑
- 补的第 2 行 = 行 8 = 黑
- 补的第 3 行 = 行 7 = 黑
- 补的第 4 行 = 行 6 = 白
- 补的第 5 行 = 行 5 = 白

下边补出来的 5 行（从外到内）：黑 黑 黑 白 白。

左右同理。于是整幅 20×20 图呈现：

- 中间：原始 4×4 白块（行 3~6）；
- 上下四边：各出现方块状的白块（由白块镜像而来）；
- 整体：像一个“方形齿轮”或“十字带缺口”的图案。

为什么不是“锯齿”而是“方块”？

因为白块本身是矩形（4×4），镜像出来也是矩形，所以凸起是方方正正的块状，不是尖尖的锯齿。准确说法是：方块状的白块镜像。

八、REFLECT_101 是怎么补的（和 REFLECT 差在哪）

REFLECT_101 的规则：以边界像素的内侧为镜面，向外镜像，边界像素不重复。

用序列写法（以行为例）：

8,7,6,5,4,3,2,1 | 0,1,2,3,4,5,6,7,8,9 | 8,7,6,5,4,3,2,1

和 REFLECT 对比：

9,8,7,6,5,4,3,2,1,0 | 0,1,2,3,4,5,6,7,8,9 | 9,8,7,6,5,4,3,2,1,0

关键区别：

- REFLECT：镜像从边界行 0 开始，行 0 出现两次；
- REFLECT_101：镜像从边界内侧行 1 开始，跳过行 0，行 0 只出现一次。

也就是 REFLECT_101 的镜像序列整体往图像内侧移了一位。

上边补 5 行，镜像序列从行 1 开始（跳过行 0）：1, 2, 3, 4, 5 ...

- 补的第 1 行 = 行 1 = 黑
- 补的第 2 行 = 行 2 = 黑
- 补的第 3 行 = 行 3 = 白
- 补的第 4 行 = 行 4 = 白
- 补的第 5 行 = 行 5 = 白

上边补出来的 5 行（从外到内）：黑 黑 白 白 白。

下边补 5 行，镜像序列从行 8 开始（跳过行 9）：8, 7, 6, 5, 4 ...

- 补的第 1 行 = 行 8 = 黑
- 补的第 2 行 = 行 7 = 黑
- 补的第 3 行 = 行 6 = 白
- 补的第 4 行 = 行 5 = 白
- 补的第 5 行 = 行 4 = 白

下边补出来的 5 行（从外到内）：黑 黑 白 白 白。

和 REFLECT 对比（补 5 行）：

| 位置 | REFLECT | REFLECT_101 |
|------|---------|-------------|
| 上边 | 黑 黑 黑 白 白 | 黑 黑 白 白 白 |
| 下边 | 黑 黑 黑 白 白 | 黑 黑 白 白 白 |

差别：

- REFLECT 补出来的白块是第 4、5 行（贴近外侧）；
- REFLECT_101 补出来的白块是第 3、4、5 行（多了一行白，位置往内侧移）。

视觉上：两图都呈现“十字带缺口的图案”，但 REFLECT_101 的白块比 REFLECT 往图像内侧错开一行，所以形状相似、相位差一格。

为什么肉眼看不太出来：

1. 白块本身是对称的（4×4），错一格后整体形状依然是“十字带缺口”，差异细微；
2. 黑背景占比太大，错一格的视觉冲击被大量黑色淹没；
3. 补边只有 5 像素，错一格只移动 1/5 的补边宽度，比例很小。

九、WRAP 是怎么补的（用行号 0~9 推导）

WRAP 的规则：把图像看成一个首尾相接的环，补边时从对侧取像素。

用序列写法（以行为例）：

5,6,7,8,9 | 0,1,2,3,4,5,6,7,8,9 | 0,1,2,3,4

上边补边时，图像“上面”没了，就顺着环从“下面”接过来。沿环从下往上走的顺序是 9,8,7,6,5,4,...，所以上边补 5 行取的是：

- 补的第 1 行（最外）= 行 9 = 黑
- 补的第 2 行 = 行 8 = 黑
- 补的第 3 行 = 行 7 = 黑
- 补的第 4 行 = 行 6 = 白
- 补的第 5 行 = 行 5 = 白

上边补出来的 5 行（从外到内）：黑 黑 黑 白 白。

下边补边时，从顶部取行 0,1,2,3,4：

- 补的第 1 行（最外）= 行 0 = 黑
- 补的第 2 行 = 行 1 = 黑
- 补的第 3 行 = 行 2 = 黑
- 补的第 4 行 = 行 3 = 白
- 补的第 5 行 = 行 4 = 白

下边补出来的 5 行（从外到内）：黑 黑 黑 白 白。

和 REFLECT 对比（上边补 5 行）：

| 方式 | 取的行号 | 补出来的 5 行 |
|------|---------|--------------|
| REFLECT | 0,1,2,3,4 | 黑 黑 黑 白 白 |
| REFLECT_101 | 1,2,3,4,5 | 黑 黑 白 白 白 |
| WRAP | 9,8,7,6,5 | 黑 黑 黑 白 白 |

巧合出现了：WRAP 取行号 9,8,7,6,5，颜色是 黑 黑 黑 白 白；REFLECT 取行号 0,1,2,3,4，颜色也是 黑 黑 黑 白 白。

因为 ROI 完全对称，从底部倒着取和从顶部正着取，得到的颜色序列一模一样。这就是这张图里 WRAP 和 REFLECT 看起来几乎一样的原因——它们是两个完全不同的规则，但因为 ROI 对称，结果碰巧相同。

WRAP 的实际用途：主要用于周期性图像（纹理、平铺图案、频域信号），补边后图像“接得上”，不会在边界处出现突变。对普通照片（如猫图），WRAP 不常用，因为照片上下左右通常不连续，“卷”过去会出现明显的错位。

十、5 张图各自是什么

| 子图 | 内容 | 尺寸 | 你看到的 |
|------|------|------|---------|
| 第 1 张 | CONSTANT | 20×20 | 白块居中，四周一整片纯黑，看不出补边分界 |
| 第 2 张 | REPLICATE | 20×20 | 和 CONSTANT 几乎一样（最外行/列本身就是黑） |
| 第 3 张 | REFLECT | 20×20 | 白块外出现方块状白色凸起（镜像到内部白块），白块贴外侧 |
| 第 4 张 | REFLECT_101 | 20×20 | 也是方块状白色凸起，白块比 REFLECT 往内侧错开一行 |
| 第 5 张 | WRAP | 20×20 | 也是方块状白色凸起，和上面两张很像 |

关键观察：

- 前两张（CONSTANT、REPLICATE）是“实心黑 + 中间白块”，没有凸起；
- 后三张（REFLECT / REFLECT_101 / WRAP）在四周各出现方块状白色凸起，所以看起来相似；
- 其中 REFLECT 和 REFLECT_101 的差异是“白块相位差一行”，肉眼看很细微。

十一、为什么后三者几乎分不出来

三个原因叠加：

1. ROI 完全对称（上下左右都对称，黑框均匀），导致镜像和环绕取到的像素高度相似；
2. 白方块太小（4×4，只占中间），四周黑边很厚，白色凸起的变化被大量黑背景淹没；
3. 黑边厚度 ≥ 补边宽度（黑边 3 像素，补 5 像素时穿到了白块，但穿得不多），差异只体现在很窄的一圈。

一句话：ROI 太对称、太“黑多白少”，所以 REFLECT / REFLECT_101 / WRAP 的差异被压到最小。

十二、怎么改，才能让 5 张真正区分开

要放大差异，得破坏对称性、让边界处对比更强。推荐几种 ROI：

1. 变体 1：上白下黑（破坏上下对称）

roi = np.zeros((10, 10), dtype=np.uint8)                     # 全黑
roi[0:5, :] = 255                                            # 上半置白

- CONSTANT：补常数黑，上下都会出现黑边，和内部白块形成对比；
- REPLICATE：顶边补白（复制上外行白），底边补黑（复制下外行黑），上下明显不对称——和 CONSTANT 立刻区分开；
- REFLECT：顶边补白，底边补黑，和 REPLICATE 类似但相位不同；
- REFLECT_101：顶边补的是第二行（还是白），底边补的是倒数第二行（还是黑），相位和 REFLECT 差一行；
- WRAP：顶部从底部拿（黑），底部从顶部拿（白），和镜像完全反过来——一眼就能区分。

这个 ROI 能同时区分 5 种方式，是最推荐的。

2. 变体 2：棋盘格（周期性）

roi = (np.indices((10, 10)).sum(axis=0) % 2 * 255).astype(np.uint8)  # 棋盘

- WRAP：因为是周期的，接得上，看起来像棋盘无缝延伸；
- REPLICATE：边界行被拉伸成条纹；
- REFLECT vs REFLECT_101：相位差一格，黑白格错开。

棋盘格是区分 WRAP 和镜像的最佳 ROI。

3. 变体 3：左白右黑

破坏左右对称，同理能拉开 REFLECT 和 WRAP 的差异。

十三、小结

> - 5 张图是补边后的结果（20×20），标题均标注了尺寸，格式统一，便于对照。
> - copyMakeBorder 先扩行、再扩列，第二步作用在“全部已扩过的行”上，所以四角自动填满，不会有空缺。
> - CONSTANT 和 REPLICATE 看起来“没有补边痕迹”，是因为它们补的黑和原图的黑边同色，视觉上无缝融合。这是这张图最容易被忽略、也最关键的一点。
> - REPLICATE 和 CONSTANT 的区别，只在“ROI 最外圈不是常数色”时才能体现出来；现在这张 ROI 最外圈恰好是纯黑（= 0 = 常数），所以两者撞了。
> - REFLECT 和 REFLECT_101 都会把 ROI 内部的白块镜像到补边区域，所以四周出现方块状白色凸起。
> - REFLECT 和 REFLECT_101 的唯一差别是“边界行是否重复”：REFLECT 重复，REFLECT_101 不重复，导致白块相位差一行。
> - WRAP 是“从对侧取像素”，本应和镜像不同；但因为 ROI 完全对称，WRAP 和 REFLECT 在这张图上碰巧得到相同的颜色序列。
> - 补边只改变画布大小，白块尺寸始终 4×4 不变。
> - 右边 3 张（REFLECT / REFLECT_101 / WRAP）相似，是因为 ROI 完全对称 + 黑多白少，导致镜像和环绕取到的像素几乎一样。
> - 想区分它们，要破坏对称、增加边界处对比。
> - 最佳选择是 上白下黑（同时区分 5 种）或 棋盘格（区分 WRAP 和镜像最有效）。
> - 现在的“中间小白块 + 厚黑框”结构，天生就是把 3 种方式的差异压到最小的。

## 6.4 空间滤波

空间滤波是图像处理中最基础也最重要的操作之一。它通过在图像上滑动一个“核”（或称“滤波器”、“窗口”），对核覆盖的邻域像素进行某种运算，从而得到输出像素值。

根据运算方式的不同，空间滤波可分为**线性滤波**和**非线性滤波**两大类。本小节将系统介绍常见的空间滤波方法，包括盒式滤波、高斯滤波、统计排序滤波、自适应滤波、双边滤波、钝化掩蔽、Laplacian 算子以及 Sobel/Scharr 算子。

> **复习**：在进入 6.4 节之前，请先回顾 6.0 节对"**边缘**"的定义——**边缘 = 相邻像素之间的差（画面内部的跳变），不是某个像素，也不是图像最外圈的边框**。后续所有讨论"保边去噪"、"边缘区"、"平坦区"的内容，都建立在这个定义之上。

### 6.4.1 盒式滤波器：boxFilter、blur

**盒式滤波**是最简单的一类线性滤波：用一个大小固定的矩形核，对核内所有像素求平均。

**算法过程**：
1. 定义一个全 1 的 $k \times k$ 核，若归一化则再除以 $k^2$。
2. 核滑过整幅图像，每个位置用核内像素的平均值代替中心像素。
3. 输出图像即为平滑结果。

特点：
- 计算简单、速度快（可利用积分图加速）。
- 缺点：会产生**块状模糊**，对边缘细节破坏较大。

**OpenCV 提供**：
- `cv2.blur()`：标准均值滤波（等价于归一化的 boxFilter）。
- `cv2.boxFilter()`：可指定 `normalize` 参数，为 `False` 时直接求和（数值可能溢出）。

#### 一、核就是一个"全 1 矩阵"

盒式滤波的核（kernel）是最简单的——**所有元素都是 1**：

```text
k×k 全 1 核（以 3×3 为例）
┌─────────┐
│ 1  1  1 │
│ 1  1  1 │
│ 1  1  1 │
└─────────┘
```

它不区分中心还是边缘，邻域内每个像素"投票权"完全相等。这一点和**高斯核**（中心权重大、边缘权重小）形成鲜明对比。

#### 二、`cv2.blur()` 到底做了什么

`cv2.blur(src, (5, 5))` 本质等价于：

1. 取一个 5×5 的全 1 核；
2. **除以 25**（核内元素个数）做归一化，得到每个元素都是 `1/25` 的核；
3. 用这个核在图像上滑动，做"乘加求和"。

**归一化后的核**：

```text
5×5 归一化核（每个元素 = 1/25 = 0.04）
┌───────────────────────┐
│ .04 .04 .04 .04 .04  │
│ .04 .04 .04 .04 .04  │
│ .04 .04 .04 .04 .04  │
│ .04 .04 .04 .04 .04  │
│ .04 .04 .04 .04 .04  │
└───────────────────────┘
```

**单像素计算示例**：

假设图像某处 5×5 邻域灰度如下，核中心对准中间那个 `50`：

```text
图像邻域 (5×5)              核 (5×5, 归一化后)
┌─────────────────────┐    ┌─────────────────────┐
│ 10  12  14  16  18  │    │ .04 .04 .04 .04 .04 │
│ 20  22  24  26  28  │    │ .04 .04 .04 .04 .04 │
│ 30  32  50  34  36  │    │ .04 .04 .04 .04 .04 │
│ 38  40  42  44  46  │    │ .04 .04 .04 .04 .04 │
│ 48  50  52  54  56  │    │ .04 .04 .04 .04 .04 │
└─────────────────────┘    └─────────────────────┘
```

计算：

```text
新值 = 10×.04 + 12×.04 + 14×.04 + ... + 56×.04
     = (10 + 12 + 14 + ... + 56) × .04
     = 求和 × 1/25
     = 平均值
```

**关键点**：因为核内每个权重都相等（都是 `1/25`），所以"乘加求和"其实就退化成"先求和、再除以 25"——也就是**求算术平均**。

这就是为什么它叫"均值滤波"：**输出 = 邻域内所有像素的算术平均值**。

#### 三、`cv2.boxFilter()` 的 `normalize` 参数

`boxFilter` 把"是否除以元素个数"这个动作拆出来，让你自己决定：

```python
cv2.boxFilter(src, ddepth, ksize, normalize=True)   # 等价于 cv2.blur
cv2.boxFilter(src, ddepth, ksize, normalize=False)  # 只求和，不除
```

**情况 1：`normalize=True`**

核 = 全 1 矩阵 ÷ (k×k)：

```text
3×3，normalize=True
┌─────────────────┐
│ 1/9 1/9 1/9     │
│ 1/9 1/9 1/9     │
│ 1/9 1/9 1/9     │
└─────────────────┘
```

输出 = 邻域和 ÷ 9 = **平均值**。

这是**正常使用**的情况——输出亮度和原图保持一致，图像只是变模糊，不会整体变亮或变暗。

**情况 2：`normalize=False`**

核 = 原封不动的全 1 矩阵：

```text
3×3，normalize=False
┌─────────┐
│ 1  1  1 │
│ 1  1  1 │
│ 1  1  1 │
└─────────┘
```

输出 = 邻域内 9 个像素**直接相加**，**不除以 9**。

**用具体数字看差异**：

假设某 3×3 邻域全是最亮值 255：

| 模式 | 计算 | 结果 | 能否存入 uint8（0~255） |
|------|------|------|----------------------|
| normalize=True | 255×9 ÷ 9 | 255 | ✅ 正常 |
| normalize=False | 255×9 | 2295 | ❌ 溢出！ |

**2295 远超 uint8 上限 255**。如果 `ddepth=-1`（输出用 uint8），OpenCV 会做**饱和截断**（saturate_cast），2295 被截成 255。整张图凡是平均亮度超过 28 的区域（28×9≈252）都会变成纯白 —— 图像几乎全白，失去意义。

所以 `normalize=False` 只在**特殊场景**下用：

- 你想拿到的不是"平均亮度"，而是"邻域亮度的总和"（比如做积分图、局部能量、纹理度量）；
- 或者你手动指定了更大的 `ddepth`（如 `cv2.CV_32F`、`cv2.CV_64F`）来存放这些大数值，事后再统一缩放。

**一句话**：`normalize=False` 是"数学上的求和"，不是"视觉上的滤波"。

#### 四、`blur` 与 `boxFilter(normalize=True)` 完全等价

从数学上：

```text
cv2.blur(src, ksize)
  = cv2.boxFilter(src, -1, ksize, normalize=True)
  = 核为全 1/(k×k) 的 filter2D
```

三者输出**逐像素完全相同**（浮点误差范围内）。代码里两个函数并排展示，结果一致，正是为了印证这一点。

写代码时用哪个都行：

- 想要"均值滤波"语义清晰 → 用 `cv2.blur()`；
- 想显式控制归一化 / 或要非归一化求和 → 用 `cv2.boxFilter()`。

#### 五、边界怎么处理

核滑动到图像边缘时，会有一部分"探出"图像之外。OpenCV 默认用 **`BORDER_REFLECT_101`**（镜像反射，不重复边界像素）来补这些探出去的像素，然后照常做乘加。

- `cv2.blur()` / `cv2.boxFilter()` 内部默认 `borderType = cv2.BORDER_DEFAULT`（就是 `BORDER_REFLECT_101`）。
- 也可以用 `cv2.copyMakeBorder` 手动先补边，效果等价。

这就解释了为什么平滑后**图像边缘不会变黑**：边界像素不是"用 0 补"（那会拉低边缘亮度），而是"镜像自己"补进来，值域连续。

> **注意术语**：这里说的"图像边缘"指**图像最外圈的边框**（上下左右四条边），不是 6.0 节讲的"画面内部的轮廓/交界"。两个"边缘"含义不同，别混淆。

#### 六、为什么叫"盒式"（box）

因为核的形状是一个**方方正正、内部平坦的盒子**：

- 权重处处相等 → 像一块平板 / 一个盒子，没有高低起伏；
- 对比高斯核：中间鼓、四周低，像个小山包。

"盒式"这个名字形象地描述了核的**平顶**特征。

#### 七、复杂度与加速

朴素实现：每个像素做 k² 次乘加，全图 O(W·H·k²)。

实际优化：**积分图（Summed Area Table）** 让每个像素的邻域和 O(1) 拿到，全图降到 O(W·H)，与核大小无关。所以盒式滤波**核再大也不慢**——这是它相对高斯滤波的一大优势（高斯核大了虽然可分离，但仍是 O(k) 每像素）。

#### 八、总结表

| 项目 | `cv2.blur` | `cv2.boxFilter(normalize=True)` | `cv2.boxFilter(normalize=False)` |
|------|-----------|-------------------------------|--------------------------------|
| 核 | 全 1/(k²) | 全 1/(k²) | 全 1 |
| 每像素运算 | 邻域平均 | 邻域平均 | 邻域求和 |
| 输出亮度 | 与原图一致 | 与原图一致 | 大幅增亮（易溢出） |
| 典型用途 | 去噪、模糊 | 同左（显式写法） | 局部能量、积分图、需自管 ddepth |
| 默认边界 | REFLECT_101 | REFLECT_101 | REFLECT_101 |
| 复杂度 | O(WH)，积分图加速 | 同左 | 同左 |

**一句话记**：盒式滤波 = 全 1 核滑动；归一化就是"除以核内元素个数"，把"求和"变成"求平均"；`blur` 就是归一化版的 `boxFilter`，两者数学上完全等价。

#### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

img = cv2.imread('P&V/cat.jpg')                              # 读取图片
if img is not None:                                          # 如果图片存在
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)         # 转换为灰度图
    blur_img = cv2.blur(img_gray, (5, 5))                    # 均值滤波
    box_img = cv2.boxFilter(img_gray, -1, (5, 5), normalize=True)  # 盒式滤波
    
    plt.figure(figsize=(15, 5))                              # 创建画布（3 张图，放宽）
    plt.subplot(1, 3, 1)                                     # 子图 1：原图
    plt.title('原灰度图')                                     # 标题
    plt.imshow(img_gray, cmap='gray')                        # 显示原图
    plt.axis('off')                                          # 关闭坐标轴
    
    plt.subplot(1, 3, 2)                                     # 子图 2
    plt.title('blur (均值滤波)')                              # 标题
    plt.imshow(blur_img, cmap='gray')                        # 显示 blur 结果
    plt.axis('off')                                          # 关闭坐标轴
    
    plt.subplot(1, 3, 3)                                     # 子图 3
    plt.title('boxFilter (归一化)')                           # 标题
    plt.imshow(box_img, cmap='gray')                         # 显示 boxFilter 结果
    plt.axis('off')                                          # 关闭坐标轴
    plt.show()                                               # 显示图像
else:
    print("请确保 'P&V/cat.jpg' 存在")                       # 提示信息

**图像解读**

一、三张子图分别是什么

| 子图 | 内容 | 对应代码 |
|------|------|---------|
| 左：原灰度图 | 输入的猫图（灰度） | `img_gray` |
| 中：blur（均值滤波） | `cv2.blur(img_gray, (5, 5))` | 归一化盒式核，邻域平均 |
| 右：boxFilter（归一化） | `cv2.boxFilter(img_gray, -1, (5, 5), normalize=True)` | 归一化盒式核，邻域平均 |

---

二、观察到的现象

1. **中图和右图看起来完全一样**：都是原图被柔和地模糊了一遍，猫的轮廓、耳朵、斑纹边缘都变模糊，但整体亮度、结构保持一致。
2. **与左图对比**：中、右两图明显比原图更"糊"——毛发、地砖纹理、胡须等细节被抹平，说明 5×5 盒式平滑确实生效了。
3. **中图和右图之间**：肉眼**看不出任何差异**（像素级上也完全一致，因为两者数学等价）。

---

三、这说明了什么

1. `blur` = `boxFilter(normalize=True)`

从数学上：

```text
cv2.blur(src, (5,5))
  = cv2.boxFilter(src, -1, (5,5), normalize=True)
  = 核为全 1/25 的 filter2D
```

三者输出**逐像素完全相同**。代码里两个函数并排展示、结果一致，正是为了印证这一点。

2. 核长什么样

`cv2.blur(img_gray, (5, 5))` 内部用的核，归一化后每个元素都是 `1/25 = 0.04`：

|  |  |  |  |  |
|---|---|---|---|---|
| 0.04 | 0.04 | 0.04 | 0.04 | 0.04 |
| 0.04 | 0.04 | 0.04 | 0.04 | 0.04 |
| 0.04 | 0.04 | 0.04 | 0.04 | 0.04 |
| 0.04 | 0.04 | 0.04 | 0.04 | 0.04 |
| 0.04 | 0.04 | 0.04 | 0.04 | 0.04 |

每个像素输出 = 邻域 25 个像素的算术平均值。

3. 为什么视觉上"看不出区别"

- `blur` 和 `boxFilter(normalize=True)` 在**数学上严格等价**；
- 都走 OpenCV 内部同一套 `boxFilter` 实现，浮点误差为零（或极小）；
- 因此两图在肉眼和像素层面都完全一致。

4. 为什么原图看起来"更清晰"

- 原图保留了所有高频细节（毛发、胡须、地砖纹理、斑纹边界）；
- 5×5 盒式核把邻域 25 个像素做了平均 → 高频被低通 → 细节被抹平，视觉变"糊"；
- 核越大，平均范围越广，模糊越明显（可对比 3×3 / 5×5 / 11×11）。

5. 和 6.2 可分离滤波示例的对照

- 6.2 用高斯核：`sepFilter2D` ≈ `GaussianBlur`，两张图一致；
- 本节用盒式核：`blur` ≈ `boxFilter(normalize=True)`，两张图一致；
- 两个示例的结构相同（原图 + 两种等价写法），共同说明"等价实现在视觉上完全一致"。

---

四、小结

> 这张图用"中图和右图完全一致"这一事实，直观地验证了 **"blur = boxFilter(normalize=True)"** 这个知识点。

- 左图是参照，保留原图细节；
- 中图：`cv2.blur()`，内部就是归一化盒式核；
- 右图：`cv2.boxFilter(normalize=True)`，显式写法；
- 中 = 右 → 等价性得证；
- 中、右 vs 左 → 5×5 均值滤波的平滑效果可见。

### 6.4.2 高斯滤波器：GaussianBlur

**高斯滤波**用一个二维高斯函数生成的核做加权平均：离中心越近，权重越大；离中心越远，权重越小。

**算法过程**：
1. 根据核大小和标准差 $\sigma$ 计算二维高斯核：
   $$G(x, y) = \frac{1}{2\pi\sigma^2} e^{-(x^2 + y^2)/(2\sigma^2)}$$
2. 归一化核使其和为 1。
3. 使用该核进行相关/卷积运算。

特点：
- 平滑效果比均值滤波更自然，不会产生明显的块状。
- 可分离（水平 + 垂直），实现效率高。
- 广泛用于预处理，如去噪、Canny 边缘检测前处理。

```python
cv2.GaussianBlur(src, ksize, sigmaX, sigmaY)
```
- `ksize`：核大小，必须为奇数。
- `sigmaX`：X 方向标准差。
- `sigmaY`：Y 方向标准差，为 0 时等于 sigmaX。

#### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

img = cv2.imread('P&V/cat.jpg')                              # 读取图片
if img is not None:                                          # 如果图片存在
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)         # 转换为灰度图
    gauss_img = cv2.GaussianBlur(img_gray, (5, 5), 1.5)      # 高斯模糊
    
    plt.figure(figsize=(10, 5))                              # 创建画布
    plt.subplot(1, 2, 1)                                     # 子图 1
    plt.title('原灰度图')                                     # 标题
    plt.imshow(img_gray, cmap='gray')                        # 显示原灰度图
    plt.axis('off')                                          # 关闭坐标轴
    
    plt.subplot(1, 2, 2)                                     # 子图 2
    plt.title('GaussianBlur')                                # 标题
    plt.imshow(gauss_img, cmap='gray')                       # 显示高斯模糊结果
    plt.axis('off')                                          # 关闭坐标轴
    plt.show()                                               # 显示图像
else:
    print("请确保 'P&V/cat.jpg' 存在")                       # 提示信息

**图像解读**

一、两张子图分别是什么

| 子图 | 内容 | 对应代码 |
|------|------|---------|
| 左：原灰度图 | 输入的猫图（灰度） | `img_gray` |
| 右：GaussianBlur | 5×5 高斯核，σ=1.5 | `cv2.GaussianBlur(img_gray, (5, 5), 1.5)` |

---

二、观察到的现象

1. **右图整体比左图柔和**：猫的轮廓、耳朵边缘、胡须、地砖纹理都变模糊，但整体亮度、姿态、结构保持一致。
2. **模糊是"平滑"而不是"块状"**：与 5×5 盒式滤波（均值）相比，高斯模糊没有出现方块状、台阶状的边界，过渡更自然。
3. **边缘处过渡更细腻**：猫脸、耳朵等高频区域被压平，但大结构（身体、黑斑、尾巴）仍清晰可辨。

---

三、这说明了什么

1. 高斯核的权重是"中间大、四周小"

`cv2.GaussianBlur(img_gray, (5, 5), 1.5)` 内部用的核（近似）：

|  |  |  |  |  |
|---|---|---|---|---|
| 0.014 | 0.028 | 0.035 | 0.028 | 0.014 |
| 0.028 | 0.056 | 0.070 | 0.056 | 0.028 |
| 0.035 | 0.070 | 0.087 | 0.070 | 0.035 |
| 0.028 | 0.056 | 0.070 | 0.056 | 0.028 |
| 0.014 | 0.028 | 0.035 | 0.028 | 0.014 |

- 中心权重最大（0.087），越往外越小；
- 每个像素输出 = 邻域 25 个像素的**加权平均**，中心像素贡献最大。

2. 为什么比均值滤波更"自然"

| 滤波 | 核权重 | 视觉效果 |
|------|--------|---------|
| 均值滤波（6.4.1） | 处处相等（全 1/25） | 出现块状模糊，边缘过渡生硬 |
| 高斯滤波（本节） | 中间大、四周小 | 平滑自然，无块状 |

原因：高斯核相当于"以中心像素为主、周围像素为辅"的加权平均，与人类视觉对"模糊"的直觉一致；均值核则把中心像素和边缘像素同等对待，容易产生方形伪影。

3. 为什么高斯核可分离

5×5 高斯核可以拆成两个一维高斯核的外积：

K_2D = g · gᵀ

其中 g = [0.014, 0.028, 0.035, 0.028, 0.014] 是一维高斯向量。

所以 `cv2.GaussianBlur` 内部实际是**先横后竖两次一维卷积**（见 6.2 节可分离卷积核），比直接二维卷积更快。

4. 参数 ksize 和 sigma 的影响

- `ksize` 越大 → 核覆盖范围越广 → 模糊越强；
- `sigma` 越大 → 高斯分布越"胖" → 权重更分散 → 模糊越强；
- 当 `sigmaY=0`（默认）时，sigmaY 自动等于 sigmaX，保证横竖两个方向模糊程度一致。

本示例用 `ksize=(5,5), sigmaX=1.5`，属于中等强度模糊，能明显压平细节但不至于丢失大结构。

---

四、小结

> 这张图直观展示了 **"高斯模糊 = 用二维高斯核做加权平均"**：中心权重最大、四周权重小，所以平滑自然、无块状伪影。

- 左图是参照，保留原图细节；
- 右图：`cv2.GaussianBlur(img_gray, (5, 5), 1.5)`，5×5 高斯核，σ=1.5；
- 右 vs 左 → 细节被柔和抹平，大结构保留；
- 与 6.4.1 均值滤波对比 → 高斯模糊更自然、无方块感。

### 6.4.3 统计排序滤波器：中值、最大值、最小值、中点、修正阿尔法均值

**统计排序滤波器**不是加权求和，而是对核内像素进行排序后取某个位置的值作为输出。

常见类型：
- **中值滤波**：取排序后的中间值，对**椒盐噪声**特别有效。
- **最大值滤波**：取最大值，可去除暗点、保留亮点（灰度膨胀等价）。
- **最小值滤波**：取最小值，可去除亮点、保留暗点（灰度腐蚀等价）。
- **中点滤波**：取最大值与最小值的平均，适合高斯/均匀噪声。
- **修正阿尔法均值滤波**：去掉 $d$ 个最小和 $d$ 个最大值后求平均，兼具均值和中值特性。

**算法过程**：
1. 取当前像素的邻域窗口。
2. 对窗口内所有像素排序。
3. 根据滤波类型选择位置或组合取值。
4. 将该值作为中心像素的输出。

**OpenCV 提供 `cv2.medianBlur()`**；其他类型可用 `np.max`、`np.min`、`np.sort` 手动实现。

#### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

img = cv2.imread('P&V/cat.jpg')                              # 读取图片
if img is not None:                                          # 如果图片存在
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)         # 转换为灰度图
    noisy_img = img_gray.copy()                              # 复制灰度图
    prob = 0.02                                              # 噪声比例
    num_salt = np.ceil(prob * img_gray.size * 0.5).astype(int)  # 盐噪声数量
    coords = [np.random.randint(0, i - 1, num_salt) for i in img_gray.shape]  # 随机坐标
    noisy_img[tuple(coords)] = 255                           # 添加盐噪声
    num_pepper = np.ceil(prob * img_gray.size * 0.5).astype(int)  # 椒噪声数量
    coords = [np.random.randint(0, i - 1, num_pepper) for i in img_gray.shape]  # 随机坐标
    noisy_img[tuple(coords)] = 0                             # 添加椒噪声
    
    # --- ① 中值滤波（OpenCV 内置，取排序中间值）---
    median_img = cv2.medianBlur(noisy_img, 5)

    # --- ② 最大值滤波（取窗口最大值，去除暗点）---
    kernel = np.ones((5, 5), np.uint8)                       # 5×5 全 1 结构元
    max_img = cv2.dilate(noisy_img, kernel)                  # 灰度膨胀 = 最大值滤波

    # --- ③ 最小值滤波（取窗口最小值，去除亮点）---
    min_img = cv2.erode(noisy_img, kernel)                   # 灰度腐蚀 = 最小值滤波

    # --- ④ 中点滤波（(max + min) / 2）---
    mid_img = cv2.addWeighted(max_img, 0.5, min_img, 0.5, 0) # 最大值和最小值各取一半

    # --- ⑤ 修正阿尔法均值滤波（去掉 d 个最小、d 个最大后求平均）---
    # 说明：用 (2d+1) 半径的窗口太慢，这里用 5×5 窗口、d=4（去 4 小 4 大，剩 17 个求平均）
    def alpha_trimmed_mean(img, ksize=5, d=4):
        pad = ksize // 2                                     # 窗口半径
        padded = cv2.copyMakeBorder(img, pad, pad, pad, pad, cv2.BORDER_REFLECT_101)  # 边界扩充
        h, w = img.shape                                     # 原始尺寸
        out = np.zeros_like(img, dtype=np.uint8)             # 输出图像
        for i in range(h):                                   # 遍历行
            for j in range(w):                               # 遍历列
                window = padded[i:i + ksize, j:j + ksize]    # 取窗口
                vals = np.sort(window.flatten())             # 排序
                trimmed = vals[d:ksize * ksize - d]          # 去掉 d 小 d 大
                out[i, j] = int(np.mean(trimmed))            # 求平均
        return out

    alpha_img = alpha_trimmed_mean(noisy_img, ksize=5, d=4)  # 修正阿尔法均值滤波

    plt.figure(figsize=(20, 8))                              # 创建画布（7 张图，2 行 4 列）
    titles = ['原图', '椒盐噪声', '中值滤波', '最大值滤波',
              '最小值滤波', '中点滤波', '修正阿尔法均值滤波']  # 标题列表
    images = [img_gray, noisy_img, median_img, max_img,
              min_img, mid_img, alpha_img]                   # 图像列表
    for i in range(7):                                       # 遍历
        plt.subplot(2, 4, i + 1)                             # 子图
        plt.title(titles[i])                                 # 标题
        plt.imshow(images[i], cmap='gray')                   # 显示图像
        plt.axis('off')                                      # 关闭坐标轴
    plt.show()                                               # 显示图像
else:
    print("请确保 'P&V/cat.jpg' 存在")                       # 提示信息

**图像解读**

一、7 张子图分别是什么

| 子图 | 输入 | 输出 | 对应代码 |
|------|------|------|---------|
| 原图 | `P&V/cat.jpg` | `img_gray` | `cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)` |
| 椒盐噪声 | `img_gray` | `noisy_img` | 随机位置置 255（盐）/ 0（椒） |
| 中值滤波 | **`noisy_img`** | `median_img` | `cv2.medianBlur(noisy_img, 5)` |
| 最大值滤波 | **`noisy_img`** | `max_img` | `cv2.dilate(noisy_img, kernel)` |
| 最小值滤波 | **`noisy_img`** | `min_img` | `cv2.erode(noisy_img, kernel)` |
| 中点滤波 | **`noisy_img`** | `mid_img` | `cv2.addWeighted(max_img, 0.5, min_img, 0.5, 0)` |
| 修正阿尔法均值滤波 | **`noisy_img`** | `alpha_img` | 手写 `alpha_trimmed_mean(noisy_img, 5, 4)` |

**关键：5 种滤波的输入都是同一张图——`noisy_img`（椒盐噪声图）。原图只做参照，不参与滤波。**

---

二、三行逻辑

**原图 → 加噪声 → 用 5 种方法去噪**

- 原图：告诉你"未被污染前长什么样"，是参照；
- 椒盐噪声：告诉你"污染后长什么样"，是 5 种滤波**共同的输入**；
- 5 种滤波：对同一张噪声图，用不同策略处理，输出 5 张不同的结果图。

**为什么必须用同一张噪声图**：控制变量。如果 5 种滤波各用不同的图，结果就无法横向对比。只有输入相同，输出的差异才能归因于"滤波方法本身"。

---

三、逐图观察

**1. 中值滤波（去噪最强）**

- 输入：`noisy_img`；输出：几乎干净的猫图；
- 椒盐噪声**几乎完全消失**，猫的轮廓、黑斑、尾巴、地砖纹理都保留；
- 原因：排序后取中间值，0 和 255 排在最两端，直接被丢弃。

**2. 最大值滤波（整体变亮）**

- 输入：`noisy_img`；输出：整体偏亮的图；
- 黑点被周围的亮像素"吃掉"，白点反而扩张；
- 原因：`cv2.dilate` 取窗口最大值，0 被邻域的亮像素顶替；
- 副作用：亮区扩张、暗区收缩，图上大片区域变白。

**3. 最小值滤波（整体变暗）**

- 输入：`noisy_img`；输出：整体偏暗的图；
- 白点被周围的暗像素"吃掉"，黑点反而扩张；
- 原因：`cv2.erode` 取窗口最小值，255 被邻域的暗像素顶替；
- 副作用：暗区扩张、亮区收缩。

**4. 中点滤波（弱化但未去除）**

- 输入：`noisy_img`；输出：噪声变灰、图像变"脏"；
- 椒盐点没被剔除，而是被平均成中灰（(0+255)/2 = 127）；
- 原因：中点 = (max + min) / 2，两端极值都被保留在计算里；
- 中点滤波更适合**高斯 / 均匀噪声**，对脉冲噪声效果一般。

**5. 修正阿尔法均值滤波（接近中值）**

- 输入：`noisy_img`；输出：噪声明显减少，比中值柔和；
- 原因：5×5 共 25 个值，去掉 4 小 4 大后剩 17 个求平均；
- 比中值更适合**混合噪声**（椒盐 + 高斯），因为它不只取中间一个值。

---

四、5 种滤波器对比小结

| 滤波器 | 输入 | 椒盐噪声处理 | 亮度变化 | 适合场景 |
|--------|------|------------|---------|---------|
| 中值 | `noisy_img` | 几乎完全去除 | 基本不变 | 椒盐噪声（首选） |
| 最大值 | `noisy_img` | 白点扩张、黑点被填 | 变亮 | 去暗点、找亮点 |
| 最小值 | `noisy_img` | 黑点扩张、白点被填 | 变暗 | 去亮点、找暗点 |
| 中点 | `noisy_img` | 弱化为灰点，未去除 | 基本不变 | 高斯 / 均匀噪声 |
| 修正阿尔法均值 | `noisy_img` | 明显减少 | 基本不变 | 混合噪声（椒盐 + 高斯） |

---

五、为什么用椒盐噪声

**椒盐噪声是什么**：图像里随机出现一些**纯白点（盐，255）**和**纯黑点（椒，0）**，像撒了盐和胡椒粉。

**它和 6.4.3 节的关系**：

- 统计排序滤波器的差异，只有在**存在极端值（0 / 255）**时才会显现；
- 用原图直接跑这 5 种滤波，结果几乎一样，看不出区别；
- 椒盐噪声制造了 0 和 255 两个极端，让"取中间、取最大、取最小、取平均"的不同处理方式产生**可视化差异**；
- 所以椒盐噪声是这一节的"试金石"，不是装饰。

**为什么不用高斯噪声**：

- 高斯噪声是小幅连续扰动，没有 0/255 的极端值；
- 中值/最大值/最小值对小幅扰动不敏感，结果差异不明显；
- 统计排序滤波器最经典的用武之地就是**椒盐噪声**，所以用它做演示最合适。

---

六、一句话总结

> 同一张椒盐噪声图 `noisy_img`，5 种统计排序滤波器给出 5 种不同结果：**中值最干净、最大值变亮、最小值变暗、中点变灰、修正阿尔法均值介于中值与均值之间**。原图只做参照，噪声图才是 5 种滤波共同的输入——这正是 6.4.3 节演示的关键。

### 6.4.4 自适应滤波器：自适应局部降噪、自适应中值滤波

**自适应滤波器**根据图像**局部统计特性**动态调整滤波参数，比固定参数的滤波更智能。

固定参数的滤波（均值、高斯、中值）在整张图上用同一套规则：平坦区和边缘区同等对待，结果要么平滑不足、要么细节被误伤。**自适应滤波的核心思路是"看情况办事"**——平坦区大胆平滑，边缘区尽量保留；噪声稀疏时用小窗口，噪声密集时扩大窗口。

> **术语复习**：这里说的"**边缘区**"指"当前窗口跨在一条边缘（画面内跳变）两侧"；"**平坦区**"指"窗口里全是差不多的值"。参见 6.0 节对"边缘"的定义。

**关于"窗口大小要不要自己设"**：两种滤波都要，但策略不同——

- **自适应局部降噪**：窗口大小**固定**（如 5×5），一旦设定就不变，自适应体现在**权值**（输出更靠近均值还是原像素）；除窗口大小外，还需提供**噪声方差** $\sigma_\eta^2$ 作为先验。
- **自适应中值滤波**：窗口**动态扩大**（3→5→7...），需要设**最大上限** $S_{max}$；初始窗口一般固定为 3×3，不用改。

所以"自适应"不是"什么都不用设"，而是把"每个像素都调参"变成了"设一两个全局参数"，剩下的交给算法按局部统计量自动决策。

---

#### 一、自适应局部降噪（Adaptive Local Noise Reduction）

##### 1.1 核心思想

用一个**局部窗口**，计算它的**局部均值** $\mu_L$ 和**局部方差** $\sigma_L^2$，再结合**已知的噪声方差** $\sigma_\eta^2$，动态决定输出像素更接近"局部均值"还是"原像素"。

##### 1.2 公式

$$
\hat{f}(x, y) = \mu_L + \frac{\sigma_L^2 - \sigma_\eta^2}{\sigma_L^2} \cdot \big( g(x, y) - \mu_L \big)
$$

- $g(x,y)$：噪声图当前像素值；
- $\mu_L$：局部窗口均值；
- $\sigma_L^2$：局部窗口方差；
- $\sigma_\eta^2$：**整幅图估计的噪声方差**（作为已知量）；
- $\hat{f}(x,y)$：输出像素。

##### 1.3 两种极端情况

| 情况 | 局部方差 $\sigma_L^2$ 与噪声方差 $\sigma_\eta^2$ 的关系 | 公式里的比例 | 输出 |
|------|--------------------------------------------------|-----------|------|
| **平坦区域**（只有噪声，无真实结构） | $\sigma_L^2 \approx \sigma_\eta^2$ | 比例 ≈ 0 | $\hat{f} \approx \mu_L$，**全部用局部均值**（强平滑） |
| **边缘区域**（有真实结构变化） | $\sigma_L^2 \gg \sigma_\eta^2$ | 比例 ≈ 1 | $\hat{f} \approx g(x,y)$，**保留原像素**（少平滑） |

##### 1.4 直观理解

想象一个 3×3 窗口，分成两种情况：

**情况 1：平坦区**

真实值 = 200，加高斯噪声（±5）后，窗口里是 203、198、201、199……变化很小。

- 局部方差 ≈ 25（全部来自噪声）；
- 噪声方差 $\sigma_\eta^2$ ≈ 25；
- 比例 ≈ 0 → 输出 ≈ 局部均值 μ_L，大胆平滑。

**情况 2：边缘区**

真实值 = 100 突然跳到 200，加噪声后窗口里既有 100±5 也有 200±5。

- 局部方差很大（约 2500，结构差异贡献）；
- 噪声方差仍 ≈ 25；
- 比例 ≈ 1 → 输出 ≈ 原像素，保留边缘。

##### 1.5 为什么这样设计

关键在于"**观测到的局部方差 = 真实结构方差 + 噪声方差**"这一关系：

| 观测 | 反推 | 策略 |
|------|------|------|
| $\sigma_L^2 \approx \sigma_\eta^2$ | 方差全来自噪声，说明此处平坦 | 大胆用均值 $\mu_L$ 平滑 |
| $\sigma_L^2 \gg \sigma_\eta^2$ | 方差主要来自结构，说明此处有边缘 | 保留原像素 $g(x,y)$ |

**一句话**：平坦区以噪声为主（信号本身不变），边缘区以结构为主（信号本身跳变），滤波器据此判断该平滑还是保留。

##### 1.6 参数设置

**窗口大小 `ksize`（固定）**：

| 窗口 | 效果 |
|------|------|
| 3×3 | 保细节好，但方差估计不稳（样本太少），平滑力度弱 |
| 5×5 | 常用折中，方差估计较稳，平滑适中 |
| 7×7 或更大 | 平滑强，但边缘也会被波及，计算量大 |

**实践建议**：5×5 起步。噪声重就用 7×7，噪声轻就用 3×3。

**噪声方差 $\sigma_\eta^2$（先验）**：

- 估小了 → 把边缘误判成平坦区 → 边缘被糊；
- 估大了 → 把平坦区误判成边缘 → 噪声没去干净；
- **估计方法**：取图里一块本来应该均匀的区域（比如地砖），算它的方差，作为 $\sigma_\eta^2$。

##### 1.7 使用条件与优缺点

- **需要预先估计噪声方差** $\sigma_\eta^2$（常用方法：取一块平坦区域，算它的方差作为估计）；
- 若 $\sigma_\eta^2$ 估计不准，效果会打折扣；
- 优点：对**高斯噪声**效果好，且能保留边缘；
- 缺点：需要先验知识（噪声方差），不适合椒盐等脉冲噪声。

---

#### 二、自适应中值滤波（Adaptive Median Filter, AMF）

##### 2.1 核心思想

中值滤波的窗口大小是**固定**的（比如一直用 5×5），问题在于：

- 噪声稀疏时，5×5 窗口偏大 → 过度平滑细节；
- 噪声密集时，5×5 窗口偏小 → 去不干净。

**自适应中值滤波**的做法：**窗口从 3×3 开始，不够就扩大**；同时判断"当前像素是不是噪声"，只有确认是噪声才替换。

##### 2.2 变量定义

- $z_{min}$：窗口最小值；
- $z_{max}$：窗口最大值；
- $z_{med}$：窗口中值；
- $z_{xy}$：窗口中心像素（当前像素）；
- $S_{max}$：允许的最大窗口尺寸（如 7 或 9）。

##### 2.3 两个层次（A 层和 B 层）

**A 层：判断"中值是否可信"**

- 若 $z_{min} < z_{med} < z_{max}$ → 中值不是极值，可信，转 B 层；
- 否则（中值本身是极值，说明窗口里噪声太多）→ **扩大窗口**，$S \leftarrow S + 2$；
- 若窗口已超过 $S_{max}$ → 直接输出 $z_{med}$（尽力了，用中值）。

**B 层：判断"当前像素是不是噪声"**

- 若 $z_{min} < z_{xy} < z_{max}$ → 当前像素不是极值，**保留原值 $z_{xy}$**；
- 否则（当前像素是极值，很可能是噪声）→ **用中值 $z_{med}$ 替换它**。

##### 2.4 算法过程

1. 初始化窗口大小 $S = 3$。
2. 计算窗口的 $z_{min}, z_{max}, z_{med}$ 和中心像素 $z_{xy}$。
3. **A 层判断**：若 $z_{min} < z_{med} < z_{max}$，转步骤 4；否则 $S \leftarrow S + 2$，若 $S > S_{max}$ 输出 $z_{med}$，否则重复步骤 2。
4. **B 层判断**：若 $z_{min} < z_{xy} < z_{max}$，输出 $z_{xy}$；否则输出 $z_{med}$。

##### 2.5 流程图

```
取窗口大小 S = 3
    ↓
计算 z_min, z_max, z_med, z_xy
    ↓
A 层：z_min < z_med < z_max ?
    ├─ 否 → S = S + 2；若 S > S_max → 输出 z_med
    │           否则回到"取窗口"
    └─ 是 ↓
B 层：z_min < z_xy < z_max ?
    ├─ 是 → 输出 z_xy（保留原值）
    └─ 否 → 输出 z_med（替换为中值）
```

##### 2.6 为什么要分两层

- **A 层的意义**：先确认"窗口中值是否可靠"。若窗口里噪声太多（中值自己都成了极值），说明窗口太小，得扩大；
- **B 层的意义**：只有确认"当前像素是极值"（可能是噪声）才替换它；若当前像素是正常值，就**保留原样**，避免误伤细节。

**对比固定中值**：

| 滤波 | 策略 | 结果 |
|------|------|------|
| 固定中值 | 无脑把每个像素都替换成窗口中值 | 细节可能被抹掉 |
| 自适应中值 | 只有确认当前像素是噪声才替换，否则保留原值 | 细节保留更好 |

##### 2.7 参数设置

**初始窗口大小**：一般固定为 **3×3**（最小奇数窗口），不需要改。

**最大窗口上限 $S_{max}$**（唯一可调参数）：

| $S_{max}$ | 效果 | 代价 |
|-----------|------|------|
| 3 | 只能处理稀疏噪声；噪声密时去不净 | 最快 |
| 5 | 中等噪声够用 | 中等 |
| 7 | 常用值，稠密椒盐噪声也能处理 | 较慢 |
| 9 或 11 | 噪声极密时用 | 很慢（每像素多次排序） |
| 无上限 | 图像小时可能跑得完，图像大时可能卡死 | 极慢，不推荐 |

**怎么选**：

- 看噪声密度：噪声越多，$S_{max}$ 越大；
- 看速度要求：$S_{max}$ 每 +2，平均排序次数翻倍以上；
- 常用经验值：**$S_{max} = 7$**，能处理大多数椒盐噪声场景。

##### 2.8 优势与缺点

**优势**：

- **去噪彻底**：噪声密集时，窗口自动扩大；
- **保细节**：像素若不是噪声，保留原值，不会像固定中值那样把每个点都替换；
- **适合椒盐 + 混合噪声**：对脉冲噪声和混合噪声都很稳。

**缺点**：

- **速度慢**：每个像素要多次排序（窗口逐次扩大），比固定中值慢很多；
- **需要设 $S_{max}$**：$S_{max}$ 太大浪费算力，太小去不干净噪声。

---

#### 三、两种自适应滤波的对比

| 项目 | 自适应局部降噪 | 自适应中值滤波 |
|------|--------------|----------|
| 针对噪声 | 高斯、加性噪声 | 椒盐、脉冲、混合噪声 |
| 依据的局部统计量 | 局部均值 $\mu_L$、局部方差 $\sigma_L^2$ | 局部极值 $z_{min}, z_{max}$、中值 $z_{med}$ |
| 窗口大小 | **固定**（常用 5×5） | **动态扩大**（3→5→7...） |
| 要手动设的参数 | 窗口大小 `ksize` + 噪声方差 $\sigma_\eta^2$ | 最大上限 $S_{max}$（初始 3×3 一般不改） |
| 是否需要先验 | 需要**已知噪声方差** $\sigma_\eta^2$ | 不需要，只需 $S_{max}$ |
| 核心策略 | 平坦就平滑、边缘就保留 | 是噪声就替换、不是就保留 |
| 速度 | 较快 | 较慢 |
| 典型用途 | 医学图像去噪 | 椒盐 / 混合噪声去除 |

---

#### 四、参数设置速查表

| 参数 | 自适应局部降噪 | 自适应中值滤波 |
|------|--------------|--------------|
| 窗口大小 | 固定，常用 `ksize=5` | 从 3×3 起，动态扩大 |
| 要手动设 | `ksize` | 初始 3×3（不改）+ 上限 $S_{max}$ |
| 另一个必需参数 | 噪声方差 $\sigma_\eta^2$ | 无 |
| 推荐初值 | `ksize=5`，`noise_var` 取一块平坦区方差 | `max_window_size=7` |
| 参数影响 | ksize 大 → 更平滑；$\sigma_\eta^2$ 估错 → 效果差 | $S_{max}$ 大 → 去噪更强、更慢 |

---

#### 五、一句话总结

> **自适应滤波的核心思想是"看情况办事"**：
>
> - **自适应局部降噪**：比较"局部方差"和"噪声方差"——平坦区用均值平滑，边缘区保留原像素；窗口大小**固定**（常用 5×5），额外要给一个噪声方差作先验；
> - **自适应中值滤波**：窗口从 3×3 起**动态扩大**——先判断中值是否可信，再判断当前像素是否为噪声，只对噪声点做替换；你只需给一个**最大上限 $S_{max}$**（常用 7）。
>
> 两者都比固定参数滤波更智能，代价是**计算量更大**、需要**额外的参数估计**（噪声方差或最大窗口）。"自适应"不是"什么都不用设"，而是把"每个像素都调参"变成了"设一两个全局参数"，剩下的交给算法按局部统计量自动决策。

#### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

# ---------- 自适应中值滤波（AMF） ----------
def adaptive_median_filter(img, max_window_size=7):          # 自适应中值滤波函数
    result = img.copy()                                      # 复制原图
    h, w = img.shape                                         # 获取高宽
    for i in range(h):                                       # 遍历行
        for j in range(w):                                   # 遍历列
            window_size = 3                                  # 初始窗口
            z_med = img[i, j]                                # 初始化中值
            while window_size <= max_window_size:            # 窗口大小限制
                r = window_size // 2                         # 半径
                r0, r1 = max(0, i - r), min(h, i + r + 1)    # 行范围
                c0, c1 = max(0, j - r), min(w, j + r + 1)    # 列范围
                window = img[r0:r1, c0:c1]                   # 窗口
                z_min, z_max = np.min(window), np.max(window)  # 极值
                z_med = np.median(window)                    # 中值
                z_xy = img[i, j]                             # 当前像素
                if z_min < z_med < z_max:                    # A 层：中值可信
                    if z_min < z_xy < z_max:                 # B 层：当前像素不是极值
                        result[i, j] = z_xy                  # 保留原值
                    else:
                        result[i, j] = z_med                 # 替换为中值
                    break                                    # 跳出
                else:
                    window_size += 2                         # 扩窗
            else:
                result[i, j] = z_med                         # 达到上限，用中值
    return result

# ---------- 自适应局部降噪 ----------
def adaptive_local_denoise(img, ksize=5, noise_var=100):     # 自适应局部降噪
    pad = ksize // 2                                         # 边界半径
    padded = cv2.copyMakeBorder(img, pad, pad, pad, pad,
                                cv2.BORDER_REFLECT_101)      # 边界扩充
    out = np.zeros_like(img, dtype=np.float32)               # 输出
    for i in range(img.shape[0]):                            # 遍历行
        for j in range(img.shape[1]):                        # 遍历列
            win = padded[i:i + ksize, j:j + ksize]           # 窗口
            mu = win.mean()                                  # 局部均值
            var = win.var()                                  # 局部方差
            ratio = max(0.0, (var - noise_var) / (var + 1e-6))  # 比例
            out[i, j] = mu + ratio * (img[i, j] - mu)        # 公式
    return np.clip(out, 0, 255).astype(np.uint8)             # 截断

img = cv2.imread('P&V/cat.jpg')                              # 读取图片
if img is not None:                                          # 如果图片存在
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)         # 灰度图
    small = cv2.resize(img_gray, (120, 120))                 # 缩放（AMF 较慢）

    # ---- ① 高斯噪声 + 自适应局部降噪 ----
    gauss_noisy = np.clip(small.astype(np.float32) +
                          np.random.normal(0, 10, small.shape), 0, 255).astype(np.uint8)
    alr = adaptive_local_denoise(gauss_noisy, ksize=5, noise_var=100)

    # ---- ② 椒盐噪声 + 自适应中值滤波 ----
    sp_noisy = small.copy()                                  # 复制
    prob = 0.05                                              # 噪声比例
    n = np.ceil(prob * sp_noisy.size * 0.5).astype(int)      # 数量
    coords = [np.random.randint(0, s - 1, n) for s in sp_noisy.shape]
    sp_noisy[tuple(coords)] = 255                            # 盐
    coords = [np.random.randint(0, s - 1, n) for s in sp_noisy.shape]
    sp_noisy[tuple(coords)] = 0                              # 椒
    amf = adaptive_median_filter(sp_noisy, max_window_size=7)

    # ---- 展示：2 行 3 列，左下留空，下排中、右放椒盐噪声和 AMF ----
    plt.figure(figsize=(15, 8))                              # 画布
    titles = ['原图 (缩放)', '高斯噪声', '自适应局部降噪',       # 上排：原图 / 高斯噪声 / 自适应局部降噪
              '',            '椒盐噪声', '自适应中值滤波']      # 下排：空 / 椒盐噪声 / 自适应中值滤波
    images = [small, gauss_noisy, alr,                       # 上排图像
              None,  sp_noisy,    amf]                       # 下排图像（第一个为 None，跳过）
    for i in range(6):                                       # 遍历 6 个位置
        if images[i] is None:                                # 跳过空位
            continue
        plt.subplot(2, 3, i + 1)                             # 子图
        plt.title(titles[i])                                 # 标题
        plt.imshow(images[i], cmap='gray')                   # 显示
        plt.axis('off')                                      # 关闭坐标轴
    plt.show()                                               # 显示
else:
    print("请确保 'P&V/cat.jpg' 存在")                       # 提示信息

**图像解读**

一、5 张子图分别是什么

| 位置 | 子图 | 输入 | 输出/说明 | 对应代码 |
|------|------|------|----------|---------|
| 上排左 | 原图（缩放） | `P&V/cat.jpg` | `small` | 参照基准，不参与滤波 |
| 上排中 | 高斯噪声 | `small` | `gauss_noisy` | `small + N(0, 10)`，σ=10 |
| 上排右 | 自适应局部降噪 | **`gauss_noisy`** | `alr` | `adaptive_local_denoise(...)` |
| 下排中 | 椒盐噪声 | `small` | `sp_noisy` | 5% 像素随机置 0 或 255 |
| 下排右 | 自适应中值滤波 | **`sp_noisy`** | `amf` | `adaptive_median_filter(...)` |

**关键：上排和上排一组，下排和下排一组。两种滤波各有各的噪声图，不是同一张。**

- **自适应局部降噪** 的输入 = **高斯噪声图**；
- **自适应中值滤波** 的输入 = **椒盐噪声图**。

---

二、为什么两种滤波用不同的噪声图

**两种滤波的"专业对口"**

| 滤波 | 设计针对的噪声 | 依据的局部统计量 |
|------|--------------|----------------|
| 自适应局部降噪 | **高斯噪声**（小幅、连续扰动） | 局部均值 $\mu_L$、局部方差 $\sigma_L^2$ |
| 自适应中值滤波 | **椒盐噪声**（0 / 255 极端值） | 局部极值 $z_{min}, z_{max}$、中值 $z_{med}$ |

- **自适应局部降噪**：靠"局部方差 vs 噪声方差"判断平坦还是边缘。高斯噪声每个像素加 ±10，局部方差在噪声方差附近，滤波可以正确判断；
- **自适应中值滤波**：靠"排序后中值 vs 极值"判断噪声。椒盐噪声的 0 和 255 在排序后处于两端，中值可以避开。

**用错噪声会怎样**

**情况 1：自适应局部降噪去处理椒盐噪声**

椒盐噪声的 0 和 255 会**把局部方差拉爆**：

- 窗口里 5 个正常像素 + 4 个 0/255 极端值；
- 局部方差变得很大 → 公式比例 ≈ 1 → 输出 ≈ **原像素**；
- 结果：**该平滑的地方不平滑，椒盐点原封不动保留**。

自适应局部降噪会"以为"这里是边缘，不敢动。

**情况 2：自适应中值滤波去处理高斯噪声**

高斯噪声不产生 0 或 255 极值：

- 窗口里最大最小都是"正常值附近"；
- B 层判断 $z_{min} < z_{xy} < z_{max}$ 几乎总成立 → **保留原像素**；
- 结果：**几乎什么都不做，高斯噪声原样留下**。

自适应中值滤波"以为"每个像素都是正常值，不敢替换。

**分开用噪声，各自才能"对症下药"**

| 组合 | 输入 | 结果 |
|------|------|------|
| 高斯噪声 → 自适应局部降噪 | 平坦区平滑、边缘保留 | **效果好，能看出"自适应"的价值** |
| 椒盐噪声 → 自适应中值滤波 | 极端值被剔除，细节保留 | **效果好，能看出"自适应"的价值** |

**如果硬用同一种噪声**，会出现"一个滤波完胜、另一个一败涂地"的结果，读者会误以为后者"没用"——其实只是"用错了场景"。

**类比：用对工具**

| 工具 | 擅长 | 用错场景会怎样 |
|------|------|--------------|
| 抹布 | 擦干水渍（高斯噪声） | 拿它去砸钉子 → 砸不动 |
| 锤子 | 砸钉子（椒盐噪声） | 拿它去擦水渍 → 越擦越脏 |

自适应局部降噪是"抹布"，自适应中值滤波是"锤子"，**用对噪声图，才能看出两者的价值**。

**混合噪声怎么办**

如果图像同时有高斯噪声和椒盐噪声，两种滤波都不完美：

- 自适应局部降噪：椒盐点还在；
- 自适应中值滤波：高斯颗粒还在。

这时才需要"**修正阿尔法均值**"这类专门针对混合噪声的滤波（见 6.4.3 节）。

6.4.4 节选"高斯 + 自适应局部降噪"和"椒盐 + 自适应中值滤波"两组，**恰好覆盖了两种典型噪声类型**，不重叠、不留空。

---

三、逐图观察

**上排：高斯噪声 → 自适应局部降噪**

**高斯噪声**：

- 图像呈现**细密的颗粒感**，每个像素周围加了一个小幅随机扰动（σ=10）；
- 猫的轮廓、黑斑仍在，但整体变"毛"、不再干净；
- 地砖纹理被淹没在颗粒感里。

**自适应局部降噪**：

- 颗粒感**明显减弱**，猫身、地砖变得平滑；
- 猫的轮廓、耳朵边缘、黑斑边界**仍清晰保留**（这是"自适应"的关键）；
- 对比"高斯滤波"（如果跑一遍），普通高斯滤波会把边缘也糊掉，而自适应局部降噪**只平滑平坦区、保留边缘**。

**下排：椒盐噪声 → 自适应中值滤波**

**椒盐噪声**：

- 图像上随机分布**黑色点（椒，0）和白色点（盐，255）**；
- 噪声比例约 5%（白点 2.5% + 黑点 2.5%）；
- 噪声点明显突兀，破坏了图像的连续性。

**自适应中值滤波**：

- 椒盐噪声**几乎完全消失**，黑白点被剔除；
- 猫的轮廓、黑斑、地砖纹理**基本保留**；
- 对比"固定中值"（如果跑一遍），固定中值会把每个像素都替换成窗口中值，可能抹掉细节；自适应中值只对"确认是噪声"的像素替换，细节保留更好。

---

四、5 张图的核心逻辑链

```
原图 (缩放)
    ├─ 加高斯噪声 → 高斯噪声图 → 自适应局部降噪 → 干净图（平坦区平滑、边缘保留）
    └─ 加椒盐噪声 → 椒盐噪声图 → 自适应中值滤波 → 干净图（噪声点剔除、细节保留）
```

**两条支线、各配各的噪声、各打各的靶。**

| 支线 | 噪声 | 滤波 | 原理 |
|------|------|------|------|
| 上排 | 高斯（小幅扰动） | 自适应局部降噪 | 比较局部方差 vs 噪声方差 → 平坦就平滑、边缘就保留 |
| 下排 | 椒盐（极端值） | 自适应中值滤波 | 排序 + 极值判断 → 是噪声就替换、不是就保留 |

---

五、参数设置速查表

| 参数 | 自适应局部降噪 | 自适应中值滤波 |
|------|--------------|--------------|
| 窗口大小 | 固定，常用 `ksize=5` | 从 3×3 起，动态扩大 |
| 要手动设 | `ksize` | 初始 3×3（不改）+ 上限 $S_{max}$ |
| 另一个必需参数 | 噪声方差 $\sigma_\eta^2$ | 无 |
| 推荐初值 | `ksize=5`，`noise_var` 取一块平坦区方差 | `max_window_size=7` |
| 参数影响 | ksize 大 → 更平滑；$\sigma_\eta^2$ 估错 → 效果差 | $S_{max}$ 大 → 去噪更强、更慢 |

---

六、一句话总结

> **这张图的核心是"两种滤波各打各的靶"**：
>
> - **上排**：高斯噪声 → 自适应局部降噪。局部方差 ≈ 噪声方差时（平坦区）大胆平滑，局部方差远大于噪声方差时（边缘）保留原像素；
> - **下排**：椒盐噪声 → 自适应中值滤波。窗口从 3×3 动态扩大，只对确认是噪声的极值像素替换，正常像素保留原值。
>
> **两种滤波针对两种不同噪声，用对噪声图才能看出各自的"自适应"价值。** 没有通吃的滤波，选滤波前先判断噪声类型——这正是 6.4.4 节要传达的核心思想。

### 6.4.5 双边滤波器：bilateralFilter

**双边滤波**同时考虑**空间距离**和**像素值差异**，是一种**保边去噪**的非线性滤波。

#### 一、什么是"保边去噪"

**"保边去噪" = 在去噪的同时，把边缘保留下来。** 拆成两半看：

- **去噪**：把噪声抹平、抹掉；
- **保边**：把边缘（轮廓、边界）留下来，不让它被抹掉。

合起来就是：**该平滑的地方平滑，该清晰的地方保持清晰。**

> **术语复习**：这里说的"边缘"指**画面内部的像素值跳变（明暗交界、物体轮廓）**，不是图像最外圈的边框。参见 6.0 节对"边缘"的定义。

#### 二、为什么"去噪"和"保边"是矛盾的

普通平滑滤波（均值、高斯）的机制是：**邻域内所有像素加权求和**。

- 平坦区：邻域里都是差不多的值，求和再平均，噪声被抹平——好事；
- 边缘区：邻域里一半暗、一半亮，求和再平均，**边缘被"糊"成中间灰度**——坏事。

**"糊"的机制**：边缘不是某个像素，而是相邻像素之间的差；均值滤波是逐像素滑动，边缘两侧的像素都会被各自的窗口平均一遍，两侧都向中间值靠拢 → 它们之间的差变小 → 边缘对比度下降 → 视觉上边缘变模糊。

**所以普通平滑是"无差别"的：噪声和边缘一起糊掉。**

| 位置 | 普通平滑 | 问题 |
|------|---------|------|
| 平坦区 | 平滑掉噪声 | 好 |
| 边缘区 | 边缘也被糊 | 坏——边缘变模糊 |

**去噪和保边天然矛盾**：想把噪声抹平，就会连边缘一起抹；想保住边缘，噪声就抹不干净。

#### 三、双边滤波怎么"两全其美"

双边滤波用**两个权重**相乘，给邻域每个像素分配"话语权"：

$$
w(p, q) = \underbrace{G_s(\|p - q\|)}_{\text{空间权重}} \cdot \underbrace{G_r(|I_p - I_q|)}_{\text{值域权重}}
$$

- **空间权重** $G_s$：离中心越近，权重越大（和高斯滤波一样）；
- **值域权重** $G_r$：像素值越接近中心，权重越大（**这是双边滤波的关键**）。

**值域权重怎么起"保边"作用**：

看边缘两侧的邻域：

```
边缘附近的一行像素：
  100  100  100  200  200  200
                 ↑
              中心像素（假设是 200）
```

- 左边三个 100：和中心 200 **差异大** → 值域权重 $G_r$ **接近 0** → 几乎不参与平均；
- 右边三个 200：和中心 200 **差异小** → 值域权重 $G_r$ **很大** → 主导平均。

结果：**中心像素只被"同侧"的像素平均，跨过边缘的像素被"劝退"了** → 边缘不会被糊。

而平坦区呢？邻域里都是差不多的值，值域权重都很大，**正常平滑**，噪声被抹掉。

**一句话**：值域权重让"差异大的邻居"闭嘴，于是边缘两侧不互相污染，边缘被保留。

#### 四、和普通高斯滤波的对比

| 场景 | 高斯滤波 | 双边滤波 |
|------|---------|---------|
| 平坦区 | 平滑掉噪声 ✅ | 平滑掉噪声 ✅ |
| 边缘区 | 边缘被糊 ❌ | 边缘保留 ✅ |
| 计算量 | 较快 | 较慢（每个像素要算两个高斯） |
| 参数 | `ksize`, `sigma` | `d`, `sigmaColor`, `sigmaSpace` |

**核心区别**：高斯滤波的核只和"空间距离"有关，整张图用同一个核；双边滤波的核**每个像素都不同**，因为它还依赖"像素值差异"。

#### 五、一个直观的类比

想象你在给一张合影"磨皮"：

- **高斯滤波**：不管三七二十一，每个人脸都糊一遍 → 皮肤平滑了，**但眼睛、鼻子、嘴唇的边界也糊了**；
- **双边滤波**：只在皮肤（颜色相近区域）内部平滑，遇到眼睛、鼻子这类"颜色突变"的地方就停手 → **皮肤平滑了，五官边界还在**。

这就是"保边去噪"——**磨皮不磨五官**。

#### 六、核心公式与算法过程

综合权重：$w(p, q) = G_s(\|p - q\|) \cdot G_r(|I_p - I_q|)$

**算法过程**：
1. 对窗口内每个邻域像素计算综合权重。
2. 归一化权重。
3. 加权求平均得到中心像素输出。

**特点**：
- 平滑均匀区域（值域核会平均接近的像素）。
- 保留强边缘（跨边缘像素值差异大，权重接近 0）。
- 计算量比高斯滤波大。

```python
cv2.bilateralFilter(src, d, sigmaColor, sigmaSpace)
```

#### 七、典型用途

| 用途 | 说明 |
|------|------|
| 美颜磨皮 | 皮肤平滑，五官边缘保留 |
| 医学影像 | 去噪同时保住病灶边界 |
| 深度图 / 视差图 | 深度不连续处保留边界，避免"糊成一团" |
| 图像预处理 | 去噪后边缘更清晰，利于后续边缘检测 |
| HDR / 图像融合 | 保边滤波做多尺度分解 |

#### 八、一句话总结

> **"保边去噪" = 去噪的同时不糊边缘。**
>
> - **去噪**：平坦区（邻域值都差不多）正常平滑，噪声被抹掉；
> - **保边**：边缘区（邻域值差异大）让"差异大的邻居"权重接近 0，边缘两侧不互相污染，边缘保留。
>
> 双边滤波靠"空间权重 × 值域权重"做到这一点：**空间权重负责"近的邻居说话"，值域权重负责"像的邻居说话"**——两个都满足的邻居才有话语权，边缘两侧自然被隔离，噪声却在平坦区被抹平。这就是"保边去噪"的含义。

#### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

img = cv2.imread('P&V/cat.jpg')                              # 读取图片
if img is not None:                                          # 如果图片存在
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)           # BGR 转 RGB
    
    # --- 新增：手动添加高斯噪声 ---
    # 生成均值为0，标准差为25的高斯噪声矩阵（标准差越大，噪声越明显）
    noise = np.random.normal(0, 25, img_rgb.shape).astype(np.float32)
    # 将噪声加到原图上，并限制像素值在 0~255 之间
    noisy_img = np.clip(img_rgb.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    
    # --- 修改：对加了噪声的图做双边滤波 ---
    # 参数 (9, 75, 75)：d=9, sigmaColor=75, sigmaSpace=75
    bilateral_img = cv2.bilateralFilter(noisy_img, 9, 75, 75)
    
    plt.figure(figsize=(15, 6))                              # 创建画布（稍微加宽）
    
    # 子图 1：加噪原图
    plt.subplot(1, 3, 1)                                     # 改为 1 行 3 列
    plt.title('加噪原图 (噪声)')                               # 标题
    plt.imshow(noisy_img)                                    # 显示加噪原图
    plt.axis('off')                                          # 关闭坐标轴
    
    # 子图 2：双边滤波结果
    plt.subplot(1, 3, 2)                                     # 第 2 张
    plt.title('双边滤波 (保边去噪)')                            # 标题
    plt.imshow(bilateral_img)                                # 显示双边滤波结果
    plt.axis('off')                                          # 关闭坐标轴
    
    # 子图 3：原图（作为参照）
    plt.subplot(1, 3, 3)                                     # 第 3 张
    plt.title('原图 (参照)')                                   # 标题
    plt.imshow(img_rgb)                                      # 显示原图
    plt.axis('off')                                          # 关闭坐标轴
    
    plt.show()                                               # 显示图像
else:
    print("请确保 'P&V/cat.jpg' 存在")                       # 提示信息

**图像解读**

一、三张子图分别是什么

| 子图 | 内容 | 对应代码 |
|------|------|---------|
| 左：加噪原图（噪声） | 在原图基础上叠加了高斯噪声 | `noisy_img` |
| 中：双边滤波（保边去噪） | 对加噪图进行双边滤波 | `cv2.bilateralFilter(noisy_img, 9, 75, 75)` |
| 右：原图（参照） | 输入的原始猫图（无噪声） | `img_rgb` |

---

二、观察到的现象

1. **左图（加噪原图）**：整张图布满了细密的彩色/灰白颗粒（高斯噪声），原本平滑的白色猫毛、地砖表面都变得“毛糙”和“脏”。
2. **中图（双边滤波）**：颗粒感被大幅抹平，猫毛和地砖恢复平滑。但猫的眼睛、耳朵边缘、黑斑轮廓、胡须等结构依然清晰锐利，没有被模糊掉。
3. **右图（原图）**：作为参照，展示了未被污染前的真实样子。

---

三、这说明了什么

1. 双边滤波的“去噪”能力

- 左图加了高斯噪声后，平坦区（如猫的白色身体、背景地砖）的像素值出现了随机波动；
- 双边滤波在平坦区正常平滑，把这些随机波动（噪声）抹平，所以中图比左图干净很多。

2. 双边滤波的“保边”能力

- 如果使用普通高斯滤波去处理左图，噪声虽然会被抹平，但猫的眼睛、耳朵、黑斑边缘也会一起被糊掉；
- 双边滤波因为加入了**值域权重 \(G_r\)**，当窗口跨在边缘两侧时，差异大的邻居权重接近 0，边缘两侧不互相污染；
- 所以中图在去噪的同时，猫的五官和轮廓依然保持清晰。

3. 与原图（右图）的对比

- 中图不是原图的简单复制，它是在“被噪声污染后”重新恢复出来的结果；
- 虽然中图比左图干净很多，但相比右图（原图），中图在一些极细微的纹理（如猫毛尖端、地砖微小凹凸）上会稍显平滑，这是去噪必然付出的代价；
- 但整体结构、边缘、亮度关系与原图高度一致，说明双边滤波在“去噪”和“保边”之间取得了很好的平衡。



### 6.4.6 钝化掩蔽：USM 锐化

**钝化掩蔽（Unsharp Masking）** 是一种经典的图像锐化方法。它的名字来源于操作步骤：先用“钝化”（模糊）原图得到一个模糊版本，再用它作为“掩蔽”来提取高频细节，最后加回原图实现锐化。

**算法过程**：
1. 对原图做高斯模糊，得到 $I_{blur}$。
2. 计算掩膜：$M = I - I_{blur}$（相当于高通分量，含边缘细节）。
3. 将掩膜按比例加回原图：$I_{sharp} = I + k \cdot M$，$k$ 是强度系数。

**关键实现细节**：
- 直接用 `cv2.subtract` 计算 $I - I_{blur}$ 时，负值会被**截断为 0**（uint8 范围 0~255），导致变暗的边缘细节丢失。
- 正确做法：先转 `float` 或 `int16` 计算掩膜，再叠加并 clip 到 0~255。

**特点**：
- 增强边缘和细节，视觉上更清晰。
- 若 $k$ 太大，会增强噪声、产生光晕。
- 是 Photoshop 中"USM 锐化"功能的原理。

#### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

# --- 读取图片 ---
img = cv2.imread('P&V/cameraman.bmp')                        # 读取图片

if img is not None:
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)         # 转灰度

    # ---------- 构造“轻微失焦”的图 ----------
    # 用非常温和的模糊 (3,3), σ=0.8，让边缘变软，但保留细节
    blurred = cv2.GaussianBlur(img_gray, (3, 3), 0.8)

    # ---------- USM 锐化函数 ----------
    def usm_sharpen(image, k=1.5, sigma=1.0):
        # 内部计算掩膜
        blurred_inner = cv2.GaussianBlur(image, (5, 5), sigma)
        img_f = image.astype(np.float32)
        blur_f = blurred_inner.astype(np.float32)
        mask = img_f - blur_f
        # 锐化
        sharpened_f = img_f + k * mask
        sharpened = np.clip(sharpened_f, 0, 255).astype(np.uint8)
        # 可视化掩膜（放大 5 倍）
        mask_vis = cv2.convertScaleAbs(mask * 5)
        return sharpened, mask_vis

    # 对轻微模糊图做 USM
    sharpened_blur, _ = usm_sharpen(blurred, k=1.5)

    # 对原图也做一次对比（看看原图直接锐化会怎样）
    sharpened_orig, _ = usm_sharpen(img_gray, k=1.5)

    # ---------- 可视化 ----------
    plt.figure(figsize=(18, 10))

    # 第一行：原图 vs 原图锐化（告诉你为什么原图不需要锐化）
    plt.subplot(2, 3, 1)
    plt.title('原图（纯净）')
    plt.imshow(img_gray, cmap='gray')
    plt.axis('off')

    plt.subplot(2, 3, 2)
    plt.title('原图直接 USM（看不出变化）')
    plt.imshow(sharpened_orig, cmap='gray')
    plt.axis('off')

    plt.subplot(2, 3, 3)
    plt.title('原图直接 USM 的差异（几乎全黑）')
    diff_orig = cv2.absdiff(img_gray, sharpened_orig)
    plt.imshow(cv2.convertScaleAbs(diff_orig * 10), cmap='gray')
    plt.axis('off')

    # 第二行：轻微失焦 vs USM 恢复（正面效果在此！）
    plt.subplot(2, 3, 4)
    plt.title('轻微失焦 (3x3, σ=0.8)')
    plt.imshow(blurred, cmap='gray')
    plt.axis('off')

    plt.subplot(2, 3, 5)
    plt.title('USM 锐化（k=1.5，恰到好处）')
    plt.imshow(sharpened_blur, cmap='gray')
    plt.axis('off')

    plt.subplot(2, 3, 6)
    plt.title('锐化恢复的细节（差异图）')
    diff_blur = cv2.absdiff(blurred, sharpened_blur)
    plt.imshow(cv2.convertScaleAbs(diff_blur * 10), cmap='gray')
    plt.axis('off')

    plt.tight_layout()
    plt.show()

else:
    print("请确保 'P&V/cameraman.bmp' 存在")

**图像解读：Cameraman 与 USM 锐化的真实边界**

一、实验设计逻辑

本组图以经典的 Cameraman 测试图为输入，通过控制变量，分别测试 USM 锐化在“清晰图”和“轻微失焦图”上的表现，并借助差异图（|M|）可视化锐化强度的空间分布。

二、第一行：原图直接 USM（演示“无效性”）

| 子图 | 内容 | 现象 |
|------|------|------|
| 左：原图（纯净） | 原始灰度图，边缘锐利、结构清晰 | 图像本身已具备充足的高频信息 |
| 中：原图直接 USM | 对原图直接做 USM（k=1.5） | 视觉上与原图几乎一致，看不出变化 |
| 右：差异图（放大10倍） | 原图与锐化图的绝对差 | 几乎全黑，仅有极其微弱的轮廓痕迹 |

**说明**：当图像本身已经清晰时，USM 的掩膜（原图 - 模糊图）值极小，锐化增量微乎其微。这证明 USM 对清晰图无效，它不是“清晰化按钮”。

三、第二行：轻微失焦 → USM（演示“有效边界”）

| 子图 | 内容 | 现象 |
|------|------|------|
| 左：轻微失焦 (3×3, σ=0.8) | 模拟轻微失焦，边缘稍微变软 | 结构尚存，但边缘锐度下降 |
| 中：USM 锐化 (k=1.5) | 对失焦图做 USM | 边缘锐度恢复，画面变“实”，但无噪点、无光晕 |
| 右：差异图（放大10倍） | 失焦图与锐化图的绝对差 | 清晰勾勒出人物、三脚架、建筑的轮廓线 |

**说明**：这是 USM 的“最佳应用场景”——边缘轻微发虚但结构尚存。USM 通过增强边缘对比度恢复锐度，且效果克制、自然。

四、核心结论

| 场景 | 输入特征 | USM 表现 | 结论 |
|------|----------|----------|------|
| 原图（清晰） | 边缘锐利，细节充足 | 几乎无变化 | USM 对清晰图无效 |
| 轻微失焦 | 边缘发虚，结构尚存 | 边缘恢复锐利，无副作用 | USM 的最佳应用场景 |
| 加噪图 | 高频噪声混入 | 噪声被放大，画面变脏 | USM 怕噪声，需先降噪 |
| 重度模糊 | 细节丢失，结构崩溃 | 只能强化残留边缘，出现光晕 | USM 无法无中生有 |

五、一句话总结

> USM 锐化的真实面貌是“精密的边缘增强工具”，而非“震撼的修复魔法”。它的有效边界非常明确：只对“轻微失焦、结构尚存”的图有效；对清晰图无效，对噪声图是灾难，对重度模糊图无能为力。这张 Cameraman 对比图，正是这一边界最标准、最克制的说明。

### 6.4.7 Laplacian 算子

**Laplacian 算子** 是二阶微分算子，对图像做二阶偏导：

$$\nabla^2 I = \frac{\partial^2 I}{\partial x^2} + \frac{\partial^2 I}{\partial y^2}$$

在离散图像中通常用卷积核近似：

- 4 邻域核：

  ```text
   0  1  0
   1 -4  1
   0  1  0
  ```

- 8 邻域核：

  ```text
   1  1  1
   1 -8  1
   1  1  1
  ```

**算法过程**：

1. 用上述核与图像进行卷积/相关运算，得到二阶导数图。
2. 结果中高值对应边缘（像素值突变处）。
3. 锐化公式：$I_{sharp} = I - k \cdot \nabla^2 I$。

**重要修正**：

- Laplacian 输出有**正有负**，锐化时必须用**带符号**的 Laplacian 做减法。
- 如果先 `convertScaleAbs` 取绝对值再加回原图，正负边缘都变成亮线，会产生错误的光晕。
- 正确流程：`lap = cv2.Laplacian(img, cv2.CV_64F)` → `sharp = img - k*lap` → `np.clip` → `uint8`。

**OpenCV 提供 `cv2.Laplacian()`**，注意使用 `cv2.CV_64F` 保存负值。

---

#### 一、Laplacian 算子到底起什么作用？

一句话概括：

> **Laplacian 算子用来检测图像中像素值发生“剧烈变化”的位置，也就是边缘、角点、细线和孤立点等高频细节。**

它不直接“增强图像”，而是先给出一张“二阶导数图”，告诉我们：

- 哪里变化快；
- 哪里是亮暗交界；
- 哪里是细节所在。

然后我们再用这张图去**锐化**原图。

---

#### 二、为什么二阶导数能检测边缘？

先看一维信号。假设一行像素亮度变化如下：

```text
暗 暗 暗 亮 亮 亮 暗 暗 暗
```

对应的一维曲线可以理解为：

```text
低 低 低 高 高 高 低 低 低
```

**1. 一阶导数**

一阶导数表示“亮度变化的快慢”。

- 在平坦区域：一阶导数接近 0；
- 在边缘处：一阶导数最大或最小；
- 所以一阶导数可以找到边缘的位置。

**2. 二阶导数**

二阶导数表示“一阶导数的变化率”。

- 在平坦区域：二阶导数接近 0；
- 在边缘的上升沿和下降沿：二阶导数会出现正负峰；
- 在边缘中心附近：二阶导数会过零。

因此：

> **二阶导数对边缘、细线、孤立点更敏感。**

Laplacian 就是二维图像上的二阶导数之和：

$$\nabla^2 I = \frac{\partial^2 I}{\partial x^2} + \frac{\partial^2 I}{\partial y^2}$$

它把水平方向和垂直方向的二阶变化加在一起，所以能同时检测：

- 水平边缘；
- 垂直边缘；
- 斜边缘；
- 角点；
- 细线；
- 孤立噪点。

---

#### 三、既然一阶导数就能找到边缘，为什么还要用二阶导数？

这是个很好的问题。**一阶导数确实能找到边缘**，而且 Sobel、Prewitt、Roberts 等经典边缘检测算子用的就是一阶导数。

但二阶导数（Laplacian）仍然有它不可替代的用途。两者不是“谁替代谁”，而是“各有所长”。

**3.1 一阶导数怎么找边缘**

一阶导数表示“亮度变化的快慢”。

- 平坦区域：一阶导数 ≈ 0；
- 边缘处：一阶导数出现极大值或极小值；
- 所以只要找一阶导数的**峰值**，就能定位边缘。

Sobel、Prewitt 就是这么做的。

**3.2 二阶导数怎么找边缘**

二阶导数表示“一阶导数的变化率”。

- 平坦区域：二阶导数 ≈ 0；
- 边缘附近：二阶导数出现正峰和负峰；
- 边缘中心：二阶导数**过零**。

所以二阶导数是通过**过零点**来定位边缘的。

**3.3 那为什么还要用二阶导数？**

主要有四个原因。

**原因一：二阶导数对“细线”和“孤立点”更敏感**

一阶导数对**粗边缘**响应很强，但对：

- 细线；
- 孤立亮点；
- 孤立暗点；
- 小角点；

响应较弱。

而二阶导数对这些结构非常敏感。
比如一条只有 1 像素宽的亮线：

- 一阶导数只有一个较小的峰；
- 二阶导数会出现明显的正负双峰。

所以 Laplacian 常用于：

- 细线检测；
- 孤立点检测；
- 角点检测；
- 纹理增强。

**原因二：二阶导数的过零点定位更精确**

一阶导数找峰值时，峰值可能比较宽，定位不够精确。
二阶导数找**过零点**，而过零点通常只有一个位置，定位更细。

所以在需要**亚像素级边缘定位**时，二阶导数更有优势。

**原因三：Laplacian 是各向同性的**

Laplacian 是：

$$\nabla^2 I = \frac{\partial^2 I}{\partial x^2} + \frac{\partial^2 I}{\partial y^2}$$

它对**所有方向**的边缘都一视同仁：

- 水平边缘；
- 垂直边缘；
- 斜边缘；
- 角点。

而一阶导数算子（如 Sobel）通常要分别算 $G_x$ 和 $G_y$，再合成梯度幅值，方向性更明显。

**原因四：Laplacian 可以直接用于锐化**

锐化公式：

$$I_{sharp} = I - k \cdot \nabla^2 I$$

它利用的是二阶导数的**符号信息**：

- 亮侧 Laplacian 为正；
- 暗侧 Laplacian 为负；
- 用原图减去它，亮侧更亮、暗侧更暗；
- 边缘对比度增强。

一阶导数也能锐化，但通常需要先算梯度，再构造锐化模板，不如 Laplacian 直接。

**3.4 一阶 vs 二阶 对比**

| 特性 | 一阶导数 | 二阶导数 |
|---|---|---|
| 典型算子 | Sobel、Prewitt、Roberts | Laplacian |
| 边缘定位 | 找峰值 | 找过零点 |
| 对粗边缘 | 强 | 较强 |
| 对细线 | 弱 | 强 |
| 对孤立点 | 弱 | 强 |
| 对噪声 | 较敏感 | 更敏感 |
| 方向性 | 有方向 | 各向同性 |
| 常见用途 | 边缘检测、梯度 | 细线检测、锐化、角点 |

**3.5 为什么实际中常两者配合**

- 一阶导数：找边缘位置、算梯度方向；
- 二阶导数：检测细线、孤立点、做锐化；
- 经典 Canny 边缘检测：先用一阶导数（Sobel）算梯度，再用二阶导数思想做非极大值抑制和双阈值。

所以不是“一阶够用就不需要二阶”，而是：

> **一阶导数擅长找边缘，二阶导数擅长找细节、定位过零点、做锐化。**

**3.6 一句话总结**

一阶导数确实能找到边缘，但：

- 它对细线、孤立点、角点不够敏感；
- 峰值定位不如二阶导数过零点精确；
- 它不能直接用于 Laplacian 锐化。

二阶导数（Laplacian）补充了这些能力，所以两者经常一起使用，而不是互相替代。

---

#### 四、Laplacian 的输出长什么样？

Laplacian 的输出不是“边缘图”那么简单，它是一张**有正有负**的二阶导数图。

- 在边缘的一侧，输出可能是正值；
- 在边缘的另一侧，输出可能是负值；
- 在平坦区域，输出接近 0；
- 在孤立亮点或暗点处，输出会出现明显的正峰或负峰。

如果直接显示带符号的 Laplacian，通常需要：

- 取绝对值；
- 或者做归一化；
- 或者用 `cv2.convertScaleAbs` 转成可见图像。

例如：

```python
lap = cv2.Laplacian(img_gray, cv2.CV_64F)   # 保留正负号
lap_abs = cv2.convertScaleAbs(lap)          # 取绝对值，便于显示
```

`lap_abs` 看起来就像一张“边缘图”：

- 亮的地方表示二阶导数大；
- 暗的地方表示二阶导数小；
- 边缘、细线、角点会比较亮。

---

#### 五、Laplacian 为什么能用于锐化？

锐化的本质是：

> **增强图像中的高频细节，让边缘更清晰。**

Laplacian 正好提取了高频细节，所以可以用它来锐化。

锐化公式：

$$I_{sharp} = I - k \cdot \nabla^2 I$$

其中：

- $I$ 是原图；
- $\nabla^2 I$ 是 Laplacian 结果；
- $k$ 是锐化强度；
- $I_{sharp}$ 是锐化后的图像。

为什么是减法？

因为 Laplacian 在边缘两侧符号相反：

- 在亮侧，Laplacian 可能为正；
- 在暗侧，Laplacian 可能为负；
- 用原图减去 Laplacian，会让亮侧更亮、暗侧更暗；
- 于是边缘对比度增强，看起来更锐。

---

#### 六、为什么必须用带符号的 Laplacian？

这是最容易出错的地方。

**错误做法**

```python
lap = cv2.Laplacian(img_gray, cv2.CV_64F)   # 有正有负
lap_abs = cv2.convertScaleAbs(lap)          # 取了绝对值
sharp = img_gray + k * lap_abs              # 错误：把绝对值加回去
```

后果：

- 原本正边缘和负边缘都被变成亮线；
- 亮线被统一加回原图；
- 边缘两侧都变亮；
- 结果出现**光晕**、**白边**、**不自然**。

**正确做法**

```python
lap = cv2.Laplacian(img_gray, cv2.CV_64F)   # 保留正负号
sharp_f = img_gray.astype(np.float32) - k * lap
sharp = np.clip(sharp_f, 0, 255).astype(np.uint8)
```

要点：

1. 用 `cv2.CV_64F` 保存负值；
2. 不要先取绝对值；
3. 用原图减去带符号的 Laplacian；
4. 最后再 `np.clip` 到 0~255；
5. 再转回 `uint8`。

---

#### 七、Laplacian 算子的实际作用总结

| 作用 | 说明 |
|---|---|
| 边缘检测 | 对像素值突变敏感，能检测边缘、角点、细线 |
| 细节提取 | 提取高频信息，如纹理、细小结构 |
| 锐化基础 | 用 $I - k\nabla^2 I$ 增强边缘对比度 |
| 孤立点检测 | 对孤立亮点、暗点、噪点非常敏感 |
| 各向同性 | 4 邻域和 8 邻域核都具有旋转不变性 |

一句话总结：

> **Laplacian 算子本身是“二阶导数边缘检测器”，它提取图像中的高频突变；把它的带符号结果按 $I - k\nabla^2 I$ 减回原图，就能实现锐化。**

---



#### 示例

In [ ]:
import cv2                                                      # 导入 OpenCV
import numpy as np                                              # 导入 NumPy
import matplotlib.pyplot as plt                                 # 导入 Matplotlib

plt.rcParams['font.sans-serif'] = ['SimHei']                    # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                      # 解决负号显示问题

img = cv2.imread('P&V/cameraman.bmp')                           # 读取 cameraman 图片

if img is not None:                                             # 如果图片存在
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)            # 转灰度
    lap = cv2.Laplacian(img_gray, cv2.CV_64F)                   # Laplacian，保留符号
    lap_abs = cv2.convertScaleAbs(lap)                          # 取绝对值用于显示

    plt.figure(figsize=(15, 10))                                # 创建画布

    # ---------- 第一行 ----------
    plt.subplot(2, 3, 1)                                        # 第 1 行第 1 列
    plt.title('原图')                                            # 标题
    plt.imshow(img_gray, cmap='gray')                           # 显示原图
    plt.axis('off')                                             # 关闭坐标轴

    plt.subplot(2, 3, 2)                                        # 第 1 行第 2 列
    plt.title(r'Laplacian 边缘 ($|\nabla^2 I|$)')               # 用 mathtext 显示 ∇²
    plt.imshow(lap_abs, cmap='gray')                            # 显示 Laplacian 边缘
    plt.axis('off')                                             # 关闭坐标轴

    k1 = 1                                                      # 第 1 行第 3 列放 k=1
    sharp1_f = img_gray.astype(np.float32) - k1 * lap           # 锐化：I - k*Laplacian
    sharp1 = np.clip(sharp1_f, 0, 255).astype(np.uint8)         # 截断并转 uint8

    plt.subplot(2, 3, 3)                                        # 第 1 行第 3 列
    plt.title(rf'Laplacian 锐化 ($I - k\nabla^2 I$, k={k1})')   # 用 mathtext 显示公式
    plt.imshow(sharp1, cmap='gray')                             # 显示 k=1 锐化图
    plt.axis('off')                                             # 关闭坐标轴

    # ---------- 第二行：k = 2, 3, 4 ----------
    k_values = [2, 3, 4]                                        # 第二行的三个 k 值

    for i, k in enumerate(k_values):                            # 遍历 k=2, 3, 4
        sharp_f = img_gray.astype(np.float32) - k * lap         # 锐化：I - k*Laplacian
        sharpened = np.clip(sharp_f, 0, 255).astype(np.uint8)   # 截断并转 uint8

        plt.subplot(2, 3, 4 + i)                                # 第 2 行第 1~3 列
        plt.title(rf'Laplacian 锐化 ($I - k\nabla^2 I$, k={k})')# 用 mathtext 显示公式
        plt.imshow(sharpened, cmap='gray')                      # 显示锐化效果图
        plt.axis('off')                                         # 关闭坐标轴

    plt.tight_layout()                                          # 自动调整间距
    plt.show()                                                  # 显示图像

else:
    print("请确保 'P&V/cameraman.bmp' 存在")                    # 提示信息

**图像解读：Laplacian 锐化在不同 k 值下的表现**

一、6 张子图分别是什么

| 位置 | 子图 | 对应代码 |
|------|------|---------|
| 第 1 行第 1 列 | 原图 | `img_gray` |
| 第 1 行第 2 列 | Laplacian 边缘 ($|\nabla^2 I|$) | `lap_abs = cv2.convertScaleAbs(lap)` |
| 第 1 行第 3 列 | Laplacian 锐化 ($I - k\nabla^2 I$, k=1) | `img_gray - 1 * lap` |
| 第 2 行第 1 列 | Laplacian 锐化 ($I - k\nabla^2 I$, k=2) | `img_gray - 2 * lap` |
| 第 2 行第 2 列 | Laplacian 锐化 ($I - k\nabla^2 I$, k=3) | `img_gray - 3 * lap` |
| 第 2 行第 3 列 | Laplacian 锐化 ($I - k\nabla^2 I$, k=4) | `img_gray - 4 * lap` |

---

二、逐图观察

**1. 原图**

- Cameraman 经典测试图，黑白分明；
- 人物轮廓、相机、三脚架、远处建筑清晰；
- 背景天空、草地相对平滑。

**2. Laplacian 边缘图 ($|\nabla^2 I|$)**

- 人物轮廓、相机、三脚架、地平线、建筑边缘被明显勾出；
- 平坦区域基本为黑；
- 说明 Laplacian 确实只对“二阶导数大”的地方有响应，即边缘、角点、细线。

**3. k=1 锐化图**

- 边缘比原图稍微锐一点；
- 但整体观感变化不大；
- 因为 Laplacian 数值本身偏小，k=1 的增强幅度有限。

**4. k=2 锐化图**

- 边缘明显变锐；
- 人物轮廓、相机、三脚架、建筑边缘更清晰；
- 背景草地开始出现轻微颗粒感。

**5. k=3 锐化图**

- 边缘非常锐；
- 但开始出现**白边 / 光晕**，尤其在人物轮廓、三脚架、建筑边缘处；
- 草地、天空的噪声被明显放大。

**6. k=4 锐化图**

- 边缘极强；
- 白边 / 光晕非常明显；
- 噪声大量出现，草地几乎变成颗粒状；
- 图像开始“失真”，不再是自然的锐化。

---

三、这说明了什么

**1. Laplacian 是二阶导数，对噪声极其敏感**

- Laplacian 不区分“真实边缘”和“噪声”；
- k 越大，边缘被增强得越强，但噪声也被同步放大；
- 所以 k 不能无限增大。

**2. 锐化强度 k 需要折中**

| k 值 | 效果 | 评价 |
|------|------|------|
| k=1 | 轻微锐化，几乎看不出 | 太保守 |
| k=2 | 边缘明显变锐，噪声可控 | 比较合适 |
| k=3 | 边缘很锐，出现白边 | 偏强 |
| k=4 | 白边、光晕、噪声明显 | 过强 |

**3. 为什么会出现白边 / 光晕**

- Laplacian 在边缘两侧符号相反；
- 用 $I - k\nabla^2 I$ 时，亮侧被加得更亮、暗侧被减得更暗；
- k 太大时，亮侧可能超过 255 被截断，暗侧可能低于 0 被截断；
- 截断导致边缘两侧出现不自然的白边或黑边，即“光晕”。

**4. 为什么背景草地先出问题**

- 草地本身纹理细碎，属于高频区域；
- Laplacian 对高频最敏感；
- 所以 k 增大时，草地比天空更早出现颗粒感。

---

四、小结

> 这张图用同一张 Cameraman，对比了 k=1、2、3、4 四种锐化强度。

- **k=1**：几乎看不出变化；
- **k=2**：边缘明显变锐，噪声可控，是比较合适的值；
- **k=3**：边缘很锐，但开始出现白边；
- **k=4**：白边、光晕、噪声都非常明显，已经过强。

**结论**：

1. Laplacian 锐化的效果随 k 增大而增强；
2. 但 k 过大时，噪声、白边、光晕会同步放大；
3. 实际使用中，k 一般取 0.5 ~ 2.0 之间；
4. 如果想在大 k 下仍保持自然，应先降噪，再用 Laplacian 锐化。

### 6.4.8 Sobel 算子与 Scharr 算子

本节讲**两个算子**：

- **Sobel 算子**：一阶微分算子，用两个卷积核分别计算 X 方向和 Y 方向的梯度；
- **Scharr 算子**：Sobel 的改进版，同样是两个核，但中心权重更大，对细节更敏感。

#### 一、Sobel 算子

Sobel 有两个核：

**Sobel X**（检测竖直边缘）：

```text
-1  0  +1
-2  0  +2
-1  0  +1
```

**Sobel Y**（检测水平边缘）：

```text
-1 -2 -1
 0  0  0
+1 +2 +1
```

- Sobel X 检测的是**左右方向的变化**（竖直边缘）；
- Sobel Y 检测的是**上下方向的变化**（水平边缘）。

两者合起来，才能算出每个像素的**梯度幅值**和**梯度方向**：

$$G = \sqrt{G_x^2 + G_y^2}, \quad \theta = \arctan(G_y / G_x)$$

#### 二、Scharr 算子

Scharr 是 Sobel 的改进版，核是：

**Scharr X**：

```text
-3   0  +3
-10  0  +10
-3   0  +3
```

**Scharr Y**：

```text
-3  -10  -3
 0    0   0
+3  +10  +3
```

Scharr 和 Sobel 的关系：

| 项目 | Sobel | Scharr |
|------|-------|--------|
| 竖直平滑权重 | 1, 2, 1 | 3, 10, 3 |
| 水平差分 | -1, 0, +1 | -3, 0, +3 |
| 3×3 时的精度 | 一般 | 更高 |
| 边缘响应 | 较弱 | 更强、更锐 |
| 常见用途 | 一般边缘检测 | Canny 中算梯度 |

**Scharr 不是另一个完全不同的方法，而是 Sobel 的“加强版”**：

- 同样是“先平滑、再差分”；
- 但平滑权重从 `1, 2, 1` 换成 `3, 10, 3`，中心权重更大；
- 这样对细节更敏感，3×3 时的精度比 Sobel 高。

#### 三、Sobel 核不只是“右减左”

以 Sobel X 为例，它做的事是：

> **每一行都做“右边像素减去左边像素”，再把三行按 1、2、1 加权求和。**

拆开：

```text
第 1 行：(右 − 左) × 1
第 2 行：(右 − 左) × 2
第 3 行：(右 − 左) × 1
最终结果 = 三行相加
```

它可以拆成两个一维核的外积：

```text
竖直方向：1, 2, 1   （平滑）
水平方向：-1, 0, +1 （差分）
```

所以中间行的权重是 2，来自竖直平滑核 `1, 2, 1` 的中心权重。

**为什么平滑核是 `1, 2, 1`**：

| 原因 | 说明 |
|------|------|
| 中心权重大 | 离中心越近越重要，中间行权重 2 |
| 近似高斯 | 是离散高斯核的近似，中间高、两边低 |
| 便于计算 | 1 + 2 + 1 = 4，整数，方便归一化 |

**“方便归一化”是什么意思**：

- 归一化 = 让核的总和变成 1；
- `1, 2, 1` 的和是 4，除以 4 得到 `0.25, 0.5, 0.25`，简单；
- 但 **Sobel 本身不归一化**，因为它的总和是 0，是差分核。

#### 四、为什么 Sobel 能抑制噪声

核心只有一句话：

> **Sobel 在竖直方向做了 `1, 2, 1` 平滑，而平滑就是把多个像素平均，噪声是随机正负的，平均后会被部分抵消。**

**4.1 噪声是随机的，有正有负**

假设真实灰度是 100，加了噪声后，上下三行可能是：

```text
102
 98
101
```

- 第 1 行：+2
- 第 2 行：−2
- 第 3 行：+1

**三个噪声加起来：+2 − 2 + 1 = +1**，比单个噪声小。

**4.2 朴素差分只看一行**

```text
-1  0  +1
```

它只用了**一行**的“右 − 左”。如果这一行正好噪声很大，输出就会被噪声主导。

**4.3 Sobel 看三行，并加权**

```text
-1  0  +1
-2  0  +2
-1  0  +1
```

它把**三行**的“右 − 左”按 `1, 2, 1` 加权求和，**三行的噪声互相抵消一部分**。

**4.4 用方差看**

假设每个像素的噪声方差是 σ²。

朴素差分：

```text
输出 = 右 − 左
方差 = σ² + σ² = 2σ²
```

Sobel X：

```text
输出 = 1×(右₁−左₁) + 2×(右₂−左₂) + 1×(右₃−左₃)
方差 = 1²·2σ² + 2²·2σ² + 1²·2σ²
     = 2σ² + 8σ² + 2σ²
     = 12σ²
```

看起来 Sobel 方差更大，但别忘了：Sobel 的**信号**也被放大了 4 倍（因为权重和是 4）。比较噪声不能只看方差，要看**信噪比**。

**4.5 用信噪比看**

| 算子 | 信号 | 噪声标准差 | 信噪比 |
|------|------|-----------|--------|
| 朴素差分 | Δ | √2·σ | 0.71·(Δ/σ) |
| Sobel X | 4Δ | √12·σ | 1.15·(Δ/σ) |

**Sobel 的信噪比比朴素差分高约 1.6 倍。**

这就是“抑制噪声”的真正含义：

> **不是 Sobel 的输出噪声更小，而是 Sobel 的边缘信号更强，信噪比更高。**

**4.6 用频域看**

- 朴素差分 `-1, 0, +1`：高频增益大，噪声被放大；
- Sobel `1,2,1 × -1,0,+1`：`1, 2, 1` 是低通因子，**先压低高频噪声**，再差分检测边缘。

**4.7 Scharr 也抑制噪声**

Scharr 的竖直平滑核是 `3, 10, 3`，同样是平滑核，机制和 Sobel 完全一样：

- 噪声随机正负，三行加权后互相抵消；
- 中心权重更大（10），对中心行更重视；
- 3×3 精度更高，边缘更锐。

**4.8 一句话总结**

> **Sobel 和 Scharr 抑制噪声，靠的是竖直方向的平滑核：**
>
> - Sobel 用 `1, 2, 1`；
> - Scharr 用 `3, 10, 3`；
> - 噪声随机正负，三行加权后互相抵消；
> - 统计上，信噪比比朴素差分高；
> - 频域上，平滑核是低通因子，先压低高频噪声，再差分。
>
> 所以两者都比朴素差分更稳，代价是**边缘定位略粗**。

#### 五、梯度幅值与方向

- 幅值：$G = \sqrt{G_x^2 + G_y^2}$
- 方向：$\theta = \arctan(G_y / G_x)$

#### 六、OpenCV 中的调用

```python
cv2.Sobel(src, ddepth, dx, dy, ksize)
cv2.Scharr(src, ddepth, dx, dy)
```

#### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

img = cv2.imread('P&V/house.jpg')                            # 读取 house 图片
if img is not None:                                          # 如果图片存在
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)         # 转换为灰度图

    # --- Sobel ---
    sobel_x = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=3)  # Sobel X
    sobel_y = cv2.Sobel(img_gray, cv2.CV_64F, 0, 1, ksize=3)  # Sobel Y
    sobel_mag = np.sqrt(sobel_x**2 + sobel_y**2)             # Sobel 幅值

    sobel_x_abs = cv2.convertScaleAbs(sobel_x)               # 取绝对值
    sobel_y_abs = cv2.convertScaleAbs(sobel_y)               # 取绝对值
    sobel_mag_abs = cv2.convertScaleAbs(sobel_mag)           # 取绝对值

    # --- Scharr ---
    scharr_x = cv2.Scharr(img_gray, cv2.CV_64F, 1, 0)        # Scharr X
    scharr_y = cv2.Scharr(img_gray, cv2.CV_64F, 0, 1)        # Scharr Y
    scharr_mag = np.sqrt(scharr_x**2 + scharr_y**2)          # Scharr 幅值

    scharr_x_abs = cv2.convertScaleAbs(scharr_x)             # 取绝对值
    scharr_y_abs = cv2.convertScaleAbs(scharr_y)             # 取绝对值
    scharr_mag_abs = cv2.convertScaleAbs(scharr_mag)         # 取绝对值

    plt.figure(figsize=(15, 8))                              # 创建画布

    # 第一行：Sobel
    plt.subplot(2, 3, 1)                                     # 第 1 行第 1 列
    plt.title('Sobel X')                                     # 标题
    plt.imshow(sobel_x_abs, cmap='gray')                     # 显示 Sobel X
    plt.axis('off')                                          # 关闭坐标轴

    plt.subplot(2, 3, 2)                                     # 第 1 行第 2 列
    plt.title('Sobel Y')                                     # 标题
    plt.imshow(sobel_y_abs, cmap='gray')                     # 显示 Sobel Y
    plt.axis('off')                                          # 关闭坐标轴

    plt.subplot(2, 3, 3)                                     # 第 1 行第 3 列
    plt.title('Sobel 幅值')                                   # 标题
    plt.imshow(sobel_mag_abs, cmap='gray')                   # 显示 Sobel 幅值
    plt.axis('off')                                          # 关闭坐标轴

    # 第二行：Scharr
    plt.subplot(2, 3, 4)                                     # 第 2 行第 1 列
    plt.title('Scharr X')                                    # 标题
    plt.imshow(scharr_x_abs, cmap='gray')                    # 显示 Scharr X
    plt.axis('off')                                          # 关闭坐标轴

    plt.subplot(2, 3, 5)                                     # 第 2 行第 2 列
    plt.title('Scharr Y')                                    # 标题
    plt.imshow(scharr_y_abs, cmap='gray')                    # 显示 Scharr Y
    plt.axis('off')                                          # 关闭坐标轴

    plt.subplot(2, 3, 6)                                     # 第 2 行第 3 列
    plt.title('Scharr 幅值')                                  # 标题
    plt.imshow(scharr_mag_abs, cmap='gray')                  # 显示 Scharr 幅值
    plt.axis('off')                                          # 关闭坐标轴

    plt.tight_layout()                                       # 自动调整间距
    plt.show()                                               # 显示图像
else:
    print("请确保 'P&V/house.jpg' 存在")                      # 提示信息

## 6.5 图像金字塔：高斯金字塔、拉普拉斯金字塔

### 内容介绍

**图像金字塔**是一种**多分辨率表示**：同一张图，在不同尺度下呈现不同的结构信息。

#### 一、什么叫"在不同尺度下看"

**尺度**，可以理解成：

> **一个像素代表多大的现实区域。**

- **原图（尺度最小）**：1 个像素 = 1 个很小的点 → 细节最丰富；
- **缩小一半（尺度变大）**：1 个像素 = 原来的 2×2 区域 → 细节减少；
- **再缩小一半（尺度更大）**：1 个像素 = 原来的 4×4 区域 → 只剩大致结构。

所以：

> **缩小图像 = 放大"尺度" = 用更粗的粒度看图像。**

用 cameraman 举例：

- **原图**：能看到脸、衣服褶皱、相机镜头、三脚架螺丝、远处建筑小窗；
- **pyrDown 1**：相机镜头还在，但螺丝看不清了；衣服褶皱变模糊；
- **pyrDown 2**：脸变成一团，相机变成一个方块，三脚架只剩三条粗线；但整体轮廓、姿态、构图仍然在。

这就是"不同尺度下看到不同结构"：

- 小尺度：细节；
- 中尺度：局部结构；
- 大尺度：整体轮廓。

#### 二、为什么需要"在不同尺度下看"

因为很多任务，**在不同尺度下做，效果完全不同**。

- **目标检测**：小尺度看五官，大尺度看整体轮廓；
- **图像融合**：小尺度融合细节，大尺度融合整体亮度；
- **图像压缩**：大尺度保存整体，小尺度保存细节；
- **去噪**：小尺度噪声多，大尺度噪声被平均掉；
- **边缘检测**：小尺度检测细边缘，大尺度检测粗边缘。

#### 三、高斯金字塔

- 向下采样（`pyrDown`）：先高斯模糊，再删去偶数行偶数列，尺寸减半。
- 向上采样（`pyrUp`）：先插零扩展，再高斯模糊，尺寸加倍。
- 向上一层是低分辨率、低通滤波后的图像。

#### 四、拉普拉斯金字塔

**它到底在干什么**：

> **拉普拉斯金字塔 = 原图 − 升采样后的模糊图。**

具体说：

- 先对原图降采样，再升采样回去；
- 升采样回来的图，尺寸和原图一样，但比原图模糊；
- 用原图减去它，剩下的就是"丢失的细节"。

**它要解决的问题**：

> 高斯金字塔是有损的：每次 `pyrDown` 都会丢掉细节。
>
> 拉普拉斯金字塔就是**把每一层丢失的细节单独保存下来**。

**具体怎么算**：

```text
G0 = 原图
G1 = pyrDown(G0)
G2 = pyrDown(G1)
...

L0 = G0 − pyrUp(G1)
L1 = G1 − pyrUp(G2)
L2 = G2 − pyrUp(G3)
...
```

注意尺寸问题：`pyrUp` 后的尺寸可能和上一层不完全一样，所以要先裁剪或 resize。

**怎么用它重建原图**：

```text
G0 = pyrUp(G1) + L0
G1 = pyrUp(G2) + L1
G2 = pyrUp(G3) + L2
...
```

从最顶层开始，一层一层往上加，最终得到的 G0 就是原图。这就是"无损重建"。

**它到底用来干什么**：

| 用途 | 说明 |
|------|------|
| 图像压缩 | 只存高斯金字塔 + 拉普拉斯金字塔，可以重建原图 |
| 图像融合 | 把两张图的金字塔按规则合并，再重建，得到融合图 |
| 多尺度分析 | 每一层保存不同尺度的细节，便于分析 |
| 图像增强 | 放大某一层的细节，再重建，得到增强图 |
| 去噪 | 修改细节层，再重建，得到去噪图 |

**一个直观类比**：

把图像想象成一栋楼：

- 高斯金字塔 = 每层楼的**大致结构**；
- 拉普拉斯金字塔 = 每层楼的**装修细节**；
- 只保存结构，重建出来是毛坯房；
- 结构 + 细节，才能还原成精装房。

**一句话总结**：

> **拉普拉斯金字塔 = 原图 − 升采样后的模糊图。**
>
> 它的作用是：
>
> 1. **保存每一层丢失的细节**；
> 2. **和高斯金字塔配合，可以无损重建原图**；
> 3. 常用于图像压缩、融合、增强、去噪。
>
> 算法关键：
>
> - `L_i = G_i − pyrUp(G_{i+1})`；
> - 重建时：`G_i = pyrUp(G_{i+1}) + L_i`。

#### 五、OpenCV 中的调用

```python
cv2.pyrDown(src)        # 向下采样
cv2.pyrUp(src)          # 向上采样
```

### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

img = cv2.imread('P&V/cameraman.bmp')                        # 读取 cameraman 图片
if img is not None:                                          # 如果图片存在
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)         # 转换为灰度图
    down1 = cv2.pyrDown(img_gray)                            # 向下采样一次
    down2 = cv2.pyrDown(down1)                               # 向下采样两次
    up1 = cv2.pyrUp(down1)                                   # 向上采样一次
    if up1.shape != img_gray.shape:                          # 尺寸匹配
        up1 = cv2.resize(up1, (img_gray.shape[1], img_gray.shape[0]))  # 调整尺寸

    # 拉普拉斯金字塔：用 float 计算，保留正负细节
    laplacian_pyr = img_gray.astype(np.float32) - up1.astype(np.float32)

    # 归一化到 0~255，让细节显示出来
    laplacian_vis = cv2.normalize(laplacian_pyr, None, 0, 255, cv2.NORM_MINMAX)
    laplacian_vis = laplacian_vis.astype(np.uint8)

    plt.figure(figsize=(15, 4))                              # 创建画布

    plt.subplot(1, 4, 1)                                     # 子图 1
    plt.title('原图')                                         # 标题
    plt.imshow(img_gray, cmap='gray')                        # 显示原图
    plt.axis('off')                                          # 关闭坐标轴

    plt.subplot(1, 4, 2)                                     # 子图 2
    plt.title('pyrDown 1')                                   # 标题
    plt.imshow(down1, cmap='gray')                           # 显示 pyrDown 1
    plt.axis('off')                                          # 关闭坐标轴

    plt.subplot(1, 4, 3)                                     # 子图 3
    plt.title('pyrDown 2')                                   # 标题
    plt.imshow(down2, cmap='gray')                           # 显示 pyrDown 2
    plt.axis('off')                                          # 关闭坐标轴

    plt.subplot(1, 4, 4)                                     # 子图 4
    plt.title(r'拉普拉斯金字塔 $L_0 = G_0 - \mathrm{pyrUp}(G_1)$')         # 标题
    plt.imshow(laplacian_vis, cmap='gray')                   # 显示拉普拉斯金字塔
    plt.axis('off')                                          # 关闭坐标轴

    plt.tight_layout()                                       # 自动调整间距
    plt.show()                                               # 显示图像
else:
    print("请确保 'P&V/cameraman.bmp' 存在")                  # 提示信息

**图像解读**

一、四张子图分别是什么

| 子图 | 内容 | 对应代码 |
|------|------|---------|
| 左：原图 | 输入的 cameraman 图 | `img_gray` |
| 中：pyrDown 1 | 向下采样一次 | `cv2.pyrDown(img_gray)` |
| 中右：pyrDown 2 | 向下采样两次 | `cv2.pyrDown(down1)` |
| 右：拉普拉斯金字塔 | 原图 − pyrUp(pyrDown 1) | `img_gray - pyrUp(down1)` |

---

二、观察到的现象

1. **原图**：人物轮廓、相机、三脚架、地平线、建筑边缘清晰可见。
2. **pyrDown 1**：尺寸减半，细节减少，但整体结构保留。
3. **pyrDown 2**：尺寸再减半，更模糊，只剩大致轮廓。
4. **拉普拉斯金字塔**：亮的地方集中在人物轮廓、相机、三脚架、地平线、建筑边缘，背景天空、草地基本为暗。

---

三、这说明了什么

1. **高斯金字塔向下采样**：
   - 尺寸减半；
   - 细节减少；
   - 图像变模糊。

2. **拉普拉斯金字塔**：
   - 保存的是原图相对 pyrDown 1 丢失的高频细节；
   - 归一化后，边缘清晰可见，不再全黑；
   - 它和高斯金字塔配合，可以无损重建原图。

3. **cameraman 的优势**：
   - 边缘清晰、噪声少；
   - 拉普拉斯金字塔干净、不噪；
   - 比用 cat.jpg 清楚得多。

---

四、小结

> 这张图用 cameraman 演示了图像金字塔：
>
> - 原图 → pyrDown 1 → pyrDown 2，尺寸减半、细节减少；
> - 拉普拉斯金字塔保存的是高频细节；
> - 归一化后细节清晰，不再全黑。

## 6.6 形态学

形态学（Mathematical Morphology）是图像处理中基于**形状**的一类操作。它以结构元（Structuring Element）为“探针”在图像上滑动，通过集合运算（交、并、补）来处理图像中的几何结构。

本小节从**二值形态学基础**开始，逐步过渡到**灰度形态学**，最后介绍形态学在图像分析中的**典型算法与应用**。

> **约定**：如无特殊说明，本节默认按 OpenCV 的约定——**白色（255）= 前景，黑色（0）= 背景**。腐蚀、膨胀等操作直接作用在“白色前景”上。

> **注意术语**：形态学中讨论的“边缘/边界”指**画面内部的物体轮廓**，不是图像最外圈的边框。参见 6.0 节对“边缘”的定义。

### 6.6.1 形态学基础：腐蚀、膨胀

**形态学操作**以形状为基础，用来处理二值或灰度图像的几何结构。

**结构元（Structuring Element）**：一个小窗口（如 3×3、5×5），定义了“邻域形状”。常见的有矩形、椭圆形、十字形。

```python
cv2.getStructuringElement(shape, ksize)
```
- `shape`：`MORPH_RECT`、`MORPH_ELLIPSE`、`MORPH_CROSS`。

**腐蚀（Erosion）**：
- 用结构元滑过图像，若结构元覆盖的区域内全部是前景（1），则中心像素保留前景；否则变为背景（0）。
- 效果：缩小前景、断开窄连接、去除小物体。

**膨胀（Dilation）**：
- 若结构元覆盖区域内只要有一个前景，则中心像素变为前景。
- 效果：扩大前景、填补小孔、连接断裂。

```python
cv2.erode(src, kernel)
cv2.dilate(src, kernel)
```

**注意**：OpenCV 的约定是：白色（255）= 前景；黑色（0）= 背景。

#### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

img = cv2.imread('P&V/noisy-fingerprint.png')                # 读取噪声指纹图片
if img is not None:                                          # 如果图片存在
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)         # 转换为灰度图
    _, binary = cv2.threshold(img_gray, 127, 255, cv2.THRESH_BINARY)  # 二值化
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))  # 结构元
    eroded = cv2.erode(binary, kernel)                       # 腐蚀
    dilated = cv2.dilate(binary, kernel)                     # 膨胀

    plt.figure(figsize=(15, 4))                              # 创建画布

    plt.subplot(1, 3, 1)                                     # 子图 1
    plt.title('二值图')                                       # 标题
    plt.imshow(binary, cmap='gray')                          # 显示二值图
    plt.axis('off')                                          # 关闭坐标轴

    plt.subplot(1, 3, 2)                                     # 子图 2
    plt.title('腐蚀（白纹变细）')                              # 标题
    plt.imshow(eroded, cmap='gray')                          # 显示腐蚀结果
    plt.axis('off')                                          # 关闭坐标轴

    plt.subplot(1, 3, 3)                                     # 子图 3
    plt.title('膨胀（白纹变粗）')                              # 标题
    plt.imshow(dilated, cmap='gray')                         # 显示膨胀结果
    plt.axis('off')                                          # 关闭坐标轴

    plt.tight_layout()                                       # 自动调整间距
    plt.show()                                               # 显示图像
else:
    print("请确保 'P&V/noisy-fingerprint.png' 存在")          # 提示信息

**图像解读**

| 子图 | 内容 | 对应代码 |
|------|------|---------|
| 左：二值图 | 噪声指纹经过阈值 127 二值化后的黑白图 | `cv2.threshold(img_gray, 127, 255, cv2.THRESH_BINARY)` |
| 中：腐蚀（白纹变细） | 对二值图做腐蚀，结构元 3×3 | `cv2.erode(binary, kernel)` |
| 右：膨胀（白纹变粗） | 对二值图做膨胀，结构元 3×3 | `cv2.dilate(binary, kernel)` |

---

前景和背景是什么

这张图是**黑底白纹**：

- 白色指纹纹路 → 灰度 > 127 → 变成 255（白）→ **前景**；
- 黑色底板 + 噪声 → 灰度 ≤ 127 → 变成 0（黑）→ **背景**。

所以：

- **前景 = 白色纹路**；
- **背景 = 黑色底板**。

这和视觉直觉一致，腐蚀、膨胀就作用在白色纹路上。

---

观察到的现象

**二值图**

- 白色指纹纹路清晰；
- 背景有大量小白点噪声；
- 纹路上有断裂、断点。

**腐蚀（白纹变细）**

- 白色纹路**明显变细、变短**；
- 细的纹路直接**断开或消失**；
- 背景的小白点噪声**被腐蚀掉**，背景变得干净；
- 整体前景**收缩**。

**膨胀（白纹变粗）**

- 白色纹路**明显变粗、变长**；
- 相邻纹路开始**粘连**；
- 背景的小白点噪声**被放大**，噪声更明显；
- 整体前景**扩张**。

---

这说明了什么

**腐蚀的作用**

> 腐蚀 = 结构元覆盖区域**全部是前景**时，中心像素才保留前景。

效果：

- 前景缩小；
- 细线断开；
- 小噪点被抹掉。

本图中：白纹变细、噪点被去掉，正是腐蚀的典型效果。

**膨胀的作用**

> 膨胀 = 结构元覆盖区域**只要有一个前景**，中心像素就变为前景。

效果：

- 前景扩大；
- 断裂被连接；
- 小噪点被放大。

本图中：白纹变粗、噪点被放大，正是膨胀的典型效果。

**腐蚀和膨胀是相反的操作**

| 项目 | 腐蚀 | 膨胀 |
|---|---|---|
| 前景 | 缩小 | 扩大 |
| 细线 | 断开 | 粘连 |
| 小噪点 | 被抹掉 | 被放大 |
| 结构元 | 3×3 | 3×3 |
| 效果 | 白纹变细 | 白纹变粗 |

**为什么噪声指纹适合做这个演示**

- 白纹黑底，前景清晰；
- 纹路细，腐蚀、膨胀的变化肉眼可见；
- 噪声多，可以顺带展示“腐蚀去噪、膨胀放大噪声”；
- 比干净指纹更能说明形态学的实际作用。

**结构元大小的影响**

本图用 3×3 结构元：

- 腐蚀后纹路变细但不至于全部消失；
- 膨胀后纹路变粗但不至于糊成一团。

如果换成 5×5：

- 腐蚀会把纹路断得更碎、甚至消失；
- 膨胀会把纹路粘成一片，分不清纹路。

所以 3×3 对噪声指纹来说是比较合适的选择。

---

实际应用上的意义

- **腐蚀**常用于：去噪、去除小物体、断开窄连接、细化纹路；
- **膨胀**常用于：填补小孔、连接断裂、加粗纹路；
- **先腐蚀后膨胀（开运算）**：去噪、去除小物体，同时保持整体形状；
- **先膨胀后腐蚀（闭运算）**：填补小孔、连接断裂。

本图只用了一步腐蚀 / 膨胀，作为形态学基础演示已经足够；实际中通常会组合使用。

---

一句话总结

> 这张图用 `noisy-fingerprint.png` 清楚演示了形态学的两个基本操作：
>
> - **腐蚀**：白纹变细，细线断开，噪点被抹掉；
> - **膨胀**：白纹变粗，相邻粘连，噪点被放大。
>
> 黑底白纹，前景清晰，噪声明显，比猫图和干净指纹图都更适合演示腐蚀、膨胀。

### 6.6.2 形态学高级运算：开运算、闭运算、梯度、顶帽、黑帽、击中击不中

基于腐蚀和膨胀的组合运算，形态学可以实现许多高级操作。

- **开运算（Opening）**：先腐蚀后膨胀。作用：去噪、断开物体间的窄连接、去除小物体。
  $$A \circ B = (A \ominus B) \oplus B$$
- **闭运算（Closing）**：先膨胀后腐蚀。作用：填补物体内部小孔、连接邻近物体。
  $$A \bullet B = (A \oplus B) \ominus B$$
- **形态学梯度（Gradient）**：膨胀减腐蚀。作用：提取物体轮廓。
  $$G = (A \oplus B) - (A \ominus B)$$
- **顶帽（Top-hat）**：原图减开运算。作用：提取比背景亮的细节。
  $$T_{hat} = A - (A \circ B)$$
- **黑帽（Black-hat）**：闭运算减原图。作用：提取比背景暗的细节。
  $$B_{hat} = (A \bullet B) - A$$
- **击中击不中（Hit-or-Miss）**：用两个互补的结构元同时匹配前景和背景，用于形状检测、角点检测、细化。

#### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

img = cv2.imread('P&V/noisy-fingerprint.png')                # 读取噪声指纹图片
if img is not None:                                          # 如果图片存在
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)         # 转换为灰度图
    _, binary = cv2.threshold(img_gray, 127, 255, cv2.THRESH_BINARY)  # 二值化
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))  # 结构元

    opening = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)  # 开运算
    closing = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)  # 闭运算
    gradient = cv2.morphologyEx(binary, cv2.MORPH_GRADIENT, kernel)  # 梯度
    tophat = cv2.morphologyEx(binary, cv2.MORPH_TOPHAT, kernel)  # 顶帽
    blackhat = cv2.morphologyEx(binary, cv2.MORPH_BLACKHAT, kernel)  # 黑帽

    plt.figure(figsize=(15, 8))                              # 创建画布

    titles = ['二值图', '开运算', '闭运算', '梯度', '顶帽', '黑帽']  # 标题列表
    images = [binary, opening, closing, gradient, tophat, blackhat]  # 图像列表

    for i in range(6):                                       # 遍历
        plt.subplot(2, 3, i+1)                               # 子图
        plt.title(titles[i])                                 # 标题
        plt.imshow(images[i], cmap='gray')                   # 显示图像
        plt.axis('off')                                      # 关闭坐标轴

    plt.tight_layout()                                       # 自动调整间距
    plt.show()                                               # 显示图像
else:
    print("请确保 'P&V/noisy-fingerprint.png' 存在")          # 提示信息

**图像解读**

| 子图 | 内容 | 对应代码 |
|------|------|---------|
| 二值图 | 噪声指纹经过阈值 127 二值化 | `cv2.threshold(img_gray, 127, 255, cv2.THRESH_BINARY)` |
| 开运算 | 先腐蚀后膨胀 | `cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)` |
| 闭运算 | 先膨胀后腐蚀 | `cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)` |
| 梯度 | 膨胀减腐蚀 | `cv2.morphologyEx(binary, cv2.MORPH_GRADIENT, kernel)` |
| 顶帽 | 原图减开运算 | `cv2.morphologyEx(binary, cv2.MORPH_TOPHAT, kernel)` |
| 黑帽 | 闭运算减原图 | `cv2.morphologyEx(binary, cv2.MORPH_BLACKHAT, kernel)` |

---

前景和背景是什么

这张图是**黑底白纹**：

- 白色指纹纹路 → 前景；
- 黑色底板 + 噪声 → 背景。

OpenCV 约定：**白色（255）= 前景，黑色（0）= 背景**。  
所以腐蚀、膨胀作用在白色纹路上。

---

逐图观察

**二值图**

- 白色指纹纹路清晰；
- 背景有大量小白点噪声；
- 纹路上有断点、断裂。

**开运算**

- 背景上的小白点噪声**被去掉**；
- 细的纹路断开、断裂；
- 整体前景收缩，图像更“干净”。

**闭运算**

- 纹路上的断点**被填平**；
- 相邻纹路**粘连**；
- 背景噪声也被稍微放大。

**梯度**

- 白色纹路的**轮廓**被提取出来；
- 边缘亮、内部暗；
- 像一张“描边图”。

**顶帽**

- 提取出**被开运算去掉的小白点、小亮块**；
- 主要是背景的小白点、纹路的细断点；
- 原图中大块的白色纹路被减去，只剩小细节。

**黑帽**

- 提取出**被闭运算填掉的小黑孔、小黑点**；
- 本图中白色前景内部的黑孔很少，所以黑帽偏暗；
- 但背景里少量孤立的小黑点仍能看出来。

---

这说明了什么

**开运算**

> 先腐蚀（去掉小物体、细连接），再膨胀（恢复主体形状）。  
> 效果：**去噪、断开窄连接、去除小物体**。

本图中：背景小白点被去掉，细纹路断开。

**闭运算**

> 先膨胀（填补小孔、连接断裂），再腐蚀（恢复主体形状）。  
> 效果：**填补小孔、连接邻近物体**。

本图中：纹路断点被填平，相邻纹路粘连。

**梯度**

> 膨胀减腐蚀。  
> 效果：**提取物体轮廓**。

本图中：白纹轮廓被清晰提取。

**顶帽**

> 原图减开运算。  
> 效果：**提取“被开运算去掉的小白点、小亮块”**。

本图中：背景小白点、纹路细断点被提取。

**黑帽**

> 闭运算减原图。  
> 效果：**提取“被闭运算填掉的小黑孔、小黑点”**。

本图中：白色前景内部黑孔很少，黑帽偏暗。

---

五种运算对比

| 运算 | 定义 | 作用 | 本图效果 |
|---|---|---|---|
| 开运算 | 先腐蚀后膨胀 | 去噪、断开窄连接、去除小物体 | 背景小白点被去掉 |
| 闭运算 | 先膨胀后腐蚀 | 填补小孔、连接邻近物体 | 纹路断点被填平 |
| 梯度 | 膨胀减腐蚀 | 提取轮廓 | 白纹轮廓清晰 |
| 顶帽 | 原图减开运算 | 提取“被开运算去掉的小白点” | 小白点、细断点 |
| 黑帽 | 闭运算减原图 | 提取“被闭运算填掉的小黑孔” | 偏暗，细节较少 |

---

结构元大小的影响

本图用 3×3 结构元：

- 开运算能去掉小白点，但不至于把纹路断得太碎；
- 闭运算能填断点，但不至于把纹路粘成一团。

如果换成 5×5：

- 开运算会把细纹路断得更碎；
- 闭运算会把纹路粘成一片，分不清纹路。

3×3 对噪声指纹来说是比较合适的选择。

---

一句话总结

> 这张图用 `noisy-fingerprint.png` 清楚演示了形态学的五种高级运算：
>
> - **开运算**：去噪、断开窄连接；
> - **闭运算**：填孔、连接断裂；
> - **梯度**：提取轮廓；
> - **顶帽**：提取“被开运算去掉的小白点、小亮块”；
> - **黑帽**：提取“被闭运算填掉的小黑孔、小黑点”（本图偏暗）。


### 6.6.3 形态学处理函数：morphologyEx

`cv2.morphologyEx()` 是 OpenCV 中所有高级形态学操作的**统一入口**。通过传入不同的 `op` 参数来调用不同的运算。

```python
cv2.morphologyEx(src, op, kernel, anchor, iterations, borderType, borderValue)
```

参数说明：
- `op`：操作类型
  - `MORPH_OPEN`：开运算
  - `MORPH_CLOSE`：闭运算
  - `MORPH_GRADIENT`：形态学梯度
  - `MORPH_TOPHAT`：顶帽
  - `MORPH_BLACKHAT`：黑帽
  - `MORPH_HITMISS`：击中击不中（用于二值图）
- `kernel`：结构元。
- `iterations`：迭代次数（对同一操作重复几次）。

**算法过程**：以开运算为例，等价于连续做 `iterations` 次腐蚀后，再做 `iterations` 次膨胀。

#### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

img = cv2.imread('P&V/noisy-fingerprint.png')                # 读取噪声指纹图片
if img is not None:                                          # 如果图片存在
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)         # 转换为灰度图
    _, binary = cv2.threshold(img_gray, 127, 255, cv2.THRESH_BINARY)  # 二值化
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))  # 结构元
    opening_iter = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=2)  # 开运算，迭代 2 次

    plt.figure(figsize=(10, 5))                              # 创建画布

    plt.subplot(1, 2, 1)                                     # 子图 1
    plt.title('二值图')                                       # 标题
    plt.imshow(binary, cmap='gray')                          # 显示二值图
    plt.axis('off')                                          # 关闭坐标轴

    plt.subplot(1, 2, 2)                                     # 子图 2
    plt.title('开运算 (迭代 2 次)')                             # 标题
    plt.imshow(opening_iter, cmap='gray')                    # 显示开运算结果
    plt.axis('off')                                          # 关闭坐标轴

    plt.tight_layout()                                       # 自动调整间距
    plt.show()                                               # 显示图像
else:
    print("请确保 'P&V/noisy-fingerprint.png' 存在")          # 提示信息

**解读**

| 子图 | 内容 | 对应代码 |
|------|------|---------|
| 左：二值图 | 噪声指纹经过阈值 127 二值化 | `cv2.threshold(img_gray, 127, 255, cv2.THRESH_BINARY)` |
| 右：开运算（迭代 2 次） | 结构元 3×3，迭代 2 次 | `cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=2)` |

---

前景和背景是什么

这张图是黑底白纹：

- 白色指纹纹路 → 前景；
- 黑色底板 + 噪声 → 背景。

OpenCV 约定：白色（255）= 前景，黑色（0）= 背景。  
所以腐蚀、膨胀作用在白色纹路上。

---

观察到的现象

左：二值图

- 白色指纹纹路清晰；
- 背景有大量小白点噪声；
- 纹路上有断点、断裂。

右：开运算（迭代 2 次）

- 背景上的小白点噪声**几乎全被去掉**；
- 指纹纹路**大量断开**，只剩少量短线段；
- 整体前景大幅收缩，图像非常“干净”，但也**丢了很多纹路**。

---

这说明了什么

开运算的定义

> 开运算 = 先腐蚀，后膨胀。  
> 作用：去噪、断开窄连接、去除小物体。

迭代 2 次意味着什么

`iterations=2` 等价于：

```text
先腐蚀 2 次 → 再膨胀 2 次
```

比迭代 1 次（`iterations=1`）腐蚀得更深、膨胀得更多，因此：

- 去噪能力更强；
- 对细节的破坏也更大。

---

一句话总结

> 同一个开运算，`iterations` 从 1 增加到 2，相当于把腐蚀和膨胀各做两次，去噪更强，但纹路断得更碎。
>
> 实际使用时，`iterations` 决定了形态学操作的“力度”：
>
> - 1 次：温和；
> - 2 次以上：力度翻倍，容易过度处理。

### 6.6.4 灰度形态学：灰度腐蚀、灰度膨胀、灰度开闭、灰度顶帽/底帽

形态学可以直接应用到灰度图。与二值形态学不同，灰度形态学不再处理"前景/背景"二值，而是直接处理像素的灰度值。

**灰度腐蚀**：
- 结构元滑过图像，每个位置取窗口内**最小值**作为输出。
- 效果：图像整体变暗，亮区域变小，暗区域变大。

**灰度膨胀**：
- 每个位置取窗口内**最大值**作为输出。
- 效果：图像整体变亮，亮区域变大。

**灰度开运算**：先腐蚀后膨胀。作用：去除比结构元小的亮细节。
**灰度闭运算**：先膨胀后腐蚀。作用：去除比结构元小的暗细节。
**灰度顶帽**：原图减开运算。作用：提取比周围亮的细节。
**灰度底帽（Black-hat）**：闭运算减原图。作用：提取比周围暗的细节。

**算法过程（灰度腐蚀）**：
1. 将结构元平移到当前像素。
2. 在结构元覆盖的区域内取最小值。
3. 输出该最小值作为当前像素。

#### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

# ---------- 合成灰度图 ----------
img_gray = np.full((300, 300), 128, dtype=np.uint8)          # 灰底（128）

# 亮方块（比背景亮，用于开运算、闭运算、顶帽）
cv2.rectangle(img_gray, (60, 60), (240, 240), 220, -1)       # 亮方块（220）

# 亮方块内部的小暗点（比背景暗，用于黑帽）
cv2.circle(img_gray, (120, 120), 6, 60, -1)                  # 小暗点
cv2.circle(img_gray, (180, 180), 6, 60, -1)                  # 小暗点

# 背景上的小亮点（比背景亮，用于顶帽）
cv2.circle(img_gray, (30, 30), 4, 220, -1)                   # 小亮点
cv2.circle(img_gray, (270, 270), 4, 220, -1)                 # 小亮点
cv2.circle(img_gray, (30, 270), 4, 220, -1)                  # 小亮点
cv2.circle(img_gray, (270, 30), 4, 220, -1)                  # 小亮点

# ---------- 灰度形态学 ----------
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))  # 结构元

eroded = cv2.erode(img_gray, kernel)                          # 灰度腐蚀
dilated = cv2.dilate(img_gray, kernel)                        # 灰度膨胀
opening = cv2.morphologyEx(img_gray, cv2.MORPH_OPEN, kernel)  # 灰度开运算
closing = cv2.morphologyEx(img_gray, cv2.MORPH_CLOSE, kernel) # 灰度闭运算
tophat = cv2.morphologyEx(img_gray, cv2.MORPH_TOPHAT, kernel) # 灰度顶帽
blackhat = cv2.morphologyEx(img_gray, cv2.MORPH_BLACKHAT, kernel)  # 灰度黑帽

# ---------- 可视化 ----------
plt.figure(figsize=(15, 10))                                  # 创建画布

titles = ['原灰度图', '灰度腐蚀', '灰度膨胀',
          '灰度开运算', '灰度闭运算', '灰度顶帽', '灰度黑帽']  # 标题列表
images = [img_gray, eroded, dilated,
          opening, closing, tophat, blackhat]                 # 图像列表

for i in range(7):                                            # 遍历
    plt.subplot(2, 4, i+1)                                    # 子图
    plt.title(titles[i])                                      # 标题
    plt.imshow(images[i], cmap='gray', vmin=0, vmax=255)      # 显示图像
    plt.axis('off')                                           # 关闭坐标轴

plt.tight_layout()                                            # 自动调整间距
plt.show()                                                    # 显示图像

**图像解读**

| 子图 | 内容 | 对应代码 |
|------|------|---------|
| 原灰度图 | 灰底 + 亮方块 + 亮方块内部的小暗点 + 背景上的小亮点 | `np.full(...)` + `cv2.rectangle(...)` + `cv2.circle(...)` |
| 灰度腐蚀 | 取窗口最小值 | `cv2.erode(img_gray, kernel)` |
| 灰度膨胀 | 取窗口最大值 | `cv2.dilate(img_gray, kernel)` |
| 灰度开运算 | 先腐蚀后膨胀 | `cv2.morphologyEx(img_gray, cv2.MORPH_OPEN, kernel)` |
| 灰度闭运算 | 先膨胀后腐蚀 | `cv2.morphologyEx(img_gray, cv2.MORPH_CLOSE, kernel)` |
| 灰度顶帽 | 原图减开运算 | `cv2.morphologyEx(img_gray, cv2.MORPH_TOPHAT, kernel)` |
| 灰度黑帽 | 闭运算减原图 | `cv2.morphologyEx(img_gray, cv2.MORPH_BLACKHAT, kernel)` |

---

前景和背景是什么

这张图是**灰底 + 亮方块 + 小细节**：

- 灰底（128）= 背景；
- 亮方块（220）= 前景（比背景亮）；
- 亮方块内部的小暗点（60）= 暗细节；
- 背景上的小亮点（220）= 亮细节。

灰度形态学不区分“前景 / 背景”二值，而是直接对灰度值取窗口最小值 / 最大值。

---

逐图观察

**原灰度图**

- 灰底 128；
- 亮方块 220；
- 亮方块内部有 2 个小暗点（60）；
- 背景四角有 4 个小亮点（220）。

**灰度腐蚀**

- 取窗口最小值，图像整体**变暗**；
- 亮方块**收缩**；
- 背景上的小亮点**消失**；
- 亮方块内部的小暗点**扩张**（变粗）。

**灰度膨胀**

- 取窗口最大值，图像整体**变亮**；
- 亮方块**扩张**；
- 背景上的小亮点**扩张**（变粗）；
- 亮方块内部的小暗点**被填平**。

**灰度开运算**

- 先腐蚀后膨胀；
- 背景上的小亮点**被去掉**；
- 亮方块**保留**；
- 亮方块内部的小暗点**保留**。

**灰度闭运算**

- 先膨胀后腐蚀；
- 亮方块内部的小暗点**被填平**；
- 亮方块**保留**；
- 背景上的小亮点**保留**。

**灰度顶帽**

- 原图减开运算；
- 提取出**比周围亮的小细节**；
- 结果是背景上的 4 个小亮点。

**灰度黑帽**

- 闭运算减原图；
- 提取出**比周围暗的小细节**；
- 结果是亮方块内部的 2 个小暗点。

---

这说明了什么

**灰度腐蚀**

> 取窗口最小值，图像整体变暗，亮的区域收缩，暗的区域扩张。

本图中：亮方块收缩，小亮点消失，小暗点扩张。

**灰度膨胀**

> 取窗口最大值，图像整体变亮，亮的区域扩张，暗的区域收缩。

本图中：亮方块扩张，小亮点扩张，小暗点被填。

**灰度开运算**

> 先腐蚀后膨胀。  
> 效果：**去掉比结构元小的亮细节**。

本图中：背景上的小亮点被去掉，亮方块保留。

**灰度闭运算**

> 先膨胀后腐蚀。  
> 效果：**去掉比结构元小的暗细节**。

本图中：亮方块内部的小暗点被填平。

**灰度顶帽**

> 原图减开运算。  
> 效果：**提取比周围亮的小细节**。

本图中：背景上的 4 个小亮点被提取出来。

**灰度黑帽**

> 闭运算减原图。  
> 效果：**提取比周围暗的小细节**。

本图中：亮方块内部的 2 个小暗点被提取出来。

---

五种运算对比

| 运算 | 定义 | 作用 | 本图效果 |
|---|---|---|---|
| 灰度腐蚀 | 取窗口最小值 | 图像变暗，亮区域收缩 | 亮方块收缩、小亮点消失 |
| 灰度膨胀 | 取窗口最大值 | 图像变亮，亮区域扩张 | 亮方块扩张、小暗点被填 |
| 灰度开运算 | 先腐蚀后膨胀 | 去掉比结构元小的亮细节 | 背景小亮点被去掉 |
| 灰度闭运算 | 先膨胀后腐蚀 | 去掉比结构元小的暗细节 | 亮方块内部小暗点被填 |
| 灰度顶帽 | 原图减开运算 | 提取比周围亮的小细节 | 背景小亮点被提取 |
| 灰度黑帽 | 闭运算减原图 | 提取比周围暗的小细节 | 亮方块内部小暗点被提取 |

---

结构元大小的影响

本图用 15×15 结构元：

- 比小亮点、小暗点都大 → 开运算能去掉小亮点，闭运算能填掉小暗点；
- 比亮方块小 → 亮方块在开、闭运算后基本保留。

如果换成 3×3：

- 开运算去不掉小亮点；
- 闭运算填不掉小暗点。

如果换成 31×31 或更大：

- 亮方块也会被去掉或填平，效果不再是“去噪”而是“大范围平滑”。

所以 15×15 对这张合成图来说是比较合适的选择。

---

一句话总结

> 这张合成灰度图清楚演示了灰度形态学的六种运算：
>
> - **灰度腐蚀**：图像变暗，亮区收缩，小亮点消失；
> - **灰度膨胀**：图像变亮，亮区扩张，小暗点被填；
> - **灰度开运算**：去掉比结构元小的亮细节（小亮点）；
> - **灰度闭运算**：去掉比结构元小的暗细节（小暗点）；
> - **灰度顶帽**：提取比周围亮的小细节；
> - **灰度黑帽**：提取比周围暗的小细节。
>
> 合成图完全可控，六种效果全部明显，比猫图、人像更适合演示灰度形态学。

### 6.6.5 边界提取

**边界提取（Boundary Extraction）**：
- 用原图减去腐蚀后的图像：$B = A - (A \ominus S)$。
- 得到的 $B$ 是物体的轮廓（1 像素宽）。

**算法过程**：
1. 对二值图做腐蚀。
2. 用原图减去腐蚀结果。
3. 差值为边界。

#### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

# ---------- 合成二值图：黑底 + 白色几何形状 ----------
binary = np.zeros((300, 300), dtype=np.uint8)                # 纯黑背景

cv2.rectangle(binary, (40, 40), (180, 180), 255, -1)         # 白色方块（实心）
cv2.circle(binary, (110, 110), 25, 0, -1)                    # 方块内部挖一个黑色圆孔

cv2.circle(binary, (230, 90), 40, 255, -1)                   # 右上白色圆形
cv2.rectangle(binary, (180, 200), (270, 260), 255, -1)       # 右下白色矩形
cv2.circle(binary, (230, 230), 15, 0, -1)                    # 矩形内部挖一个小黑孔

# ---------- 边界提取 ----------
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))   # 3×3 结构元

eroded = cv2.erode(binary, kernel)                           # 腐蚀：前景缩小一圈
boundary = cv2.subtract(binary, eroded)                      # 边界 = 原图 - 腐蚀

# ---------- 可视化 ----------
plt.figure(figsize=(15, 5))                                  # 创建画布

plt.subplot(1, 3, 1)                                         # 子图 1
plt.title('合成二值图')                                       # 标题
plt.imshow(binary, cmap='gray', vmin=0, vmax=255)            # 显示二值图
plt.axis('off')                                              # 关闭坐标轴

plt.subplot(1, 3, 2)                                         # 子图 2
plt.title('腐蚀 (前景缩小一圈)')                               # 标题
plt.imshow(eroded, cmap='gray', vmin=0, vmax=255)            # 显示腐蚀结果
plt.axis('off')                                              # 关闭坐标轴

plt.subplot(1, 3, 3)                                         # 子图 3
plt.title('边界提取 (原图 - 腐蚀)')                            # 标题（用普通减号，避免渲染成方框）
plt.imshow(boundary, cmap='gray', vmin=0, vmax=255)          # 显示边界提取结果
plt.axis('off')                                              # 关闭坐标轴

plt.tight_layout()                                           # 自动调整间距
plt.show()                                                   # 显示图像

**图像解读**

| 子图 | 内容 | 对应代码 |
|------|------|---------|
| 左：合成二值图 | 黑底 + 白色方块（含黑孔）+ 白色圆形 + 白色矩形（含小黑孔） | `binary` |
| 中：腐蚀 | 每个白色前景都向内缩小一圈 | `cv2.erode(binary, kernel)` |
| 右：边界提取 | 原图减去腐蚀结果，得到 1 像素宽的轮廓 | `cv2.subtract(binary, eroded)` |

**关键点**：

1. **为什么用合成图**  
   - 背景纯黑、前景纯白，没有噪声干扰；  
   - 腐蚀和相减的效果一目了然；  
   - 不会像自然图像那样因为二值化不干净而出现大量碎片。

2. **腐蚀的作用**  
   - 3×3 结构元让每个白色前景向**内**收缩一圈；  
   - 方块、圆形、矩形都会缩小；  
   - 内部的黑孔反而会**变大**（因为腐蚀对白色前景收缩，黑孔相当于被“让出”更多空间）。

3. **边界提取的原理**  
   - `binary - eroded`：  
     - 前景内部：原图 = 255，腐蚀后 = 255 → 差 = 0（黑）；  
     - 前景边缘一圈：原图 = 255，腐蚀后 = 0 → 差 = 255（白）；  
   - 所以输出正好是**物体外轮廓 + 内孔轮廓**，宽度等于结构元半径。

4. **结构元大小的影响**  
   - 3×3 → 边界宽 1 像素；  
   - 5×5 → 边界宽 2 像素；  
   - 结构元越大，边界越粗，但定位越粗。

5. **和自然图像对比**  
   - 用 `cat.jpg` 做边界提取时，二值化质量差、背景噪声多，结果全是碎片；  
   - 合成图则干净、清晰，能准确展示“原图 − 腐蚀 = 边界”这一核心公式。

**一句话总结**：

> 这张合成图用“黑底 + 白色几何形状”直观演示了边界提取：**腐蚀让前景缩小一圈，原图减去腐蚀结果，就得到 1 像素宽的物体轮廓**。合成图干净可控，比自然图像更适合教学演示。

### 6.6.6 直线提取

**直线提取（Line Extraction）**：
- 用**方向性**结构元（例如 1×n 的水平线、n×1 的垂直线）对图像做开运算。
- 保留符合该方向的直线，去除其他方向的结构。

#### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

# ---------- 合成二值图：黑底 + 横线 + 竖线 + 斜线 ----------
binary = np.zeros((300, 300), dtype=np.uint8)                # 纯黑背景

cv2.line(binary, (30, 80), (270, 80), 255, 3)                # 水平线
cv2.line(binary, (30, 150), (270, 150), 255, 3)              # 水平线
cv2.line(binary, (80, 30), (80, 270), 255, 3)                # 垂直线
cv2.line(binary, (150, 30), (150, 270), 255, 3)              # 垂直线
cv2.line(binary, (200, 30), (270, 100), 255, 3)              # 斜线（45°）

# ---------- 方向性结构元 ----------
kernel_h = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 1))  # 水平长条
kernel_v = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 25))  # 垂直长条

# ---------- 开运算提取直线 ----------
horizontal = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel_h)  # 提取水平线
vertical   = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel_v)  # 提取垂直线

# ---------- 可视化 ----------
plt.figure(figsize=(15, 4))                                  # 创建画布

plt.subplot(1, 3, 1)                                         # 子图 1
plt.title('合成二值图')                                       # 标题
plt.imshow(binary, cmap='gray', vmin=0, vmax=255)            # 显示原图
plt.axis('off')                                              # 关闭坐标轴

plt.subplot(1, 3, 2)                                         # 子图 2
plt.title('水平结构元开运算 (提取水平线)')                     # 标题
plt.imshow(horizontal, cmap='gray', vmin=0, vmax=255)        # 显示水平线提取结果
plt.axis('off')                                              # 关闭坐标轴

plt.subplot(1, 3, 3)                                         # 子图 3
plt.title('垂直结构元开运算 (提取垂直线)')                     # 标题
plt.imshow(vertical, cmap='gray', vmin=0, vmax=255)          # 显示垂直线提取结果
plt.axis('off')                                              # 关闭坐标轴

plt.tight_layout()                                           # 自动调整间距
plt.show()                                                   # 显示图像

**图像解读**

| 子图 | 内容 | 对应代码 |
|------|------|---------|
| 左：合成二值图 | 黑底 + 2 条水平线 + 2 条垂直线 + 1 条斜线 | `binary` |
| 中：水平结构元开运算 | 只剩两条水平线，竖线和斜线被去掉 | `cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel_h)` |
| 右：垂直结构元开运算 | 只剩两条垂直线，横线和斜线被去掉 | `cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel_v)` |

**关键点**：

1. **为什么用开运算**  
   - 开运算 = 先腐蚀后膨胀；  
   - 腐蚀时，只有“能容纳整个结构元”的直线才能存活；  
   - 水平结构元（25×1）只能容纳水平线，竖线和斜线会被腐蚀掉；  
   - 膨胀再把保留下来的直线恢复原宽度。

2. **方向性结构元的选择**  
   - 水平结构元：`(25, 1)`，长 25、宽 1；  
   - 垂直结构元：`(1, 25)`，长 25、高 1；  
   - 长度要大于直线宽度（这里线宽 3），才能有效筛选方向。

3. **斜线为什么被去掉**  
   - 斜线在水平方向上的投影虽然也长，但它的像素是“阶梯状”的；  
   - 水平结构元在腐蚀时要求整条 25×1 区域全是前景，斜线无法满足；  
   - 所以斜线在两种开运算中都会消失。

4. **典型用途**  
   - 表格线提取（先提取横线，再提取竖线）；  
   - 道路标线检测；  
   - 指纹方向场分析；  
   - 文档版面分析。

### 6.6.7 线条细化

**线条细化（Thinning / Skeletonization）**：
- 反复进行形态学腐蚀，同时保持连通性。
- 最终将线条缩减为单像素宽的骨架。
- OpenCV contrib 提供 `cv2.ximgproc.thinning`。

#### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

# ---------- 合成二值图：粗线条图形 ----------
binary = np.zeros((200, 200), dtype=np.uint8)                # 纯黑背景

cv2.line(binary, (20, 100), (180, 100), 255, 9)              # 粗水平线
cv2.line(binary, (100, 20), (100, 180), 255, 9)              # 粗垂直线
cv2.circle(binary, (100, 100), 40, 255, 9)                   # 粗圆环

# ---------- 方法一：使用 cv2.ximgproc.thinning（推荐）----------
try:
    thin = cv2.ximgproc.thinning(binary)                     # 骨架提取
except AttributeError:
    # 如果没有 opencv-contrib，退化为形态学细化（简化版）
    thin = binary.copy()
    kernel = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))
    while True:
        eroded = cv2.erode(thin, kernel)
        temp = cv2.dilate(eroded, kernel)
        temp = cv2.subtract(thin, temp)
        thin = eroded.copy()
        if cv2.countNonZero(temp) == 0:
            break

# ---------- 可视化 ----------
plt.figure(figsize=(10, 5))                                  # 创建画布

plt.subplot(1, 2, 1)                                         # 子图 1
plt.title('合成二值图 (粗线条)')                              # 标题
plt.imshow(binary, cmap='gray', vmin=0, vmax=255)            # 显示原图
plt.axis('off')                                              # 关闭坐标轴

plt.subplot(1, 2, 2)                                         # 子图 2
plt.title('线条细化 (单像素骨架)')                            # 标题
plt.imshow(thin, cmap='gray', vmin=0, vmax=255)              # 显示细化结果
plt.axis('off')                                              # 关闭坐标轴

plt.tight_layout()                                           # 自动调整间距
plt.show()                                                   # 显示图像

**图像解读**

| 子图 | 内容 | 对应代码 |
|------|------|---------|
| 左：合成二值图 | 黑底 + 粗水平线 + 粗垂直线 + 粗圆环 | `binary` |
| 右：线条细化 | 粗线条被缩减为单像素宽的骨架 | `cv2.ximgproc.thinning(binary)` |

**关键点**：

1. **细化的目的**  
   - 把粗线条缩减成单像素宽，便于后续分析（如计数、追踪、形态测量）；  
   - 保留原线条的拓扑结构（连通性、分支、端点）；  
   - 不改变线条的位置和形状。

2. **OpenCV 中的实现**  
   - `cv2.ximgproc.thinning` 需要 `opencv-contrib-python`；  
   - 参数 `THINNING_ZHANGSUEN`（默认）或 `THINNING_GUOHALL`；  
   - 如果没有 contrib，可用形态学腐蚀 + 击中击不中近似实现。

3. **细化和腐蚀的区别**  
   - 腐蚀是无差别地缩小前景，会破坏连通性；  
   - 细化是“有选择地”腐蚀，只去掉不影响连通性的边缘像素；  
   - 最终结果是单像素骨架，而不是整体缩小一圈。

4. **典型用途**  
   - 指纹骨架提取；  
   - 字符笔画细化；  
   - 血管/道路中心线提取；  
   - 手写数字识别预处理。

### 6.6.8 形态学重建：概念与测地膨胀

**形态学重建** 是利用形态学运算在一幅图像中做"种子扩展"的技术。核心运算是**测地膨胀**（用掩模限制的膨胀）。

**算法过程（重建）**：
1. 给定种子图像 $J$ 和掩模图像 $I$（$J \subseteq I$）。
2. 反复做：$D = \min(I, \text{dilate}(D, S))$。
3. 直到稳定，得到重建图像 $R$。

**一句话**：种子在掩模内部不断膨胀，膨胀结果每次都被掩模"剪掉"越界部分，直到不再变化。

**典型应用**：

- 孔洞填充（6.6.9）
- 边界清除（6.6.10）
- 泛洪填充（6.6.11）
- 骨架提取（6.6.12）
- 粒径分离（6.6.13）

### 6.6.9 孔洞填充

**孔洞填充（Hole Filling）**：把物体内部的封闭孔洞填满，物体外形不变。

**算法过程**：
1. 从图像边界 (0,0) 出发，对背景做泛洪填充，标记所有"外部背景"。
2. 反转掩码，得到"内部孔洞 + 前景"。
3. 原图与反转掩码取并，孔洞被填满。

#### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

# 合成图：带孔洞的图形
img = np.zeros((200, 200), dtype=np.uint8)
cv2.circle(img, (60, 60), 40, 255, -1)
cv2.circle(img, (60, 60), 15, 0, -1)
cv2.rectangle(img, (110, 30), (180, 100), 255, -1)
cv2.rectangle(img, (130, 50), (160, 80), 0, -1)

# 孔洞填充
im_floodfill = img.copy()
h, w = img.shape[:2]
mask = np.zeros((h + 2, w + 2), np.uint8)
cv2.floodFill(im_floodfill, mask, (0, 0), 255)
im_floodfill_inv = cv2.bitwise_not(im_floodfill)
im_filled = img | im_floodfill_inv

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title('原图 (带孔洞)')
plt.imshow(img, cmap='gray', vmin=0, vmax=255)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title('孔洞填充')
plt.imshow(im_filled, cmap='gray', vmin=0, vmax=255)
plt.axis('off')
plt.tight_layout()
plt.show()

**图像解读**

| 子图 | 内容 |
|------|------|
| 左：原图 | 带孔圆形 + 带孔矩形 |
| 右：孔洞填充 | 孔洞被填满，物体外形不变 |

**关键点**：

- 种子在边界背景 (0,0)，填的是"被前景包围的背景"；
- 泛洪填充 + 反转 + 取并，是孔洞填充的标准三步；
- 典型用途：缺陷检测、细胞计数、PCB 焊点分析。

### 6.6.10 边界清除

**边界清除（Border Clearing）**：去掉与图像边界接触的物体，保留内部物体。

**算法过程**：
1. 种子 = 图像四条边界上的前景像素；
2. 掩模 = 原图；
3. 反复做"膨胀 + 与掩模相交"，直到收敛；
4. 重建结果 = 所有与边界连通的前景；
5. 原图减去重建结果 = 去掉接触边界的物体。

#### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

# 合成图：内部物体 + 贴边物体
img = np.zeros((200, 200), dtype=np.uint8)
cv2.circle(img, (60, 60), 40, 255, -1)
cv2.circle(img, (60, 60), 15, 0, -1)
cv2.rectangle(img, (110, 30), (180, 100), 255, -1)
cv2.rectangle(img, (130, 50), (160, 80), 0, -1)
cv2.rectangle(img, (170, 120), (199, 190), 255, -1)

# 边界清除
border_seed = np.zeros_like(img)
border_seed[0, :] = img[0, :]
border_seed[-1, :] = img[-1, :]
border_seed[:, 0] = img[:, 0]
border_seed[:, -1] = img[:, -1]

kernel = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))
prev = border_seed.copy()
while True:
    dilated = cv2.dilate(prev, kernel)
    prev_new = cv2.bitwise_and(dilated, img)
    if np.array_equal(prev_new, prev):
        break
    prev = prev_new
border_cleared = cv2.subtract(img, prev)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title('原图 (含贴边矩形)')
plt.imshow(img, cmap='gray', vmin=0, vmax=255)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title('边界清除')
plt.imshow(border_cleared, cmap='gray', vmin=0, vmax=255)
plt.axis('off')
plt.tight_layout()
plt.show()

**图像解读**

| 子图 | 内容 |
|------|------|
| 左：原图 | 内部圆形/矩形 + 贴右边界的矩形 |
| 右：边界清除 | 贴边矩形被去掉，内部物体保留 |

**关键点**：

- 种子在图像边界前景，删的是"和边界连通的前景"；
- 示例图必须让至少一个物体接触边界，否则边界清除无效果；
- 典型用途：去除接触边界的干扰物体、区域分割前处理。

### 6.6.11 泛洪填充

**泛洪填充（Flood Fill）**：从种子点出发，按颜色相似度把连通的同色区域填成新颜色。

**算法过程**：
1. 指定种子点 seedPoint；
2. 从种子点开始，检查邻域像素；
3. 若与种子颜色差在 loDiff/upDiff 范围内，则填为新颜色并继续扩散；
4. 直到没有可扩散的像素。

#### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

# 合成图：两个色块
img = np.zeros((200, 300, 3), dtype=np.uint8)
cv2.rectangle(img, (20, 20), (120, 180), (200, 100, 50), -1)
cv2.rectangle(img, (150, 20), (280, 180), (50, 150, 200), -1)

# 泛洪填充
h, w = img.shape[:2]
mask = np.zeros((h + 2, w + 2), np.uint8)
flood = img.copy()
cv2.floodFill(flood, mask, (70, 100), (0, 255, 0), loDiff=(20, 20, 20), upDiff=(20, 20, 20))

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.title('原图 (两个色块)')
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title('泛洪填充 (左块填绿)')
plt.imshow(cv2.cvtColor(flood, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.tight_layout()
plt.show()

**图像解读**

| 子图 | 内容 |
|------|------|
| 左：原图 | 两个不同颜色块 |
| 右：泛洪填充 | 左侧块被填成绿色，右侧块不受影响 |

**关键点**：

- `mask` 必须比图像大 2（高+2, 宽+2），防止边界扩散越界；
- `loDiff` / `upDiff` 控制颜色相似度容差，容差越大越容易扩散；
- 典型用途：区域生长、连通域标记、图像编辑中的"油漆桶"。

### 6.6.12 骨架提取

**骨架提取（Skeletonization）**：把粗线条缩减为单像素宽的骨架，保留拓扑结构。

**算法过程**：
1. 反复腐蚀前景，但每次只去掉不影响连通性的边缘像素；
2. 直到剩余像素不能再被细化；
3. 结果即单像素宽的骨架。

#### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

# 合成图：粗线条
binary = np.zeros((200, 200), dtype=np.uint8)
cv2.line(binary, (20, 100), (180, 100), 255, 9)
cv2.line(binary, (100, 20), (100, 180), 255, 9)
cv2.circle(binary, (100, 100), 40, 255, 9)

# 骨架提取
try:
    thin = cv2.ximgproc.thinning(binary)
except AttributeError:
    thin = binary.copy()
    kernel = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))
    while True:
        eroded = cv2.erode(thin, kernel)
        temp = cv2.dilate(eroded, kernel)
        temp = cv2.subtract(thin, temp)
        thin = eroded.copy()
        if cv2.countNonZero(temp) == 0:
            break

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title('合成二值图 (粗线条)')
plt.imshow(binary, cmap='gray', vmin=0, vmax=255)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title('骨架提取 (单像素宽)')
plt.imshow(thin, cmap='gray', vmin=0, vmax=255)
plt.axis('off')
plt.tight_layout()
plt.show()

**图像解读**

| 子图 | 内容 |
|------|------|
| 左：合成二值图 | 粗水平线 + 粗垂直线 + 粗圆环 |
| 右：骨架提取 | 单像素宽的骨架 |

**关键点**：

- 骨架提取保留连通性、分支、端点，不改变线条位置；
- OpenCV contrib 提供 `cv2.ximgproc.thinning`；
- 典型用途：指纹骨架、字符笔画细化、血管中心线。

### 6.6.13 粒径分离

**粒径分离（Particle Size Separation）**：用不同大小的结构元做开运算，按尺寸分离物体。

**算法过程**：
1. 选择一系列逐渐增大的结构元 $S_1, S_2, ..., S_n$；
2. 对每个 $S_k$ 做开运算，得到 $O_k$；
3. $O_k$ 保留"能容纳 $S_k$"的物体，去掉更小的；
4. 相邻 $O_{k-1} - O_k$ 即尺寸约为 $k$ 的物体。

#### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

# 合成图：三种尺寸的圆
binary = np.zeros((200, 200), dtype=np.uint8)
cv2.circle(binary, (40, 40), 8, 255, -1)
cv2.circle(binary, (100, 40), 15, 255, -1)
cv2.circle(binary, (160, 40), 25, 255, -1)

# 用不同结构元做开运算
kernel_small = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
kernel_mid   = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (21, 21))
kernel_large = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (35, 35))

open_small = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel_small)
open_mid   = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel_mid)
open_large = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel_large)

plt.figure(figsize=(15, 5))
plt.subplot(1, 4, 1)
plt.title('原图 (三种尺寸)')
plt.imshow(binary, cmap='gray', vmin=0, vmax=255)
plt.axis('off')

plt.subplot(1, 4, 2)
plt.title('开运算 (11×11)')
plt.imshow(open_small, cmap='gray', vmin=0, vmax=255)
plt.axis('off')

plt.subplot(1, 4, 3)
plt.title('开运算 (21×21)')
plt.imshow(open_mid, cmap='gray', vmin=0, vmax=255)
plt.axis('off')

plt.subplot(1, 4, 4)
plt.title('开运算 (35×35)')
plt.imshow(open_large, cmap='gray', vmin=0, vmax=255)
plt.axis('off')

plt.tight_layout()
plt.show()

**图像解读**

| 子图 | 内容 |
|------|------|
| 原图 | 小圆 + 中圆 + 大圆 |
| 开运算 11×11 | 小圆被去掉，中圆、大圆保留 |
| 开运算 21×21 | 小圆、中圆被去掉，只剩大圆 |
| 开运算 35×35 | 全部被去掉 |

**关键点**：

- 结构元越大，能保留的物体越大；
- 相邻两次结果的差，即该尺寸段的物体；
- 典型用途：颗粒尺寸统计、细胞计数、材料粒度分析。

### 6.6.14 基于形态学的粒度测定

**粒度测定（Granulometry）**：用不同大小的结构元去"筛"图像，统计每个尺寸段的物体有多少，从而得到**尺寸分布**。

---

#### 一、先讲清楚：什么叫"粒度"

**粒度** = 物体的大小。

**粒度测定** = 统计图像里"多大的物体有多少个"。

比如一堆沙子，你想知道：

- 直径 1mm 的颗粒有多少；
- 直径 2mm 的颗粒有多少；
- 直径 5mm 的颗粒有多少；
- ……

最后画成一张"横轴是尺寸、纵轴是数量"的曲线，就叫**粒度分布曲线**。

---

#### 二、核心思路：用"筛子"筛物体

想象你有一堆大小不一的球，手边有一组**筛孔大小递增**的筛子：

- 筛子 A：筛孔 5mm；
- 筛子 B：筛孔 10mm；
- 筛子 C：筛孔 20mm；
- ……

**关键操作**：用一个筛孔为 $k$ 的筛子去筛这堆球：

- 直径 **小于** $k$ 的球 → 漏下去，**被筛掉**；
- 直径 **大于等于** $k$ 的球 → 留在筛面上，**被保留**。

于是：

> **筛孔越大，漏下去的球越多，留在筛面上的球越少。**

把"筛孔尺寸"和"留在筛面上的球的数量"画成曲线，就是粒度分布。

---

#### 三、形态学里的"筛子"就是开运算

在形态学里，**开运算**正好扮演"筛子"的角色：

| 现实中的筛子 | 形态学里的开运算 |
|-------------|----------------|
| 筛孔大小 | 结构元尺寸 |
| 球直径 | 物体尺寸 |
| 直径 < 筛孔 → 漏下去 | 物体 < 结构元 → 被开运算去掉 |
| 直径 ≥ 筛孔 → 留在筛面 | 物体 ≥ 结构元 → 保留 |

**为什么开运算能"筛"**：

- 开运算 = 先腐蚀后膨胀；
- 腐蚀时，只有"能容纳整个结构元"的物体才不会被完全腐蚀掉；
- 结构元越大，能"容纳"它的物体越少；
- 所以**结构元越大，开运算后剩下的物体越少**。

这就是"筛子"的形态学版本。

---

#### 四、关键补充：结构元的形状决定"筛孔"的判定方式

**很多人只注意"结构元有多大"，却忽略了"结构元是什么形状"。**

结构元的形状不同，"筛孔"的形状就不同，筛选结果也会不同。

**举一个直观的例子**：图像里有一个**长条形**的物体，比如 30×3 的长条。

**用 5×5 的方形结构元**：

- 长条宽度只有 3，塞不进 5×5 的方形结构元；
- 长条会被开运算去掉；
- 结果："这个长条被当作小于 5×5 的物体"。

**用 5×1 的水平长条结构元**：

- 长条能轻松容纳 5×1 的水平结构元；
- 长条会被保留；
- 结果："这个长条被当作大于等于 5×1 的物体"。

**结论**：

> **同样尺寸、不同形状的结构元，对同一个物体会给出完全不同的"大小判定"。**

**常见结构元形状及其"筛孔"含义**：

| 结构元形状 | 几何含义 | 适合筛什么 |
|-----------|---------|-----------|
| 矩形 `MORPH_RECT` | 方形/长方形窗口 | 有方向性的结构（如表格线） |
| 椭圆/圆 `MORPH_ELLIPSE` | 圆盘形窗口 | **各向同性的颗粒**（最常用于粒度测定） |
| 十字 `MORPH_CROSS` | 十字形窗口 | 细线、骨架相关操作 |

---

#### 五、为什么粒度测定通常用"圆盘形"结构元

**关键理由：各向同性**。

- **圆盘形结构元**在**所有方向上**的"半径"都相等；
- 用它去筛物体，**不管物体朝哪个方向摆放**，判定标准都一致；
- 所以它能测量的是"物体在**各个方向上的最小尺寸**"——也就是物体的**真实宽度/直径**。

**对比方形结构元**：

- 方形结构元在对角线方向比边长方向"更长"（对角线是边长的 √2 倍）；
- 用它筛一个斜放的细长条，结果会**随物体摆放角度变化**；
- 换句话说，方形结构元**不是各向同性**的，测量结果依赖于物体的朝向。

**圆盘形结构元的优势**：

| 特性 | 圆盘形 | 方形 |
|------|--------|------|
| 各向同性 | ✅ 是 | ❌ 否 |
| 测量结果依赖朝向 | 不依赖 | 依赖 |
| 判定的是"物体最小宽度" | ✅ 是 | ❌ 只对轴向物体是 |
| 粒度测定常用 | ✅ 首选 | ⚠️ 只在特定场景用 |

**一句话**：

> **粒度测定时，选圆盘形（`MORPH_ELLIPSE`）结构元，是为了让"筛孔"在每个方向上都一致，从而真实反映物体的尺寸，而不是物体的朝向。**

---

#### 六、算法过程（逐步骤说明）

**第 1 步：准备一组逐渐增大的结构元**

$$S_1, S_2, ..., S_n$$

- 尺寸上递增：比如 1×1、3×3、5×5、……、29×29；
- 形状上统一：比如都用 `MORPH_ELLIPSE`。

**第 2 步：对每个结构元做一次开运算**

$$O_k = \text{open}(I, S_k)$$

$O_k$ 是"用筛孔为 $k$ 的筛子筛过之后，还留在筛面上的物体"。

**第 3 步：数一数 $O_k$ 里还剩多少像素**

$$M_k = \sum O_k$$

$M_k$ 就是"筛孔为 $k$ 时，所有保留下来的物体的总面积"。

**第 4 步：相邻两次的差 = 被这一号筛子筛掉的物体**

$$D_k = M_{k-1} - M_k$$

- $M_{k-1}$：用 $S_{k-1}$ 筛后剩下的像素数；
- $M_k$：用 $S_k$ 筛后剩下的像素数；
- 两者相减 = **筛孔从 $S_{k-1}$ 增大到 $S_k$ 时，被筛掉的那部分物体的像素数**。

这部分物体有多大？——**大小在 $S_{k-1}$ 和 $S_k$ 之间**。

**第 5 步：画曲线**

- 横轴：结构元尺寸 $k$；
- 纵轴：被筛掉的像素数 $D_k$；
- 曲线上的峰 = 那个尺寸段的物体最多。

这就是**粒度分布曲线**。

---

#### 七、一个通俗类比

把粒度测定想象成"分拣橙子"：

| 步骤 | 现实 | 形态学 |
|------|------|--------|
| 第 1 步 | 准备几种孔径的筛子 | 准备几种尺寸的结构元 |
| 第 2 步 | 用每种筛子筛一遍 | 对每个结构元做一次开运算 |
| 第 3 步 | 称一称留在筛面上的橙子重量 | 数一数开运算后剩多少像素 |
| 第 4 步 | 相邻两次重量差 = 该孔径的橙子重量 | $D_k = M_{k-1} - M_k$ |
| 第 5 步 | 画"孔径—重量"曲线 | 画"结构元尺寸—像素数"曲线 |

**曲线峰值的位置**，就是"哪种尺寸的橙子最多"。

**关键补充**：

- 现实里筛橙子用的是**圆孔**筛子，不是方孔；
- 因为橙子是球形，圆孔筛能真实反映橙子的直径；
- 形态学里用**圆盘形**结构元，道理一模一样。

---

#### 八、算法过程中的两个易错点

**易错点 1：`np.sum` 对 uint8 会溢出**

- `uint8` 最大 255，`np.sum` 累加会"回绕"（wrap）；
- 正确做法：先转 `int64` 或 `float` 再求和；
- 代码：`np.sum(opened.astype(np.int64))`。

**易错点 2：差分方向容易搞反**

- $M_k$ 随结构元增大而**减小**（筛掉更多）；
- 所以 $M_{k-1} - M_k > 0$，表示"被去掉的物体数量"；
- 如果写成 $M_k - M_{k-1}$，结果全是负数，曲线就反了；
- 代码里用 `-np.diff(M)`，等价于 $M_{k-1} - M_k$。

**易错点 3：结构元形状没统一**

- 如果一组结构元里，有的用矩形、有的用椭圆，结果不可比；
- 粒度测定时必须**固定形状，只变尺寸**；
- 通常统一用 `MORPH_ELLIPSE`。

---

#### 九、应用场景

| 场景 | 说明 |
|------|------|
| 材料颗粒分析 | 统计沙粒、粉末的尺寸分布 |
| 医学细胞计数 | 按细胞大小分类，判断异常 |
| 工业质检 | 检测产品表面颗粒、缺陷的尺寸分布 |
| 天文图像 | 统计星体、陨石坑的尺寸分布 |
| 生物显微 | 统计细菌、细胞核的尺寸分布 |

---

#### 十、一句话总结

> **粒度测定 = 用一组"形状统一、尺寸递增"的结构元做开运算，模拟"用一套圆孔筛子筛物体"**：
>
> - **形状**决定"筛孔的形状"，圆盘形各向同性，最常用；
> - **尺寸**决定"筛孔的大小"，尺寸越大，剩下的物体越少；
> - 相邻两次的差 $M_{k-1} - M_k$，就是"大小在这个尺寸段的物体数量"；
> - 以结构元尺寸为横轴、差值为纵轴画曲线，就得到粒度分布；
> - 峰值位置 = 该尺寸的物体最多。
>
> **记住两个关键词：形状统一（通常圆盘形）、尺寸递增。**

#### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

img = cv2.imread('P&V/cat.jpg')                              # 读取图片
if img is not None:                                          # 如果图片存在
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)         # 转换为灰度图
    _, binary = cv2.threshold(img_gray, 127, 255, cv2.THRESH_BINARY)  # 二值化
    sizes = range(1, 15)                                     # 尺寸范围
    granulometry = []                                        # 存储结果
    for size in sizes:                                       # 遍历尺寸
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (size, size))  # 结构元
        opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)  # 开运算
        # 先转 int64 再求和，避免 uint8 溢出
        granulometry.append(np.sum(opened.astype(np.int64)))  # 计算总亮度
    granulometry = np.array(granulometry, dtype=np.float64)  # 转换为数组
    # 差分：M_{k-1} - M_k，表示被开运算去掉的物体数量
    granulometry_diff = -np.diff(granulometry)               # 取负，使差值为正
    
    plt.figure(figsize=(10, 5))                              # 创建画布
    plt.plot(list(sizes)[1:], granulometry_diff, marker='o')  # 绘制粒度分布
    plt.title('粒度分布')                                     # 标题
    plt.xlabel('结构元尺寸')                                   # x 轴标签
    plt.ylabel('被去掉的像素数 (M_{k-1} - M_k)')               # y 轴标签
    plt.grid(True)                                           # 网格
    plt.show()                                               # 显示图像
else:
    print("请确保 'P&V/cat.jpg' 存在")                       # 提示信息

6.6.15 形态学边缘检测与角点检测（通俗版）

前面讲的腐蚀、膨胀、开运算、闭运算，都是在“改变白色区域的大小和形状”。
边缘检测和角点检测，其实就是在问两个问题：

- 边缘在哪里？ —— 白色区域和黑色区域交界的地方，就是边缘。
- 角点在哪里？ —— 边缘突然拐弯、分叉、成直角的地方，就是角点。

形态学做这两件事，不需要求导数，也不需要算梯度，而是靠“腐蚀”和“膨胀”的差值，以及“模板盖章”来找。

一、形态学边缘检测

1. 核心公式

形态学梯度（Morphological Gradient）：

$$
G = (A \oplus S) - (A \ominus S)
$$

其中：

- $A$：原图（二值图或灰度图）
- $S$：结构元（可以理解成“刷子”“模板”）
- $\oplus$：膨胀
- $\ominus$：腐蚀

2. 通俗理解：“胖一圈”减“瘦一圈”

想象一个白色方块：

- 膨胀：白色方块“胖一圈”，向外扩出去一点。
- 腐蚀：白色方块“瘦一圈”，向内缩回来一点。
- 膨胀结果 减去 腐蚀结果：剩下的正好是原来边界附近的一圈。

图示：

```text
原图：        ########
膨胀后：      ##########
腐蚀后：        ######
膨胀 - 腐蚀： ##      ##   <- 这一圈就是边缘带
```

所以形态学梯度得到的边缘通常是一条“带子”，不是单像素细线。

3. 内边界：更细的边缘

如果只想要物体最外层的内部边缘，可以用：

$$
E = A - (A \ominus S)
$$

意思是：

> 原图减去腐蚀后的图，剩下的就是物体往里缩时露出来的那一圈。

对比：

| 公式 | 结果 | 特点 |
|---|---|---|
| $(A \oplus S) - (A \ominus S)$ | 边缘带 | 较粗，跨在边界两侧 |
| $A - (A \ominus S)$ | 内边界 | 较细，偏向物体内部 |
| $(A \oplus S) - A$ | 外边界 | 较细，偏向物体外部 |

4. OpenCV 实现

```python
import cv2
import numpy as np

img = cv2.imread("shape.png", 0)
_, bw = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)

kernel = np.ones((3, 3), np.uint8)

# 形态学梯度：边缘带
gradient = cv2.morphologyEx(bw, cv2.MORPH_GRADIENT, kernel)

# 内边界：原图 - 腐蚀
eroded = cv2.erode(bw, kernel)
inner_edge = cv2.subtract(bw, eroded)
```

一句话总结：

> 膨胀减腐蚀，得到边缘带；原图减腐蚀，得到内边界。

二、形态学角点检测

1. 什么是角点？

角点不是普通的边缘点，而是边缘发生拐弯、交叉、分叉的地方。

比如下面这两种最典型的角点：

```text
L 形角点：              T 形角点：

## 0 0 0                ## ## ##
## 0 0 0                ## ## ##
## 0 0 0                   ##
## ## ##                   ##
## ## ##                   ##
```

- L 形角点：一条竖边 + 一条横边，交出一个直角。
- T 形角点：一条竖线顶在一条横线中间，形成“T”字。

⚠️ 注意：不要把 L 形画成一个矩形块，也不要把 T 形画成“工”字形或“I”字形。那样看不出拐角或分叉，就不是角点了。

2. 核心工具：击中击不中变换（Hit-or-Miss）

思想非常像“盖章匹配”：

- 设计一个结构元，里面有三种值：
  - `1`：这里必须是白色前景；
  - `0`：这里必须是黑色背景；
  - `-1` 或空：这里无所谓，不关心。
- 用这个结构元扫描整张图。
- 如果图像中某个位置，白的地方白、黑的地方黑，完全符合模板，就算“击中”。
- 击中的位置，就是角点。

也就是说：

> 用一个小模板去扫描整张图，哪里长得像角点，哪里就被标记出来。

3. 左上角 L 形角点模板示例

假设图像中白色物体是：

```text
0 0 0
0 1 1
0 1 1
```

想找这种左上角，可以设计模板：

```text
-1 -1  0
-1  1  0
 0  0  0
```

含义：

- 左上、上、左必须是背景（-1）；
- 中心必须是前景（1）；
- 其他位置不关心（0）。

如果图像某处满足这个模式，就说明这里有一个左上角。

4. OpenCV 实现

```python
import cv2
import numpy as np

img = cv2.imread("corner.png", 0)
_, bw = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)

# 找左上角 L 形角点
kernel = np.array([
    [-1, -1,  0],
    [-1,  1,  0],
    [ 0,  0,  0]
], dtype=np.int8)

hitmiss = cv2.morphologyEx(bw, cv2.MORPH_HITMISS, kernel)
```

注意：

- 只能用于二值图；
- 白色是前景，黑色是背景；
- kernel 里通常用 `1` 表示前景，`-1` 表示背景，`0` 表示不关心。

5. 找不同方向的角点

如果想找不同方向的角点，就分别设计不同的 kernel，然后分别做 Hit-or-Miss，最后把结果合并：

- 左上角；
- 右上角；
- 左下角；
- 右下角；
- T 形角点；
- 十字角点。

通俗理解：

> Hit-or-Miss 就像拿一个“角点印章”在图上到处盖。
> 只有图像局部和印章完全一致，才盖得上。
> 盖上的位置，就是角点。

三、两者对比

| 方法 | 做什么 | 核心思想 | 结果 |
|---|---|---|---|
| 形态学梯度 | 边缘检测 | 膨胀减腐蚀 | 边缘带 |
| 原图减腐蚀 | 内边界检测 | 原图减腐蚀 | 较细的内边缘 |
| Hit-or-Miss | 角点检测 | 模板匹配前景和背景 | 特定方向的角点 |

一句话记忆：

> 边缘 = 胖一圈减瘦一圈。
> 角点 = 拿角点模板去盖章。

四、最终记忆

- 形态学梯度 $(A \oplus S) - (A \ominus S)$：适合找粗边缘（边缘带）。
- 原图减腐蚀 $A - (A \ominus S)$：适合找内边界。
- Hit-or-Miss：适合找有明确形状模式的角点、端点、交叉点。
- 核心心法：边缘靠膨胀腐蚀相减，角点靠模板盖章匹配。
- 角点示意要画对：L 形是横竖直角相交，T 形是竖线顶在横线中间，不是矩形块，也不是工字形。

#### 示例

In [ ]:
import cv2                                                   # 导入 OpenCV 库
import numpy as np                                           # 导入 NumPy 库
import matplotlib.pyplot as plt                              # 导入 Matplotlib 库

plt.rcParams['font.sans-serif'] = ['SimHei']                 # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                   # 解决负号显示问题

# ---------- 合成二值图：黑底 + 白色几何形状 ----------
binary = np.zeros((300, 300), dtype=np.uint8)                # 纯黑背景

cv2.rectangle(binary, (40, 40), (140, 140), 255, -1)         # 白色方块
cv2.circle(binary, (220, 90), 50, 255, -1)                   # 白色圆形
cv2.rectangle(binary, (60, 180), (240, 260), 255, -1)        # 白色长条

# ---------- 形态学梯度（边缘检测） ----------
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))   # 3×3 椭圆结构元

gradient = cv2.morphologyEx(binary, cv2.MORPH_GRADIENT, kernel)  # 膨胀 - 腐蚀

# ---------- 角点检测（击中法） ----------
# 注意：结构元要小，只匹配角点局部，而不是整个物体
# 检测“左上角”：
#   中心点必须是前景（1）
#   正上方、正左方必须是背景（-1）—— 说明这是物体的外角
#   正右方、正下方必须是前景（1）—— 说明这是物体的内部
#   其余位置不关心（0）
kernel_corner = np.array([
    [ 0, -1,  0],
    [-1,  1,  1],
    [ 0,  1,  0]
], dtype=np.int8)

# 击中击不中要求结构元是 0/1/-1，且输入是标准二值图（0/255）
corners = cv2.morphologyEx(binary, cv2.MORPH_HITMISS, kernel_corner)

# 为了让角点更明显，放大显示（每个角点画成小圆）
corners_vis = cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR)       # 转彩色，便于标红
ys, xs = np.where(corners == 255)                            # 找到角点位置
for (x, y) in zip(xs, ys):                                   # 逐个标红
    cv2.circle(corners_vis, (x, y), 6, (0, 0, 255), -1)      # 红色实心圆

# ---------- 可视化 ----------
plt.figure(figsize=(15, 5))                                  # 创建画布

plt.subplot(1, 3, 1)                                         # 子图 1
plt.title('合成二值图')                                       # 标题
plt.imshow(binary, cmap='gray', vmin=0, vmax=255)            # 显示二值图
plt.axis('off')                                              # 关闭坐标轴

plt.subplot(1, 3, 2)                                         # 子图 2
plt.title('形态学梯度 (边缘)')                                # 标题
plt.imshow(gradient, cmap='gray', vmin=0, vmax=255)          # 显示形态学梯度
plt.axis('off')                                              # 关闭坐标轴

plt.subplot(1, 3, 3)                                         # 子图 3
plt.title('角点检测 (击中法, 左上角)')                     # 标题
plt.imshow(cv2.cvtColor(corners_vis, cv2.COLOR_BGR2RGB))     # 显示角点（红点）
plt.axis('off')                                              # 关闭坐标轴

plt.tight_layout()                                           # 自动调整间距
plt.show()                                                   # 显示图像

**图像解读**

**第一张：合成二值图**

- 黑底白图，包含三个几何形状：
  - 左上：正方形
  - 右上：圆形
  - 下方：长方形
- 这是后面所有形态学操作的输入。

**第二张：形态学梯度（边缘）**

- 用形态学梯度公式 $G = (A \oplus S) - (A \ominus S)$ 得到。
- 结果就是每个白色形状外面包了一圈白边，形状内部是黑的。
- 这圈白边就是“膨胀减腐蚀”留下的边界带。
- 可以看到：
  - 正方形和长方形的边缘是直边，很规整；
  - 圆形的边缘是圆弧，也比较平滑；
  - 边缘带比较粗，不是单像素细线。

**第三张：角点检测（击中击不中，左上角）**

- 用 Hit-or-Miss 模板匹配“左上角”模式，匹配到的位置用红点标出。
- 结果：
  - 正方形左上角有一个红点 —— 正确，这是标准 L 形左上角。
  - 长方形左上角有一个红点 —— 正确，这也是标准 L 形左上角。
  - 圆形左上区域出现了一段红色弧线 —— 这是**误检**。

**为什么圆形会误检？**

圆形的边缘是连续弯曲的，在左上角附近，局部看起来也像“上方和左方是背景，右方和下方是前景”，所以 Hit-or-Miss 模板在这个位置也匹配成功了。

但是圆形并没有真正的“角点”，它只是曲率连续变化的圆弧。这说明：

> **Hit-or-Miss 找的是“局部模式匹配”，不是真正意义上的几何角点。**
> 只要局部像素排列符合模板，就会被判为角点，哪怕它是圆弧。

**怎么改进？**

1. **换更严格的角点模板**：比如要求右上、左下也是背景，排除圆弧。
2. **多方向模板结合**：真正的角点应该在多个方向上都表现出“拐角”特征，圆弧只在一个方向匹配。
3. **后处理**：把连成一片的红色区域（如圆弧那段）去掉，只保留孤立的红点。
4. **先做边缘细化**：在细边缘上做角点检测，比在实心区域上做更准确。

**一句话总结：**

- 第二张图：形态学梯度能正确找出所有形状的边界。
- 第三张图：Hit-or-Miss 能找到正方形和长方形的左上角，但会在圆形左上角误检出一段弧。
- 原因是圆形局部也满足“左上角”模板，说明形态学角点检测对圆弧不鲁棒。